# ❄️ Pré-dimensionamento de ciclos de refrigeração para visi-coolers

**CAExperts × IMBERA** — rotina de pré-dimensionamento termodinâmico de refrigeradores comerciais de
bebidas (porta de vidro), cujas saídas alimentam o modelo de sistema no **Simcenter Amesim**.

| Entradas do usuário | Saídas (entradas do Amesim) |
|---|---|
| COP desejado (capacidade / potência elétrica do compressor) | potência elétrica e deslocamento do compressor, η_v, η_is |
| Capacidade: litros **ou** nº de garrafas de 600 mL | condensador e evaporador: área, comprimento de tubo, diâmetros, fileiras, tubos, face, vazão de ar |
| Temperatura ambiente T∞ | tubo capilar: diâmetro interno e comprimento |
| Temperatura alvo interna (ex.: 2,5 °C) | carga de refrigerante, estados do ciclo, balanços de energia e exergia |
| Eficácia ε do condensador e do evaporador (padrão 0,65) | velocidade de face e nº de fileiras que atendem ε, UA, NTU |
| Potência máxima do compressor (opcional, requisito de ensaio) | pull-down estimado: tempo e pico de potência; capacidade limitada se preciso |

**Papel da rotina:** é o *step-down* do modelo do Amesim. Gera parâmetros físicos e coerentes entre si para montar
o modelo de sistema; as curvas de pull-down, consumo, fuga térmica e água drenada saem depois do próprio Amesim.

**Fluxo de cálculo:** capacidade do gabinete → carga térmica → ciclo (equação de estado de referência via
CoolProp) → verificação do COP contra Carnot e contra o compressor real → compressor → trocadores tubo-aleta
dimensionados **segmento a segmento** (ε-NTU local, correlações locais, queda de pressão acoplada ao ciclo) →
capilar pelo **modelo homogêneo com blocagem crítica** → carga de fluido → exergia → avisos de projeto.

> **Natureza do resultado.** É um *pré-dimensionamento*: as correlações empíricas têm incerteza típica de
> ±15–30 % no coeficiente local e as premissas de uso (aberturas de porta, reposição de produto) dominam a
> carga térmica. O notebook mostra essa incerteza (seção 7); a conferência em regime permanente é feita contra o
> modelo do Amesim (bloco `amesim.alvo_validacao`, seção 6).

**Como usar:** `Ambiente de execução → Executar tudo`. Altere as entradas na seção 3 e rode de novo da seção 4
em diante. Para premissas avançadas (geometria dos trocadores, fluido, perfil de uso), edite a célula
*Parâmetros avançados*.

**Índice:** 0. Ambiente · 1. Código do pacote · 2. Fundamentação teórica · 3. Entradas · 4. Dimensionamento ·
5. Gráficos · 6. Saídas para o Amesim · 7. Sensibilidade e incerteza · 8. Verificação e validação ·
9. Hipóteses e limitações

In [ ]:
#@title 0. Ambiente: CoolProp e estrutura do pacote `predimref`
import os, subprocess, sys
try:
    import CoolProp
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "CoolProp"], check=True)
    import CoolProp
os.makedirs("predimref/correlacoes", exist_ok=True)
print("CoolProp", CoolProp.__version__)

## 1. Código-fonte do pacote `predimref`

As células abaixo gravam os módulos do pacote no disco do Colab (`%%writefile`). Não é preciso lê-las para
usar a ferramenta — podem ficar recolhidas. Cada módulo traz no cabeçalho as equações e as referências usadas.

| Módulo | Conteúdo |
|---|---|
| `propriedades` | refrigerante (CoolProp/HEOS), saturação com cache, ar seco e psicrometria (ASHRAE RP-1485) |
| `correlacoes.monofasico` | Churchill (atrito), Gnielinski (Nusselt) |
| `correlacoes.bifasico` | Shah 2013/2022, Chato 1962 (condensação); Gungor-Winterton 1987, Liu-Winterton 1991 (ebulição); Müller-Steinhagen & Heck; Rouhani-Axelsson |
| `correlacoes.lado_ar` | Wang, Chi & Chang 2000 (j, f), eficiência de aleta de Schmidt, ΔP de Kays & London |
| `gabinete` | litros ↔ garrafas, transmissão com fatores de forma, porta, gaxeta, infiltração, produto |
| `ciclo` / `compressor` | estados, COP × Carnot, rendimentos, exergia; deslocamento e ponto de catálogo |
| `trocador` | condensador e evaporador por zonas e segmentos |
| `capilar` | modelo homogêneo, flash isentálpico, blocagem, problema inverso, seleção de diâmetro |
| `projeto` / `relatorio` / `graficos` | orquestração iterativa, JSON para o Amesim, figuras |

In [ ]:
%%writefile predimref/constantes.py
"""Constantes físicas e conversões de unidade."""

G = 9.80665            # aceleração da gravidade padrão [m/s²]
P_ATM = 101325.0       # pressão atmosférica padrão [Pa]
T0_K = 273.15          # 0 °C em kelvin
W_PARA_BTUH = 3.412142  # 1 W = 3,412142 Btu/h
SEGUNDOS_DIA = 86400.0


def C_para_K(t_c: float) -> float:
    return t_c + T0_K


def K_para_C(t_k: float) -> float:
    return t_k - T0_K


def bar(p_pa: float) -> float:
    return p_pa / 1e5

In [ ]:
%%writefile predimref/avisos.py
"""Registro de avisos de projeto (premissas fora de faixa, riscos, inconsistências)."""
from __future__ import annotations

from dataclasses import dataclass, field


class ErroProjeto(ValueError):
    """Entrada fisicamente impossível ou incoerente (p. ex. COP acima do de Carnot)."""


@dataclass
class Aviso:
    nivel: str      # "INFO", "ATENCAO" ou "CRITICO"
    origem: str
    mensagem: str

    def __str__(self) -> str:
        return f"[{self.nivel}] {self.origem}: {self.mensagem}"


@dataclass
class Avisos:
    itens: list[Aviso] = field(default_factory=list)

    def info(self, origem: str, msg: str) -> None:
        self.itens.append(Aviso("INFO", origem, msg))

    def atencao(self, origem: str, msg: str) -> None:
        self.itens.append(Aviso("ATENCAO", origem, msg))

    def critico(self, origem: str, msg: str) -> None:
        self.itens.append(Aviso("CRITICO", origem, msg))

    def extend(self, origem: str, msgs: list[str], nivel: str = "ATENCAO") -> None:
        for m in msgs:
            self.itens.append(Aviso(nivel, origem, m))

    def __iter__(self):
        return iter(self.itens)

    def __len__(self) -> int:
        return len(self.itens)

    def texto(self) -> str:
        ordem = {"CRITICO": 0, "ATENCAO": 1, "INFO": 2}
        return "\n".join(str(a) for a in sorted(self.itens, key=lambda a: ordem[a.nivel]))

In [ ]:
%%writefile predimref/propriedades.py
"""Propriedades termofísicas: refrigerante (CoolProp, equações de estado de referência) e ar úmido.

Por que CoolProp/HEOS: as equações de estado usadas são as mesmas de referência do NIST
REFPROP (p. ex. Lemmon, McLinden & Wagner 2009 para o propano; Bücker & Wagner 2006 para o
isobutano; Tillner-Roth & Baehr 1994 para o R134a), com modelos de transporte
(viscosidade, condutividade, tensão superficial) da literatura. Bell et al. (2014),
Ind. Eng. Chem. Res. 53, 2498-2508.

A interface de baixo nível (``AbstractState``) é usada por desempenho: cada chamada leva
poucos microssegundos, o que permite discretizar trocadores e capilar finamente.
"""
from __future__ import annotations

from dataclasses import dataclass
from functools import lru_cache
from typing import Optional

import CoolProp
import CoolProp.CoolProp as CP

from .constantes import P_ATM

# Fluidos puros ou pseudo-puros (glide desprezível) aceitos. Misturas zeotrópicas
# (R407C, R448A, R449A...) exigiriam tratar o glide de temperatura e ficam de fora.
# R744 (CO2) opera transcrítico a 32 °C de ambiente e não é coberto por este ciclo subcrítico.
FLUIDOS_SUPORTADOS = {
    "R290": "propano (HC, A3)",
    "R600a": "isobutano (HC, A3)",
    "R1270": "propileno (HC, A3)",
    "R134a": "HFC (A1)",
    "R1234yf": "HFO (A2L)",
    "R1234ze(E)": "HFO (A2L)",
    "R152a": "HFC (A2)",
    "R404A": "HFC pseudo-puro (A1)",
    "R507A": "HFC azeotrópico (A1)",
    "R22": "HCFC (A1), legado",
    "R12": "CFC, legado (apenas validação com dados históricos)",
}

# Classe de segurança (ISO 817 / ASHRAE 34).
CLASSE_SEGURANCA = {"R290": "A3", "R600a": "A3", "R1270": "A3", "R152a": "A2", "R1234yf": "A2L",
                    "R1234ze(E)": "A2L", "R134a": "A1", "R404A": "A1", "R507A": "A1", "R22": "A1", "R12": "A1"}

# Limite inferior de inflamabilidade [kg/m³] (ISO 817 / ASHRAE 34), usado nos limites de carga.
LFL_KG_M3 = {"R290": 0.038, "R600a": 0.043, "R1270": 0.046, "R152a": 0.130,
             "R1234yf": 0.289, "R1234ze(E)": 0.303}


@dataclass(frozen=True)
class Saturacao:
    """Propriedades de saturação a uma pressão (líquido 'l' e vapor 'v')."""
    P: float
    T: float
    hl: float
    hv: float
    rhol: float
    rhov: float
    sl: float
    sv: float
    mul: float
    muv: float
    kl: float
    kv: float
    cpl: float
    cpv: float
    sigma: float
    pr: float  # pressão reduzida P/Pc

    @property
    def hlv(self) -> float:
        return self.hv - self.hl

    @property
    def Prl(self) -> float:
        return self.cpl * self.mul / self.kl

    @property
    def Prv(self) -> float:
        return self.cpv * self.muv / self.kv

    @property
    def vl(self) -> float:
        return 1.0 / self.rhol

    @property
    def vv(self) -> float:
        return 1.0 / self.rhov


@dataclass(frozen=True)
class Estado:
    """Estado termodinâmico do refrigerante.

    ``x`` é o título termodinâmico (h - hl)/(hv - hl) quando P < Pc: pode ser < 0
    (líquido sub-resfriado) ou > 1 (vapor superaquecido). ``bifasico`` indica 0 < x < 1.
    Propriedades de transporte só são preenchidas fora da região bifásica.
    """
    P: float
    T: float
    h: float
    s: float
    rho: float
    x: float
    cp: Optional[float] = None
    mu: Optional[float] = None
    k: Optional[float] = None

    @property
    def bifasico(self) -> bool:
        return 0.0 < self.x < 1.0

    @property
    def v(self) -> float:
        return 1.0 / self.rho

    @property
    def Pr(self) -> Optional[float]:
        if self.cp is None:
            return None
        return self.cp * self.mu / self.k


class Fluido:
    """Refrigerante com acesso rápido a estados e saturação (SI)."""

    def __init__(self, nome: str, backend: str = "HEOS"):
        if nome not in FLUIDOS_SUPORTADOS:
            raise ValueError(
                f"Fluido '{nome}' não suportado. Opções: {', '.join(FLUIDOS_SUPORTADOS)}. "
                "R744 (transcrítico) e misturas zeotrópicas exigem outro modelo de ciclo."
            )
        self.nome = nome
        self._as = CoolProp.AbstractState(backend, nome)
        self.Tc = self._as.T_critical()
        self.Pc = self._as.p_critical()
        self.M = self._as.molar_mass()  # kg/mol
        self.Ttriplo = self._as.Ttriple()
        self.hidrocarboneto = "HC" in FLUIDOS_SUPORTADOS[nome]

    # ------------------------------------------------------------------ saturação
    def Psat(self, T: float) -> float:
        self._as.update(CoolProp.QT_INPUTS, 0.0, T)
        return self._as.p()

    def Tsat(self, P: float) -> float:
        self._as.update(CoolProp.PQ_INPUTS, P, 0.0)
        return self._as.T()

    def sat(self, P: float) -> Saturacao:
        """Propriedades de saturação (com cache, pressão arredondada a 0,1 Pa)."""
        return _sat_cache(self.nome, round(P, 1))

    # ------------------------------------------------------------------ estados
    def _estado_atual(self, P: float, transporte: bool = True) -> Estado:
        a = self._as
        T, h, s, rho = a.T(), a.hmass(), a.smass(), a.rhomass()
        if P < self.Pc:
            sat = self.sat(P)
            x = (h - sat.hl) / sat.hlv
        else:
            x = -1.0 if T < self.Tc else 2.0
        if 0.0 < x < 1.0 or not transporte:
            return Estado(P, T, h, s, rho, x)
        try:
            cp, mu, k = a.cpmass(), a.viscosity(), a.conductivity()
        except ValueError:
            cp = mu = k = None
        return Estado(P, T, h, s, rho, x, cp, mu, k)

    def estado_PH(self, P: float, h: float, transporte: bool = True) -> Estado:
        """Estado por (P, h). ``transporte=False`` pula viscosidade, condutividade e c_p (mais rápido)."""
        self._as.update(CoolProp.HmassP_INPUTS, h, P)
        return self._estado_atual(P, transporte)

    def estado_PT(self, P: float, T: float) -> Estado:
        """Estado monofásico por (P, T). Na saturação exata use ``estado_PQ``."""
        self._as.update(CoolProp.PT_INPUTS, P, T)
        return self._estado_atual(P)

    def estado_PS(self, P: float, s: float) -> Estado:
        self._as.update(CoolProp.PSmass_INPUTS, P, s)
        return self._estado_atual(P)

    def estado_PQ(self, P: float, Q: float) -> Estado:
        self._as.update(CoolProp.PQ_INPUTS, P, Q)
        st = self._estado_atual(P)
        if Q in (0.0, 1.0):
            # Estado saturado: o título calculado pode sair 1e-12 fora de [0,1]; fixa e
            # preenche transporte da fase saturada.
            sat = self.sat(P)
            if Q == 0.0:
                return Estado(P, st.T, sat.hl, sat.sl, sat.rhol, 0.0, sat.cpl, sat.mul, sat.kl)
            return Estado(P, st.T, sat.hv, sat.sv, sat.rhov, 1.0, sat.cpv, sat.muv, sat.kv)
        return st

    def estado_PX(self, P: float, x: float) -> Estado:
        """Estado por título termodinâmico (aceita x<0 ou x>1 via entalpia)."""
        sat = self.sat(P)
        return self.estado_PH(P, sat.hl + x * sat.hlv)

    def kappa(self, P: float, T: float, rho: Optional[float] = None) -> float:
        """Razão cp/cv do vapor (expoente isentrópico de gás ideal equivalente). Com ``rho`` usa
        (ρ, T), que vale também para vapor saturado (onde a entrada (P, T) é ambígua)."""
        if rho is not None:
            self._as.update(CoolProp.DmassT_INPUTS, rho, T)
        else:
            try:
                self._as.update(CoolProp.PT_INPUTS, P, T)
            except ValueError:            # (P, T) na saturação: vapor saturado
                self._as.update(CoolProp.PQ_INPUTS, P, 1.0)
        return self._as.cpmass() / self._as.cvmass()

    def __repr__(self) -> str:
        return f"Fluido({self.nome}, Tc={self.Tc - 273.15:.2f} °C, Pc={self.Pc / 1e5:.2f} bar)"


@lru_cache(maxsize=200_000)
def _sat_cache(nome: str, P: float) -> Saturacao:
    a = _as_cache(nome)
    a.update(CoolProp.PQ_INPUTS, P, 0.0)
    T = a.T()
    hl, rhol, sl = a.hmass(), a.rhomass(), a.smass()
    mul, kl, cpl = a.viscosity(), a.conductivity(), a.cpmass()
    try:
        sigma = a.surface_tension()
    except ValueError:
        sigma = float("nan")
    a.update(CoolProp.PQ_INPUTS, P, 1.0)
    hv, rhov, sv = a.hmass(), a.rhomass(), a.smass()
    muv, kv, cpv = a.viscosity(), a.conductivity(), a.cpmass()
    return Saturacao(P, T, hl, hv, rhol, rhov, sl, sv, mul, muv, kl, kv, cpl, cpv, sigma,
                     P / a.p_critical())


@lru_cache(maxsize=32)
def _as_cache(nome: str):
    return CoolProp.AbstractState("HEOS", nome)


# ============================================================================ ar
@dataclass(frozen=True)
class PropAr:
    rho: float
    cp: float
    mu: float
    k: float

    @property
    def Pr(self) -> float:
        return self.cp * self.mu / self.k


@lru_cache(maxsize=4096)
def ar_seco(T: float, P: float = P_ATM) -> PropAr:
    """Ar seco (Lemmon et al. 2000 via CoolProp). Usado nas correlações de transferência
    do lado do ar: a umidade altera cp e μ em menos de 1 % nas condições de projeto."""
    a = _as_cache("Air")
    a.update(CoolProp.PT_INPUTS, P, round(T, 3))
    return PropAr(a.rhomass(), a.cpmass(), a.viscosity(), a.conductivity())


def ar_umido(T: float, UR: float, P: float = P_ATM) -> dict:
    """Psicrometria rigorosa (ASHRAE RP-1485, Herrmann, Kretzschmar & Gatley 2009, via
    CoolProp HAPropsSI). Retorna entalpia e volume por kg de ar seco.

    Chaves: W [kg_v/kg_as], h [J/kg_as], v [m³/kg_as], rho_mist [kg_mistura/m³],
    Tdp [K], UR.
    """
    W = CP.HAPropsSI("W", "T", T, "P", P, "R", UR)
    h = CP.HAPropsSI("H", "T", T, "P", P, "W", W)
    v = CP.HAPropsSI("Vda", "T", T, "P", P, "W", W)
    Tdp = CP.HAPropsSI("Tdp", "T", T, "P", P, "W", W) if W > 0 else float("nan")
    return {"T": T, "UR": UR, "W": W, "h": h, "v": v, "rho_mist": (1.0 + W) / v, "Tdp": Tdp}


def calor_latente_agua(T: float) -> float:
    """Entalpia de vaporização da água [J/kg] (IAPWS-95 via CoolProp)."""
    Tq = min(max(T, 273.16), 373.0)
    return (CP.PropsSI("H", "T", Tq, "Q", 1, "Water") - CP.PropsSI("H", "T", Tq, "Q", 0, "Water"))

In [ ]:
%%writefile predimref/correlacoes/__init__.py
"""Correlações de transferência de calor e queda de pressão (fontes citadas em cada função)."""

In [ ]:
%%writefile predimref/correlacoes/monofasico.py
"""Escoamento monofásico interno em tubos."""
from __future__ import annotations

import math


def fator_atrito_churchill(Re: float, rug_rel: float = 0.0) -> float:
    """Fator de atrito de Darcy, Churchill (1977), Chem. Eng. 84(24):91-92.

    Expressão única para laminar, transição e turbulento (rugoso), equivalente a
    Colebrook-White no turbulento (desvio < 1 %) e a 64/Re no laminar.
    """
    if Re <= 0:
        raise ValueError("Re deve ser positivo")
    A = (2.457 * math.log(1.0 / ((7.0 / Re) ** 0.9 + 0.27 * rug_rel))) ** 16
    B = (37530.0 / Re) ** 16
    return 8.0 * ((8.0 / Re) ** 12 + (A + B) ** -1.5) ** (1.0 / 12.0)


def fator_atrito_blasius_laminar(Re: float) -> float:
    """Fator de Darcy usado por Müller-Steinhagen & Heck (1986): 64/Re até Re = 1187 e
    Blasius 0,3164 Re^-0,25 acima (as duas expressões se cruzam em Re = 1187)."""
    return 64.0 / Re if Re <= 1187.0 else 0.3164 * Re ** -0.25


def nusselt_gnielinski(Re: float, Pr: float) -> float:
    """Nusselt médio totalmente desenvolvido, Gnielinski (1976), Int. Chem. Eng. 16:359-368,
    com fator de atrito de Filonenko f = (0,790 ln Re − 1,64)^-2.

    * Re ≤ 2300: laminar desenvolvido, Nu = 3,66 (parede isotérmica; conservador frente a
      4,36 de fluxo uniforme e ao aumento por desenvolvimento térmico na entrada).
    * 2300 < Re < 10⁴: interpolação linear de Gnielinski (2013), VDI Heat Atlas, G1 §4.
    * Re ≥ 10⁴: Gnielinski (válido 0,1 ≤ Pr ≤ 1000, até Re = 5·10⁶).
    """
    nu_lam = 3.66
    if Re <= 2300.0:
        return nu_lam

    def turb(re: float) -> float:
        f = (0.790 * math.log(re) - 1.64) ** -2
        return (f / 8.0) * (re - 1000.0) * Pr / (1.0 + 12.7 * math.sqrt(f / 8.0) * (Pr ** (2.0 / 3.0) - 1.0))

    if Re >= 1.0e4:
        return turb(Re)
    gamma = (Re - 2300.0) / (1.0e4 - 2300.0)
    return (1.0 - gamma) * nu_lam + gamma * turb(1.0e4)


def h_interno_monofasico(G: float, D: float, mu: float, k: float, Pr: float) -> float:
    """Coeficiente convectivo interno [W/m²K] para fluxo mássico G [kg/m²s]."""
    Re = G * D / mu
    return nusselt_gnielinski(Re, Pr) * k / D


def gradiente_pressao_monofasico(G: float, D: float, rho: float, mu: float, rug: float) -> float:
    """−dP/dz por atrito [Pa/m] (Darcy-Weisbach com Churchill)."""
    Re = G * D / mu
    f = fator_atrito_churchill(Re, rug / D)
    return f * G * G / (2.0 * rho * D)

In [ ]:
%%writefile predimref/correlacoes/bifasico.py
"""Escoamento bifásico em tubos horizontais lisos: condensação, ebulição, atrito, fração de vazio.

Convenções: G fluxo mássico total [kg/m²s], x título, D diâmetro interno [m],
``sat`` um objeto ``Saturacao`` na pressão local.
"""
from __future__ import annotations

import math

from ..constantes import G as g
from .monofasico import fator_atrito_blasius_laminar, nusselt_gnielinski

_X_MIN, _X_MAX = 1e-4, 1.0 - 1e-4


def _clip_x(x: float) -> float:
    return min(max(x, _X_MIN), _X_MAX)


# ============================================================================ condensação
def _shah_hI_hNu(G: float, x: float, D: float, sat) -> tuple[float, float, float, float]:
    """Termos comuns às correlações de Shah: h_I (eq. 2 de Shah 2022), h_Nu, Z e J_g.

    h_I  = h_LT [(1−x)^0,8 + 3,8 x^0,76 (1−x)^0,04 / p_r^0,38] (μ_l/14μ_v)^n ,  n = 0,0058 + 0,557 p_r
           (h_LT: Dittus-Boelter com todo o fluxo como líquido, Re_LT = G D/μ_l)
    h_Nu = 1,32 Re_LS^(−1/3) [ρ_l(ρ_l−ρ_v) g k_l³ / μ_l²]^(1/3) ,  Re_LS = G (1−x) D / μ_l
    Z = (1/x − 1)^0,8 p_r^0,4 ;  J_g = x G / [g D ρ_v (ρ_l − ρ_v)]^0,5
    """
    pr = sat.pr
    Re_LT = G * D / sat.mul
    h_LT = 0.023 * Re_LT ** 0.8 * sat.Prl ** 0.4 * sat.kl / D
    n = 0.0058 + 0.557 * pr
    h_I = h_LT * (sat.mul / (14.0 * sat.muv)) ** n * (
        (1.0 - x) ** 0.8 + 3.8 * x ** 0.76 * (1.0 - x) ** 0.04 / pr ** 0.38)
    Re_LS = G * (1.0 - x) * D / sat.mul
    h_Nu = 1.32 * Re_LS ** (-1.0 / 3.0) * (
        sat.rhol * (sat.rhol - sat.rhov) * g * sat.kl ** 3 / sat.mul ** 2) ** (1.0 / 3.0)
    Z = (1.0 / x - 1.0) ** 0.8 * pr ** 0.4
    Jg = x * G / math.sqrt(g * D * sat.rhov * (sat.rhol - sat.rhov))
    return h_I, h_Nu, Z, Jg


def condensacao_shah_2013(G: float, x: float, D: float, sat) -> tuple[float, str]:
    """Condensação em tubo horizontal, Shah (2013), ASHRAE Trans. 119(2), com os regimes:
    Regime I   : J_g ≥ 0,98 (Z + 0,263)^−0,62           → h = h_I
    Regime III : J_g ≤ 0,95 (1,254 + 2,27 Z^1,249)^−1     → h = h_Nu   (escoamento estratificado)
    Regime II  : demais casos                           → h = h_I + h_Nu
    Shah (2022), Int. J. Heat Mass Transfer 194:123069, recomenda esta forma para
    hidrocarbonetos em qualquer orientação (base de 2931 pontos, G de 1,2 a 1360 kg/m²s).
    Retorna (h [W/m²K], regime)."""
    x = _clip_x(x)
    h_I, h_Nu, Z, Jg = _shah_hI_hNu(G, x, D, sat)
    if Jg >= 0.98 * (Z + 0.263) ** -0.62:
        return h_I, "I"
    if Jg <= 0.95 / (1.254 + 2.27 * Z ** 1.249):
        return h_Nu, "III"
    return h_I + h_Nu, "II"


def condensacao_shah_2022(G: float, x: float, D: float, sat, hidrocarboneto: bool) -> tuple[float, str]:
    """Correlação melhorada de Shah (2022), Int. J. Heat Mass Transfer 194:123069, tubo horizontal.

    * hidrocarbonetos, ou Re_LT < 100: Shah (2013) (``condensacao_shah_2013``);
    * demais fluidos: We_GT = G² D/(ρ_v σ), Fr_LT = G²/(ρ_l² g D)
        Regime I  se We_GT > 100, Fr_LT > 0,026 e J_g ≥ 0,98 (Z + 0,263)^−0,62
        Regime III se Fr_LT > 0,026 e J_g ≤ 0,95 (1,254 + 2,27 Z^1,249)^−1
        Regime II nos demais casos;
      h_I pela eq. 2 se D > 6 mm, ou pela forma de Cavallini et al. (2006) se D ≤ 6 mm:
        h_I = h_LT [1 + 1,128 x^0,817 (ρ_l/ρ_v)^0,3685 (μ_l/μ_v)^0,2363 (1 − μ_v/μ_l)^2,144 Pr_l^−0,1]
    """
    x = _clip_x(x)
    Re_LT = G * D / sat.mul
    if hidrocarboneto or Re_LT < 100.0:
        return condensacao_shah_2013(G, x, D, sat)
    h_I, h_Nu, Z, Jg = _shah_hI_hNu(G, x, D, sat)
    if D <= 6.0e-3:
        h_LT = 0.023 * Re_LT ** 0.8 * sat.Prl ** 0.4 * sat.kl / D
        h_I = h_LT * (1.0 + 1.128 * x ** 0.817 * (sat.rhol / sat.rhov) ** 0.3685
                      * (sat.mul / sat.muv) ** 0.2363 * (1.0 - sat.muv / sat.mul) ** 2.144 * sat.Prl ** -0.1)
    We = G * G * D / (sat.rhov * sat.sigma)
    Fr = G * G / (sat.rhol ** 2 * g * D)
    if We > 100.0 and Fr > 0.026 and Jg >= 0.98 * (Z + 0.263) ** -0.62:
        return h_I, "I"
    if Fr > 0.026 and Jg <= 0.95 / (1.254 + 2.27 * Z ** 1.249):
        return h_Nu, "III"
    return h_I + h_Nu, "II"


def condensacao_shah_2009(G: float, x: float, D: float, sat) -> tuple[float, str]:
    """Shah (2009), HVAC&R Research 15(5):889-913, tubo horizontal: só regimes I e II, com o
    regime II recomendado pelo próprio autor apenas para Re_GT > 35 000 (mantida para referência)."""
    x = _clip_x(x)
    h_I, h_Nu, Z, Jg = _shah_hI_hNu(G, x, D, sat)
    if Jg >= 0.98 * (Z + 0.263) ** -0.62:
        return h_I, "I"
    return h_I + h_Nu, "II"


def condensacao_shah_1979(G: float, x: float, D: float, sat) -> tuple[float, str]:
    """Shah (1979), Int. J. Heat Mass Transfer 22:547-556 (referência clássica; o autor a
    recomendou só para Re_LT > 350, Re_GT > 35 000 e velocidade do vapor > 3 m/s)."""
    x = _clip_x(x)
    Re_LT = G * D / sat.mul
    h_LT = 0.023 * Re_LT ** 0.8 * sat.Prl ** 0.4 * sat.kl / D
    return h_LT * ((1 - x) ** 0.8 + 3.8 * x ** 0.76 * (1 - x) ** 0.04 / sat.pr ** 0.38), "-"


def condensacao_chato_1962(D: float, sat, dT_parede: float) -> tuple[float, str]:
    """Condensação estratificada em tubo horizontal a baixa vazão (Re_GT < 35 000),
    Chato (1962), ASHRAE J. 4(2):52-60 (forma de Nusselt com constante 0,555):
        h = 0,555 [ρ_l (ρ_l − ρ_v) g k_l³ h'_lv / (μ_l D ΔT_p)]^0,25 ,  h'_lv = h_lv + 3/8 c_p,l ΔT_p
    ΔT_p = T_sat − T_parede (calculado iterativamente a partir do fluxo de calor)."""
    dT = max(dT_parede, 0.05)
    hlv = sat.hlv + 0.375 * sat.cpl * dT
    return 0.555 * (sat.rhol * (sat.rhol - sat.rhov) * g * sat.kl ** 3 * hlv / (sat.mul * D * dT)) ** 0.25, "estratificado"


# ============================================================================ ebulição
def ebulicao_gungor_winterton_1987(G: float, x: float, D: float, q: float, sat) -> float:
    """Ebulição convectiva saturada, Gungor & Winterton (1987), Chem. Eng. Res. Des. 65:148-156.

    h = E·h_l ;  h_l = 0,023 Re_l^0,8 Pr_l^0,4 k_l/D com Re_l = G(1−x)D/μ_l
    E = 1 + 3000 Bo^0,86 + 1,12 [x/(1−x)]^0,75 (ρ_l/ρ_v)^0,41 ,  Bo = q/(G h_lv)
    Tubo horizontal com Fr_lo = G²/(ρ_l² g D) < 0,05: E ← E · Fr_lo^(0,1 − 2 Fr_lo)
    (correção de estratificação: parte superior do tubo seca em fluxos mássicos baixos).
    """
    x = _clip_x(x)
    Re_l = G * (1.0 - x) * D / sat.mul
    h_l = 0.023 * Re_l ** 0.8 * sat.Prl ** 0.4 * sat.kl / D
    Bo = max(q, 0.0) / (G * sat.hlv)
    E = 1.0 + 3000.0 * Bo ** 0.86 + 1.12 * (x / (1.0 - x)) ** 0.75 * (sat.rhol / sat.rhov) ** 0.41
    Fr = G * G / (sat.rhol ** 2 * g * D)
    if Fr < 0.05:
        E *= Fr ** (0.1 - 2.0 * Fr)
    return E * h_l


def ebulicao_nucleada_cooper(q: float, pr: float, M_kg_kmol: float, Rp_um: float = 1.0) -> float:
    """Ebulição nucleada em piscina, Cooper (1984), Adv. Heat Transfer 16:157-239.
    h = 55 p_r^(0,12 − 0,2 log10 Rp) (−log10 p_r)^−0,55 M^−0,5 q^0,67 (q em W/m², M em kg/kmol)."""
    return (55.0 * pr ** (0.12 - 0.2 * math.log10(Rp_um)) * (-math.log10(pr)) ** -0.55
            * M_kg_kmol ** -0.5 * max(q, 1.0) ** 0.67)


def ebulicao_liu_winterton_1991(G: float, x: float, D: float, q: float, sat, M_kg_kmol: float) -> float:
    """Liu & Winterton (1991), Int. J. Heat Mass Transfer 34(11):2759-2766.

    h² = (F h_lo)² + (S h_nb)² ; h_lo Dittus-Boelter com o fluxo total como líquido;
    F = [1 + x Pr_l (ρ_l/ρ_v − 1)]^0,35 ; S = [1 + 0,055 F^0,1 Re_lo^0,16]^−1 ; h_nb de Cooper.
    Horizontal com Fr_lo < 0,05: F ← F·Fr_lo^(0,1−2Fr_lo), S ← S·Fr_lo^0,5.
    """
    x = _clip_x(x)
    Re_lo = G * D / sat.mul
    h_lo = 0.023 * Re_lo ** 0.8 * sat.Prl ** 0.4 * sat.kl / D
    F = (1.0 + x * sat.Prl * (sat.rhol / sat.rhov - 1.0)) ** 0.35
    S = 1.0 / (1.0 + 0.055 * F ** 0.1 * Re_lo ** 0.16)
    Fr = G * G / (sat.rhol ** 2 * g * D)
    if Fr < 0.05:
        F *= Fr ** (0.1 - 2.0 * Fr)
        S *= Fr ** 0.5
    h_nb = ebulicao_nucleada_cooper(q, sat.pr, M_kg_kmol)
    return math.sqrt((F * h_lo) ** 2 + (S * h_nb) ** 2)


def h_vapor_saturado(G: float, D: float, sat) -> float:
    """Vapor monofásico com o fluxo total (limite x → 1), Gnielinski."""
    Re = G * D / sat.muv
    return nusselt_gnielinski(Re, sat.Prv) * sat.kv / D


def ebulicao_com_secagem(G, x, D, q, sat, M_kg_kmol, correlacao="gungor_winterton_1987", x_di=0.90):
    """Coeficiente de ebulição com tratamento de secagem de parede (dryout) para x > x_di:
    interpolação linear entre h(x_di) e o vapor saturado em x = 1, no espírito do modelo de
    névoa de Wojtan, Ursenbacher & Thome (2005), Int. J. Heat Mass Transfer 48:2970-2985.
    Sem isso as correlações de ebulição superestimam h perto de x = 1."""
    def h_tp(xx):
        if correlacao == "gungor_winterton_1987":
            return ebulicao_gungor_winterton_1987(G, xx, D, q, sat)
        if correlacao == "liu_winterton_1991":
            return ebulicao_liu_winterton_1991(G, xx, D, q, sat, M_kg_kmol)
        raise ValueError(f"correlação de ebulição desconhecida: {correlacao}")

    if x <= x_di:
        return h_tp(x)
    h_di = h_tp(x_di)
    h_v = h_vapor_saturado(G, D, sat)
    w = (min(x, 1.0) - x_di) / (1.0 - x_di)
    return h_di + w * (h_v - h_di)


# ============================================================================ atrito
def gradiente_atrito_msh(G: float, x: float, D: float, sat) -> float:
    """−dP/dz por atrito bifásico [Pa/m], Müller-Steinhagen & Heck (1986),
    Chem. Eng. Process. 20:297-308:
        (dP/dz) = Λ (1−x)^(1/3) + B x³ ,  Λ = A + 2(B − A) x
    A e B: gradientes só-líquido e só-vapor com o fluxo total (64/Re ou Blasius)."""
    x = min(max(x, 0.0), 1.0)
    Re_lo = G * D / sat.mul
    Re_vo = G * D / sat.muv
    A = fator_atrito_blasius_laminar(Re_lo) * G * G / (2.0 * sat.rhol * D)
    B = fator_atrito_blasius_laminar(Re_vo) * G * G / (2.0 * sat.rhov * D)
    lam = A + 2.0 * (B - A) * x
    return lam * (1.0 - x) ** (1.0 / 3.0) + B * x ** 3


def fracao_vazio_rouhani_axelsson(x: float, G: float, sat) -> float:
    """Fração de vazio, Rouhani & Axelsson (1970) na forma para tubos horizontais
    (Steiner 1993, VDI Heat Atlas), usada também por Wojtan et al. (2005):
    α = (x/ρ_v) {[1 + 0,12(1−x)](x/ρ_v + (1−x)/ρ_l) + 1,18(1−x)[gσ(ρ_l−ρ_v)]^0,25 / (G ρ_l^0,5)}^−1
    """
    if x <= 0.0:
        return 0.0
    if x >= 1.0:
        return 1.0
    sigma = sat.sigma if math.isfinite(sat.sigma) else 0.0
    termo = ((1.0 + 0.12 * (1.0 - x)) * (x / sat.rhov + (1.0 - x) / sat.rhol)
             + 1.18 * (1.0 - x) * (g * sigma * (sat.rhol - sat.rhov)) ** 0.25 / (G * sat.rhol ** 0.5))
    return (x / sat.rhov) / termo


def fracao_vazio_homogenea(x: float, sat) -> float:
    if x <= 0.0:
        return 0.0
    if x >= 1.0:
        return 1.0
    return 1.0 / (1.0 + (1.0 - x) / x * sat.rhov / sat.rhol)


def fluxo_momento_especifico(x: float, alpha: float, sat) -> float:
    """[x²/(ρ_v α) + (1−x)²/(ρ_l (1−α))] — ΔP_acel = G² Δ(·) no modelo de fases separadas."""
    x = min(max(x, 0.0), 1.0)
    if x <= 0.0:
        return 1.0 / sat.rhol
    if x >= 1.0:
        return 1.0 / sat.rhov
    # limites α → 0 e α → 1 (títulos a um arredondamento de 0 ou 1): o termo da fase ausente se anula
    vap = x * x / (sat.rhov * alpha) if alpha > 1e-12 else 0.0
    liq = (1.0 - x) ** 2 / (sat.rhol * (1.0 - alpha)) if alpha < 1.0 - 1e-12 else 0.0
    return vap + liq

In [ ]:
%%writefile predimref/correlacoes/lado_ar.py
"""Lado do ar de trocadores tubo-aleta (aletas planas contínuas, tubos redondos escalonados).

Referências
-----------
* Wang, C.C., Chi, K.Y., Chang, C.J. (2000). Heat transfer and friction characteristics of
  plain fin-and-tube heat exchangers, part II: Correlation. Int. J. Heat Mass Transfer
  43:2693-2700. Base: 74 trocadores; j em ±15 % para 88,6 % e f em ±15 % para 85,1 % dos dados.
* Schmidt, T.E. (1949). Heat transfer calculations for extended surfaces. Refrig. Eng. 57:351-357
  (eficiência de aleta por raio equivalente; é a mesma usada na redução de dados de Wang et al.,
  o que mantém a correlação consistente).
* Kays, W.M., London, A.L. (1984). Compact Heat Exchangers, 3ª ed. (queda de pressão do núcleo).
"""
from __future__ import annotations

import math
from dataclasses import dataclass


@dataclass
class GeometriaAletada:
    """Geometria de trocador tubo-aleta. Todas as dimensões em metros."""
    Do: float = 9.52e-3            # diâmetro externo do tubo (3/8")
    esp_parede: float = 0.35e-3    # espessura da parede do tubo
    esp_aleta: float = 0.12e-3     # δf
    passo_aleta: float = 2.5e-3    # Fp (distância entre aletas, centro a centro)
    Pt: float = 25.4e-3            # passo transversal (perpendicular ao ar)
    Pl: float = 22.0e-3            # passo longitudinal (na direção do ar)
    n_fileiras: int = 3            # N, fileiras na direção do ar
    k_aleta: float = 222.0         # alumínio 1100 [W/mK]
    k_tubo: float = 390.0          # cobre [W/mK]
    rugosidade_int: float = 1.5e-6  # rugosidade interna do tubo de cobre [m]

    # ---------------------------------------------------------------- derivadas
    @property
    def Dc(self) -> float:
        """Diâmetro do colar da aleta, Dc = Do + 2δf (dimensão característica de Wang)."""
        return self.Do + 2.0 * self.esp_aleta

    @property
    def Di(self) -> float:
        return self.Do - 2.0 * self.esp_parede

    @property
    def area_aleta_por_m(self) -> float:
        """Área de aleta (duas faces) por metro de tubo [m²/m]."""
        return 2.0 * (self.Pt * self.Pl - math.pi * self.Dc ** 2 / 4.0) / self.passo_aleta

    @property
    def area_tubo_exposta_por_m(self) -> float:
        return math.pi * self.Dc * (1.0 - self.esp_aleta / self.passo_aleta)

    @property
    def area_externa_por_m(self) -> float:
        """Ao por metro de tubo [m²/m]."""
        return self.area_aleta_por_m + self.area_tubo_exposta_por_m

    @property
    def area_interna_por_m(self) -> float:
        return math.pi * self.Di

    @property
    def sigma(self) -> float:
        """Razão área mínima de escoamento / área de face (tubos escalonados com Pl usual,
        a área mínima é a transversal; ver verificação em ``area_minima_diagonal_ok``)."""
        return (self.Pt - self.Dc) * (self.passo_aleta - self.esp_aleta) / (self.Pt * self.passo_aleta)

    @property
    def area_minima_diagonal_ok(self) -> bool:
        """Verdadeiro se a passagem diagonal não é a mínima: 2·(Sd − Dc) ≥ (Pt − Dc)."""
        Sd = math.sqrt((self.Pt / 2.0) ** 2 + self.Pl ** 2)
        return 2.0 * (Sd - self.Dc) >= (self.Pt - self.Dc)

    @property
    def Dh(self) -> float:
        """Diâmetro hidráulico do lado do ar, Dh = 4 Amin L / Ao (definição de Wang et al.)."""
        amin_por_m = (self.Pt - self.Dc) * (1.0 - self.esp_aleta / self.passo_aleta)
        return 4.0 * amin_por_m * self.Pl / self.area_externa_por_m

    @property
    def fracao_aleta(self) -> float:
        return self.area_aleta_por_m / self.area_externa_por_m

    def resistencia_parede_por_m(self) -> float:
        """Resistência condutiva da parede do tubo por metro [m·K/W]."""
        return math.log(self.Do / self.Di) / (2.0 * math.pi * self.k_tubo)

    def verificar_faixa_wang(self) -> list[str]:
        """Faixa aproximada da base de dados de Wang et al. (2000)."""
        av = []
        fp_mm, dc_mm = self.passo_aleta * 1e3, self.Dc * 1e3
        if not 1.19 <= fp_mm <= 8.7:
            av.append(f"passo de aleta {fp_mm:.2f} mm fora de 1,19–8,7 mm (base de Wang 2000)")
        if not 6.35 <= dc_mm <= 12.7:
            av.append(f"diâmetro de colar {dc_mm:.2f} mm fora de 6,35–12,7 mm (base de Wang 2000)")
        if not 17.7 <= self.Pt * 1e3 <= 31.75:
            av.append(f"Pt {self.Pt * 1e3:.1f} mm fora de 17,7–31,75 mm (base de Wang 2000)")
        if not 12.4 <= self.Pl * 1e3 <= 27.5:
            av.append(f"Pl {self.Pl * 1e3:.1f} mm fora de 12,4–27,5 mm (base de Wang 2000)")
        if not 1 <= self.n_fileiras <= 6:
            av.append(f"{self.n_fileiras} fileiras fora de 1–6 (base de Wang 2000)")
        if not self.area_minima_diagonal_ok:
            av.append("a passagem diagonal é a área mínima: σ calculado é otimista")
        return av


def wang_2000_j_f(Re_Dc: float, geo: GeometriaAletada) -> tuple[float, float]:
    """Fatores j de Colburn e f de Fanning do núcleo, Wang, Chi & Chang (2000).

    N = 1:  j = 0,108 Re^−0,29 (Pt/Pl)^P1 (Fp/Dc)^−1,084 (Fp/Dh)^−0,786 (Fp/Pt)^P2
            P1 = 1,9 − 0,23 ln Re ;  P2 = −0,236 + 0,126 ln Re
    N ≥ 2:  j = 0,086 Re^P3 N^P4 (Fp/Dc)^P5 (Fp/Dh)^P6 (Fp/Pt)^−0,93
            P3 = −0,361 − 0,042 N/ln Re + 0,158 ln[N (Fp/Dc)^0,41]
            P4 = −1,224 − 0,076 (Pl/Dh)^1,42 / ln Re
            P5 = −0,083 + 0,058 N/ln Re
            P6 = −5,735 + 1,21 ln(Re/N)
    f = 0,0267 Re^F1 (Pt/Pl)^F2 (Fp/Dc)^F3
            F1 = −0,764 + 0,739 Pt/Pl + 0,177 Fp/Dc − 0,00758/N
            F2 = −15,689 + 64,021/ln Re ;  F3 = 1,696 − 15,695/ln Re
    Re = ρ V_max Dc / μ. Nota: algumas transcrições omitem o termo 0,158 ln[N(Fp/Dc)^0,41] de P3;
    sem ele j cai 4–5× de N=1 para N=6, o que contraria os próprios dados (j quase
    independente de N para Re > 2000) — ver tests/test_correlacoes.py.
    """
    Fp, Dc, Dh, Pt, Pl, N = geo.passo_aleta, geo.Dc, geo.Dh, geo.Pt, geo.Pl, geo.n_fileiras
    lnR = math.log(Re_Dc)
    if N == 1:
        P1 = 1.9 - 0.23 * lnR
        P2 = -0.236 + 0.126 * lnR
        j = 0.108 * Re_Dc ** -0.29 * (Pt / Pl) ** P1 * (Fp / Dc) ** -1.084 * (Fp / Dh) ** -0.786 * (Fp / Pt) ** P2
    else:
        P3 = -0.361 - 0.042 * N / lnR + 0.158 * math.log(N * (Fp / Dc) ** 0.41)
        P4 = -1.224 - 0.076 * (Pl / Dh) ** 1.42 / lnR
        P5 = -0.083 + 0.058 * N / lnR
        P6 = -5.735 + 1.21 * math.log(Re_Dc / N)
        j = 0.086 * Re_Dc ** P3 * N ** P4 * (Fp / Dc) ** P5 * (Fp / Dh) ** P6 * (Fp / Pt) ** -0.93
    F1 = -0.764 + 0.739 * Pt / Pl + 0.177 * Fp / Dc - 0.00758 / N
    F2 = -15.689 + 64.021 / lnR
    F3 = 1.696 - 15.695 / lnR
    f = 0.0267 * Re_Dc ** F1 * (Pt / Pl) ** F2 * (Fp / Dc) ** F3
    return j, f


def eficiencia_aleta_schmidt(h: float, geo: GeometriaAletada) -> float:
    """Eficiência de aleta plana em tubos escalonados (hexágono), Schmidt (1949):
    η = tanh(m r φ)/(m r φ), m = (2h/kδ)^0,5, φ = (Req/r − 1)[1 + 0,35 ln(Req/r)],
    Req/r = 1,27 ψ (β − 0,3)^0,5, ψ = X_M/r, β = X_L/X_M,
    X_M = Pt/2, X_L = ½[(Pt/2)² + Pl²]^0,5, r = Dc/2."""
    r = geo.Dc / 2.0
    XM = geo.Pt / 2.0
    XL = 0.5 * math.sqrt((geo.Pt / 2.0) ** 2 + geo.Pl ** 2)
    beta = XL / XM
    Req_r = 1.27 * (XM / r) * math.sqrt(max(beta - 0.3, 1e-9))
    phi = (Req_r - 1.0) * (1.0 + 0.35 * math.log(Req_r))
    m = math.sqrt(2.0 * h / (geo.k_aleta * geo.esp_aleta))
    z = m * r * phi
    return 1.0 if z < 1e-9 else math.tanh(z) / z


@dataclass
class DesempenhoAr:
    Re_Dc: float
    j: float
    f: float
    h: float          # coeficiente convectivo [W/m²K]
    eta_aleta: float
    eta_superficie: float
    G_max: float      # fluxo mássico na área mínima [kg/m²s]


def desempenho_lado_ar(V_face: float, rho: float, cp: float, mu: float, Pr: float,
                       geo: GeometriaAletada, fator_umidade: float = 1.0) -> DesempenhoAr:
    """h do lado do ar (Wang 2000) e eficiência global de superfície.

    ``fator_umidade`` ξ = Q_total/Q_sensível ≥ 1 aplica a analogia de Lewis para serpentina
    com desumidificação (Threlkeld 1970; McQuiston 1978): h_eq = ξ·h na eficiência da aleta
    e no fluxo de calor total."""
    G_max = rho * V_face / geo.sigma
    Re = G_max * geo.Dc / mu
    j, f = wang_2000_j_f(Re, geo)
    h = j * G_max * cp / Pr ** (2.0 / 3.0)
    eta_f = eficiencia_aleta_schmidt(h * fator_umidade, geo)
    eta_o = 1.0 - geo.fracao_aleta * (1.0 - eta_f)
    return DesempenhoAr(Re, j, f, h, eta_f, eta_o, G_max)


def queda_pressao_ar(G_max: float, f: float, geo: GeometriaAletada, rho_in: float,
                     rho_out: float) -> float:
    """ΔP do núcleo [Pa], Kays & London (mesma expressão da redução de dados de Wang):
    ΔP = G²/(2ρ_in) [ (1+σ²)(ρ_in/ρ_out − 1) + f (Ao/Amin)(ρ_in/ρ_m) ]"""
    rho_m = 2.0 / (1.0 / rho_in + 1.0 / rho_out)
    Ao_Amin = geo.area_externa_por_m * geo.n_fileiras / (geo.sigma * geo.Pt)
    return G_max ** 2 / (2.0 * rho_in) * (
        (1.0 + geo.sigma ** 2) * (rho_in / rho_out - 1.0) + f * Ao_Amin * rho_in / rho_m)

In [ ]:
%%writefile predimref/gabinete.py
"""Capacidade do gabinete (litros <-> garrafas) e carga térmica de refrigeração.

Componentes da carga
--------------------
1. Transmissão pelas paredes isoladas: resistências em série (convecção interna,
   condução na espuma, convecção externa) com fatores de forma de condução para arestas
   (S = 0,54 L) e cantos (S = 0,15 δ) — Incropera et al., Fundamentals of Heat and Mass
   Transfer, Tab. 4.1 — mais um fator de pontes térmicas (parafusos, passagens de tubos, dreno).
2. Porta de vidro (U global com moldura) e gaxeta (transmitância linear ψ no perímetro).
3. Infiltração por abertura de porta: vazão de Gosney & Olama (1975), a mesma da ASHRAE
   Handbook—Refrigeration (cap. "Refrigeration Load"):
       I = 0,221 A (gH)^0,5 (1 − ρ_e/ρ_i)^0,5 F_m ,  F_m = [2/(1 + (ρ_i/ρ_e)^(1/3))]^1,5
   com fator de fluxo da porta D_f (0,8 para ΔT > 11 K; 1,1 abaixo). Em gabinetes pequenos
   o fluxo estabelecido de Gosney-Olama trocaria várias vezes o volume interno numa abertura
   de 8 s, o que é impossível; a troca por abertura é limitada ao volume de ar livre
   (volume interno menos o volume das garrafas) vezes um número máximo de renovações.
4. Produto: pull-down da carga completa e reposição diária; cp da bebida por
   Choi & Okos (1986), recomendado na ASHRAE Handbook—Refrigeration (cap. "Thermal
   Properties of Foods").
5. Cargas internas: iluminação, ventilador do evaporador, resistência anti-embaçante.

A carga de projeto é o maior entre (a) regime diário dividido pelo fator de utilização
máximo do compressor e (b) pull-down com o compressor ligado 100 % do tempo, mais a margem.
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field
from typing import Optional

from .avisos import Avisos, ErroProjeto
from .constantes import G, SEGUNDOS_DIA, C_para_K
from .propriedades import ar_umido, calor_latente_agua


# ============================================================================ produto
@dataclass
class Garrafa:
    """Garrafa PET de bebida (padrão: 600 mL de refrigerante de cola com açúcar).

    Dimensões padrão de uma garrafa genérica de 600 mL (≈ 65 mm × 239 mm). Para precisão,
    meça a garrafa usada pelo cliente: o volume do envelope entra direto na conversão
    litros <-> garrafas."""
    volume_nominal_L: float = 0.600
    diametro_m: float = 0.065
    altura_m: float = 0.239
    densidade_bebida: float = 1040.0      # kg/m³ (refrigerante com ~10,6 °Brix)
    acucar_g_por_L: float = 106.0          # 0 para versões sem açúcar
    massa_embalagem_kg: float = 0.025      # PET + tampa
    cp_embalagem: float = 1100.0           # PET [J/kgK]
    fator_volume_deslocado: float = 1.10   # volume externo / volume nominal (PET + espaço livre)

    @property
    def volume_envelope_m3(self) -> float:
        """Prisma de base quadrada ocupado por garrafa em arranjo retangular (D² H)."""
        return self.diametro_m ** 2 * self.altura_m

    @property
    def volume_deslocado_m3(self) -> float:
        return self.volume_nominal_L * 1e-3 * self.fator_volume_deslocado

    @property
    def massa_bebida_kg(self) -> float:
        return self.volume_nominal_L * 1e-3 * self.densidade_bebida

    @property
    def fracao_carboidrato(self) -> float:
        return self.acucar_g_por_L / self.densidade_bebida

    def cp_bebida(self, T_C: float) -> float:
        """cp [J/kgK] por Choi & Okos (1986): soma ponderada de água e carboidrato."""
        xc = self.fracao_carboidrato
        cp_w = 4.1762 - 9.0864e-5 * T_C + 5.4731e-6 * T_C ** 2
        cp_c = 1.5488 + 1.9625e-3 * T_C - 5.9399e-6 * T_C ** 2
        return 1000.0 * ((1.0 - xc) * cp_w + xc * cp_c)

    def energia_resfriamento(self, T1_C: float, T2_C: float) -> float:
        """Energia [J] para resfriar uma garrafa cheia de T1 a T2 (cp integrado por Simpson)."""
        Tm = 0.5 * (T1_C + T2_C)
        cp_m = (self.cp_bebida(T1_C) + 4.0 * self.cp_bebida(Tm) + self.cp_bebida(T2_C)) / 6.0
        return (self.massa_bebida_kg * cp_m + self.massa_embalagem_kg * self.cp_embalagem) * (T1_C - T2_C)


# ============================================================================ gabinete
@dataclass
class Gabinete:
    """Construção do gabinete vertical de porta de vidro (visi-cooler)."""
    dimensoes_internas_m: Optional[tuple] = None  # (largura, profundidade, altura) — opcional
    razao_altura_largura: float = 2.4
    razao_profundidade_largura: float = 0.85
    fator_ocupacao: float = 0.55        # Σ envelope das garrafas / volume interno (carga cheia)
    fator_ocupacao_max: float = 0.70    # limite físico (prateleiras, dutos, folgas)
    esp_lateral_m: float = 0.050
    esp_fundo_m: float = 0.050
    esp_topo_m: float = 0.050
    esp_base_m: float = 0.060
    k_isolamento: float = 0.022          # PU expandido com ciclopentano [W/mK]
    h_interno: float = 10.0              # convecção forçada interna [W/m²K]
    h_externo: float = 7.7               # convecção natural + radiação (ISO 6946: Rs = 0,13)
    U_porta: float = 2.0                 # porta de vidro duplo low-e com moldura [W/m²K]
    psi_gaxeta: float = 0.10             # transmitância linear da região da gaxeta [W/mK]
    fator_pontes_termicas: float = 1.10  # multiplicador sobre as paredes isoladas
    dT_compartimento_maquina: float = 5.0  # base sobre o compartimento do compressor [K]


@dataclass
class Uso:
    """Perfil de uso e premissas de carga."""
    UR_interna: float = 0.75                      # ponto de partida (ou valor fixo, ver umidade_interna)
    umidade_interna: str = "equilibrio"           # "equilibrio": fecha o balanço infiltração × serpentina
    aberturas_por_dia: float = 100.0
    tempo_abertura_s: float = 8.0
    fator_fluxo_porta: Optional[float] = None   # D_f; None -> 0,8 (ΔT > 11 K) ou 1,1
    renovacoes_max_por_abertura: float = 1.0
    T_produto_entrada_C: Optional[float] = None  # None -> temperatura ambiente
    tempo_pulldown_h: float = 24.0
    fracao_reposicao_diaria: float = 0.5         # fração das garrafas trocada por dia
    massa_adicional_kg: float = 0.0              # prateleiras etc. no pull-down
    cp_adicional: float = 500.0                  # aço [J/kgK]
    potencia_iluminacao_W: float = 10.0
    potencia_ventilador_evap_W: float = 8.0
    potencia_antiembacante_W: float = 0.0
    fracao_antiembacante_interna: float = 0.5
    fator_utilizacao_max: float = 0.70           # fração do tempo ligado em regime
    margem_seguranca: float = 0.10


@dataclass
class Capacidade:
    volume_interno_L: float
    n_garrafas: int
    largura_m: float
    profundidade_m: float
    altura_m: float
    volume_ar_livre_L: float
    massa_produto_kg: float
    origem: str


def resolver_capacidade(volume_L: Optional[float], n_garrafas: Optional[int], garrafa: Garrafa,
                        gab: Gabinete, avisos: Avisos) -> Capacidade:
    """Converte volume interno <-> número de garrafas e define as dimensões internas.

    Atenção à conversão ingênua 'litros / 0,6 L': ela dá o volume de LÍQUIDO, não o volume
    de gabinete necessário. Cada garrafa ocupa o envelope D²H e só uma fração do volume
    interno (``fator_ocupacao``) é aproveitável (prateleiras, folgas de ar, evaporador)."""
    Venv = garrafa.volume_envelope_m3
    if gab.dimensoes_internas_m is not None:
        W, D, H = gab.dimensoes_internas_m
        V_dims = W * D * H
        if volume_L is not None and abs(V_dims * 1e3 - volume_L) / volume_L > 0.05:
            avisos.atencao("capacidade", f"dimensões internas dão {V_dims * 1e3:.0f} L, diferente dos "
                           f"{volume_L:.0f} L informados; usando as dimensões.")
        volume_L = V_dims * 1e3
    if volume_L is None and n_garrafas is None:
        raise ErroProjeto("Informe o volume interno [L] ou o número de garrafas.")

    if volume_L is not None and n_garrafas is None:
        V = volume_L * 1e-3
        n = int(math.floor(gab.fator_ocupacao * V / Venv))
        origem = "volume informado; garrafas estimadas pelo fator de ocupação"
    elif volume_L is None:
        n = int(n_garrafas)
        V = n * Venv / gab.fator_ocupacao
        origem = "garrafas informadas; volume estimado pelo fator de ocupação"
    else:
        V, n = volume_L * 1e-3, int(n_garrafas)
        origem = "volume e garrafas informados"
        ocup = n * Venv / V
        if ocup > gab.fator_ocupacao_max:
            n_max = int(gab.fator_ocupacao_max * V / Venv)
            avisos.critico("capacidade", f"{n} garrafas não cabem em {volume_L:.0f} L (ocupação "
                           f"{ocup:.0%} > {gab.fator_ocupacao_max:.0%}). Máximo físico ≈ {n_max} garrafas; "
                           f"para {n} garrafas são necessários ≈ {n * Venv / gab.fator_ocupacao * 1e3:.0f} L.")
    vol_liquido_L = n * garrafa.volume_nominal_L
    if abs(vol_liquido_L - V * 1e3) / (V * 1e3) < 0.05:
        avisos.atencao("capacidade", f"{n} garrafas × {garrafa.volume_nominal_L * 1e3:.0f} mL = "
                       f"{vol_liquido_L:.0f} L de líquido ≈ volume interno: parece uma conversão por volume de "
                       "líquido. O gabinete precisa de mais volume que o líquido que guarda.")

    if gab.dimensoes_internas_m is not None:
        W, D, H = gab.dimensoes_internas_m
    else:
        W = (V / (gab.razao_altura_largura * gab.razao_profundidade_largura)) ** (1.0 / 3.0)
        D, H = gab.razao_profundidade_largura * W, gab.razao_altura_largura * W
    V_livre = V - n * garrafa.volume_deslocado_m3
    if V_livre <= 0.1 * V:
        avisos.atencao("capacidade", "volume de ar livre < 10 % do interno: circulação de ar comprometida.")
        V_livre = max(V_livre, 0.1 * V)
    massa = n * (garrafa.massa_bebida_kg + garrafa.massa_embalagem_kg)
    return Capacidade(V * 1e3, n, W, D, H, V_livre * 1e3, massa, origem)


# ============================================================================ transmissão
@dataclass
class Face:
    nome: str
    area_int: float
    area_ext: float
    espessura: float
    fator_forma: float
    UA: float
    dT: float
    Q: float


def transmissao(cap: Capacidade, gab: Gabinete, T_amb: float, T_int: float) -> dict:
    """Carga de transmissão [W] pelas cinco faces isoladas, porta de vidro e gaxeta."""
    W, D, H = cap.largura_m, cap.profundidade_m, cap.altura_m
    dT = T_amb - T_int
    faces = {
        "topo": (W, D, gab.esp_topo_m, 0.0),
        "base": (W, D, gab.esp_base_m, gab.dT_compartimento_maquina),
        "lateral_esq": (D, H, gab.esp_lateral_m, 0.0),
        "lateral_dir": (D, H, gab.esp_lateral_m, 0.0),
        "fundo": (W, H, gab.esp_fundo_m, 0.0),
    }
    arestas = [("fundo", "topo", W), ("fundo", "base", W), ("fundo", "lateral_esq", H),
               ("fundo", "lateral_dir", H), ("topo", "lateral_esq", D), ("topo", "lateral_dir", D),
               ("base", "lateral_esq", D), ("base", "lateral_dir", D)]
    cantos = [("fundo", "topo", "lateral_esq"), ("fundo", "topo", "lateral_dir"),
              ("fundo", "base", "lateral_esq"), ("fundo", "base", "lateral_dir")]
    S = {n: a * b / esp for n, (a, b, esp, _) in faces.items()}
    for f1, f2, L in arestas:            # S_aresta = 0,54 L, dividido entre as duas faces
        S[f1] += 0.27 * L
        S[f2] += 0.27 * L
    for trio in cantos:                  # S_canto = 0,15 δ, dividido entre as três faces
        esp_m = sum(faces[n][2] for n in trio) / 3.0
        for n in trio:
            S[n] += 0.05 * esp_m
    lista = []
    for nome, (a, b, esp, dT_extra) in faces.items():
        Ai, Ae = a * b, (a + 2 * esp) * (b + 2 * esp)
        R = 1.0 / (gab.h_interno * Ai) + 1.0 / (gab.k_isolamento * S[nome]) + 1.0 / (gab.h_externo * Ae)
        UA = gab.fator_pontes_termicas / R
        lista.append(Face(nome, Ai, Ae, esp, S[nome], UA, dT + dT_extra, UA * (dT + dT_extra)))
    A_porta = W * H
    Q_paredes = sum(f.Q for f in lista)
    Q_porta = gab.U_porta * A_porta * dT
    perimetro = 2.0 * (W + H)
    Q_gaxeta = gab.psi_gaxeta * perimetro * dT
    return {"faces": lista, "Q_paredes": Q_paredes, "Q_porta": Q_porta, "Q_gaxeta": Q_gaxeta,
            "A_porta": A_porta, "perimetro_porta": perimetro,
            "UA_total": sum(f.UA for f in lista) + gab.U_porta * A_porta + gab.psi_gaxeta * perimetro,
            "Q_total": Q_paredes + Q_porta + Q_gaxeta}


# ============================================================================ infiltração
def infiltracao(cap: Capacidade, uso: Uso, T_amb: float, UR_amb: float, T_int: float,
                avisos: Avisos) -> dict:
    """Carga média de infiltração [W] por abertura de porta (sensível + latente)."""
    ext, inn = ar_umido(T_amb, UR_amb), ar_umido(T_int, uso.UR_interna)
    rho_e, rho_i = ext["rho_mist"], inn["rho_mist"]
    zero = {"Q_media": 0.0, "Q_sensivel": 0.0, "Q_latente": 0.0, "agua_kg_dia": 0.0,
            "m_ar_abertura": 0.0, "limitado_por_volume": False, "V_gosney_olama_m3": 0.0}
    if rho_i <= rho_e or uso.aberturas_por_dia <= 0:
        return zero
    A, H = cap.largura_m * cap.altura_m, cap.altura_m
    Fm = (2.0 / (1.0 + (rho_i / rho_e) ** (1.0 / 3.0))) ** 1.5
    I = 0.221 * A * math.sqrt(G * H) * math.sqrt(1.0 - rho_e / rho_i) * Fm   # m³/s de ar frio
    dT = T_amb - T_int
    Df = uso.fator_fluxo_porta if uso.fator_fluxo_porta is not None else (0.8 if dT > 11.0 else 1.1)
    V_GO = I * Df * uso.tempo_abertura_s
    m_GO = V_GO / inn["v"]                                    # kg de ar seco (ASHRAE usa ρ do ar frio)
    m_lim = uso.renovacoes_max_por_abertura * cap.volume_ar_livre_L * 1e-3 / ext["v"]
    m_ab = min(m_GO, m_lim)
    limitado = m_lim < m_GO
    if limitado:
        avisos.info("infiltração", f"Gosney-Olama trocaria {m_GO / m_lim:.1f}× o ar livre por abertura; "
                    f"troca limitada a {uso.renovacoes_max_por_abertura:g} renovação(ões) do volume livre.")
    q_ab = m_ab * (ext["h"] - inn["h"])
    dW = max(ext["W"] - inn["W"], 0.0)
    q_lat_ab = m_ab * dW * calor_latente_agua(T_int)
    n = uso.aberturas_por_dia
    return {"Q_media": n * q_ab / SEGUNDOS_DIA, "Q_latente": n * q_lat_ab / SEGUNDOS_DIA,
            "Q_sensivel": n * (q_ab - q_lat_ab) / SEGUNDOS_DIA, "agua_kg_dia": n * m_ab * dW,
            "m_ar_abertura": m_ab, "limitado_por_volume": limitado, "V_gosney_olama_m3": V_GO,
            "vazao_gosney_olama_m3s": I, "Df": Df, "Tdp_interno_K": inn["Tdp"]}


# ============================================================================ total
@dataclass
class CargaTermica:
    capacidade: Capacidade
    transmissao: dict
    infiltracao: dict
    Q_paredes: float
    Q_porta: float
    Q_gaxeta: float
    Q_infiltracao: float
    Q_internas: float
    Q_reposicao: float
    Q_pulldown_produto: float
    Q_regime: float              # média diária com o uso definido [W]
    Q_requerida_regime: float    # Q_regime / fator de utilização
    Q_pulldown: float            # pull-down com compressor 100 % ligado
    caso_governante: str
    Q_projeto: float             # capacidade frigorífica de projeto do evaporador [W]
    Q_latente_projeto: float
    fator_umidade: float         # ξ = Q_total / Q_sensível no evaporador
    componentes: dict = field(default_factory=dict)

    def tabela(self) -> list[tuple[str, float]]:
        return list(self.componentes.items())


def cargas_internas(uso: Uso) -> float:
    """Iluminação, ventilador do evaporador e a parte interna da resistência anti-embaçante [W]."""
    return (uso.potencia_iluminacao_W + uso.potencia_ventilador_evap_W
            + uso.fracao_antiembacante_interna * uso.potencia_antiembacante_W)


def calcular_carga(cap: Capacidade, gab: Gabinete, uso: Uso, garrafa: Garrafa, T_amb_C: float,
                   UR_amb: float, T_alvo_C: float, avisos: Avisos) -> CargaTermica:
    T_amb, T_int = C_para_K(T_amb_C), C_para_K(T_alvo_C)
    if T_int >= T_amb:
        raise ErroProjeto("A temperatura alvo deve ser menor que a ambiente.")
    tr = transmissao(cap, gab, T_amb, T_int)
    inf = infiltracao(cap, uso, T_amb, UR_amb, T_int, avisos)
    Q_int = cargas_internas(uso)
    T_prod = T_amb_C if uso.T_produto_entrada_C is None else uso.T_produto_entrada_C
    E_garrafa = garrafa.energia_resfriamento(T_prod, T_alvo_C)
    E_cheia = cap.n_garrafas * E_garrafa + uso.massa_adicional_kg * uso.cp_adicional * (T_prod - T_alvo_C)
    Q_rep = uso.fracao_reposicao_diaria * cap.n_garrafas * E_garrafa / SEGUNDOS_DIA
    Q_pd_prod = E_cheia / (uso.tempo_pulldown_h * 3600.0)

    Q_regime = tr["Q_total"] + inf["Q_media"] + Q_int + Q_rep
    Q_req_reg = Q_regime / uso.fator_utilizacao_max
    Q_pd = tr["Q_total"] + Q_int + Q_pd_prod      # porta fechada no pull-down
    if Q_req_reg >= Q_pd:
        caso, Q_base, Q_lat = "regime (uso diário / fator de utilização)", Q_req_reg, inf["Q_latente"] / uso.fator_utilizacao_max
    else:
        caso, Q_base, Q_lat = "pull-down da carga completa", Q_pd, 0.0
    Q_proj = Q_base * (1.0 + uso.margem_seguranca)
    Q_lat_proj = Q_lat * (1.0 + uso.margem_seguranca)
    xi = Q_proj / max(Q_proj - Q_lat_proj, 1e-9)
    comp = {"paredes isoladas": tr["Q_paredes"], "porta de vidro": tr["Q_porta"], "gaxeta": tr["Q_gaxeta"],
            "infiltração (média)": inf["Q_media"], "cargas internas": Q_int,
            "reposição diária de produto": Q_rep, "pull-down do produto": Q_pd_prod}
    if uso.fator_utilizacao_max > 0.85:
        avisos.atencao("carga", "fator de utilização > 85 % deixa pouca folga para degelo e controle.")
    return CargaTermica(cap, tr, inf, tr["Q_paredes"], tr["Q_porta"], tr["Q_gaxeta"], inf["Q_media"], Q_int,
                        Q_rep, Q_pd_prod, Q_regime, Q_req_reg, Q_pd, caso, Q_proj, Q_lat_proj, xi, comp)


__all__ = ["Garrafa", "Gabinete", "Uso", "Capacidade", "CargaTermica", "resolver_capacidade",
           "transmissao", "infiltracao", "cargas_internas", "calcular_carga"]

In [ ]:
%%writefile predimref/ciclo.py
"""Ciclo de compressão de vapor subcrítico com superaquecimento, sub-resfriamento,
trocador interno (linha de sucção / capilar) opcional, e verificação termodinâmica do COP.

Pontos (numeração usada em todo o pacote)
  1   saída do evaporador (P_evap, T_evap + SA)
  1s  sucção do compressor (após o trocador interno, se houver)
  2s  descarga isentrópica;  2 descarga real (P_cond)
  3   saída do condensador (P_cond − ΔP_cond, T_sat − SR)
  3i  entrada do capilar (após o trocador interno)
  4   entrada do evaporador (P_evap + ΔP_evap, h4 = h3i: capilar adiabático)

Definições de rendimento do compressor
  η_global = ṁ (h2s − h1s) / W_el           (isentrópico global, referido à potência elétrica)
  η_is     = (h2s − h1s) / (h2 − h1s)       (isentrópico interno, define a temperatura de descarga)
  η_em     = η_global / η_is                (eletromecânico: motor × mecânico)
  Calor perdido pela carcaça: Q_carc = W_el − ṁ (h2 − h1s) ≥ 0.

Com o COP informado, η_global = COP / COP_ciclo_ideal, onde COP_ciclo_ideal é o COP do mesmo
ciclo com compressão isentrópica. COP > COP_ciclo_ideal (ou > Carnot) é impossível.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Optional

from scipy.optimize import brentq

from .avisos import Avisos, ErroProjeto
from .propriedades import Fluido


@dataclass
class ParametrosCiclo:
    dT_evap: float = 10.0            # T_ar entrada evaporador − T_evap [K]
    dT_cond: float = 13.0            # T_cond − T_ar entrada condensador [K]
    dT_recirculacao_cond: float = 0.0  # aquecimento do ar no compartimento antes do condensador [K]
    dT_retorno_evap: float = 0.0     # ar de retorno ao evaporador − T alvo [K]
    superaquecimento: float = 5.0    # na saída do evaporador [K]
    subresfriamento: float = 3.0     # na saída do condensador [K]
    eficacia_ihx: float = 0.0        # trocador capilar/linha de sucção (0 = sem)
    eta_eletromecanica: float = 0.72  # motor × mecânico, compressor hermético pequeno
    eta_global_sem_cop: float = 0.55  # usado só quando o COP não é informado
    T_descarga_max_C: float = 130.0
    eta_global_faixa: tuple = (0.35, 0.70)  # faixa realista para herméticos alternativos pequenos


@dataclass
class Ciclo:
    fluido: str
    T_evap: float
    T_cond: float
    estados: dict
    COP: float
    COP_ciclo_ideal: float      # compressão isentrópica, mesmas temperaturas
    COP_carnot_ciclo: float     # entre T_evap e T_cond
    eta_global: float
    eta_is: float
    eta_em: float
    q_evap: float               # J/kg
    w_s: float                  # h2s − h1s
    q_ihx: float
    razao_pressao: float
    extras: dict = field(default_factory=dict)

    @property
    def w_ref(self) -> float:
        """Trabalho específico entregue ao refrigerante (h2 − h1s)."""
        return self.estados["2"].h - self.estados["1s"].h


def resolver_ciclo(fl: Fluido, T_evap: float, T_cond: float, p: ParametrosCiclo,
                   cop: Optional[float], dP_cond: float = 0.0, dP_evap: float = 0.0,
                   avisos: Optional[Avisos] = None) -> Ciclo:
    av = avisos if avisos is not None else Avisos()
    if T_cond >= fl.Tc - 5.0:
        raise ErroProjeto(f"T_cond = {T_cond - 273.15:.1f} °C muito próxima da crítica do {fl.nome} "
                          f"({fl.Tc - 273.15:.1f} °C): ciclo subcrítico inválido.")
    if T_evap <= fl.Ttriplo + 5:
        raise ErroProjeto("T_evap abaixo da faixa válida do fluido.")
    P_evap, P_cond = fl.Psat(T_evap), fl.Psat(T_cond)
    if p.superaquecimento > 0:
        e1 = fl.estado_PT(P_evap, T_evap + p.superaquecimento)
    else:
        e1 = fl.estado_PQ(P_evap, 1.0)
    P3 = P_cond - dP_cond
    T3sat = fl.Tsat(P3)
    e3 = fl.estado_PT(P3, T3sat - p.subresfriamento) if p.subresfriamento > 0 else fl.estado_PQ(P3, 0.0)
    # trocador interno: a corrente de vapor (menor capacidade térmica) é a de referência
    q_ihx = 0.0
    if p.eficacia_ihx > 0:
        h_v_max = fl.estado_PT(P_evap, e3.T).h
        q_ihx = p.eficacia_ihx * (h_v_max - e1.h)
    e1s = fl.estado_PH(P_evap, e1.h + q_ihx)
    e3i = fl.estado_PH(P3, e3.h - q_ihx)
    e4 = fl.estado_PH(P_evap + dP_evap, e3i.h)
    e2s = fl.estado_PS(P_cond, e1s.s)
    w_s = e2s.h - e1s.h
    q_evap = e1.h - e4.h
    cop_ideal = q_evap / w_s
    cop_carnot = T_evap / (T_cond - T_evap)
    if cop is not None:
        if cop >= cop_carnot:
            raise ErroProjeto(f"COP {cop:.2f} ≥ COP de Carnot {cop_carnot:.2f} entre T_evap = "
                              f"{T_evap - 273.15:.1f} °C e T_cond = {T_cond - 273.15:.1f} °C: viola a 2ª lei.")
        if cop >= cop_ideal:
            raise ErroProjeto(f"COP {cop:.2f} ≥ COP do ciclo com compressão isentrópica ({cop_ideal:.2f}): "
                              "impossível com essas temperaturas. Reduza as diferenças de temperatura "
                              "dos trocadores (trocadores maiores) ou reveja o COP.")
        eta_g = cop / cop_ideal
    else:
        eta_g = p.eta_global_sem_cop
        cop = eta_g * cop_ideal
    eta_is = eta_g / p.eta_eletromecanica
    eta_em = p.eta_eletromecanica
    if eta_is > 1.0:
        eta_is, eta_em = 1.0, eta_g
        av.critico("compressor", f"η_global = {eta_g:.2f} exige η_eletromecânica = η_global (η_is = 1): "
                   "o COP pedido só é atingível com compressor sem perdas internas.")
    lo, hi = p.eta_global_faixa
    if eta_g > hi:
        av.atencao("compressor", f"COP pedido implica η_global = {eta_g:.2f} (> {hi:.2f}, acima dos melhores "
                   "herméticos alternativos de pequena capacidade). Verificar com catálogo ou rever o COP.")
    elif eta_g < lo:
        av.atencao("compressor", f"COP pedido implica η_global = {eta_g:.2f} (< {lo:.2f}): COP baixo para estas "
                   "temperaturas; há margem para pedir um COP maior.")
    e2 = fl.estado_PH(P_cond, e1s.h + w_s / eta_is)
    if e2.T - 273.15 > p.T_descarga_max_C:
        av.atencao("compressor", f"temperatura de descarga {e2.T - 273.15:.0f} °C > {p.T_descarga_max_C:.0f} °C "
                   "(risco de degradação do óleo).")
    if P_evap < 101325.0:
        av.atencao("ciclo", f"P_evap = {P_evap / 1e5:.3f} bar abs < 1 atm: risco de entrada de ar/umidade "
                   "em vazamentos.")
    estados = {"1": e1, "1s": e1s, "2s": e2s, "2": e2, "3": e3, "3i": e3i, "4": e4}
    if e4.x <= 0:
        raise ErroProjeto("Estado 4 sem vapor: verifique sub-resfriamento/IHX.")
    return Ciclo(fl.nome, T_evap, T_cond, estados, cop, cop_ideal, cop_carnot, eta_g, eta_is, eta_em,
                 q_evap, w_s, q_ihx, P_cond / P_evap)


@dataclass
class BalancoSistema:
    m: float
    Q_evap: float
    W_el: float
    Q_cond: float
    Q_carcaca: float
    Q_ihx: float
    erro_balanco: float  # (Q_evap + W_el − Q_cond − Q_carc)/W_el


def balanco(ciclo: Ciclo, Q_evap: float, avisos: Optional[Avisos] = None) -> BalancoSistema:
    e = ciclo.estados
    m = Q_evap / ciclo.q_evap
    W_el = Q_evap / ciclo.COP
    Q_cond = m * (e["2"].h - e["3"].h)
    Q_carc = W_el - m * (e["2"].h - e["1s"].h)
    if Q_carc < -1e-9 and avisos is not None:
        avisos.critico("balanço", "calor de carcaça negativo: η_is incoerente com η_global.")
    err = (Q_evap + W_el - Q_cond - Q_carc) / W_el
    return BalancoSistema(m, Q_evap, W_el, Q_cond, Q_carc, m * ciclo.q_ihx, err)


def exergia(ciclo: Ciclo, b: BalancoSistema, T0: float, T_frio: float) -> dict:
    """Destruição de exergia por componente [W], estado morto T0 = ambiente.

    O efeito útil é a exergia do calor retirado do espaço refrigerado à T_frio:
    Ex_Q = Q_evap (T0/T_frio − 1). Fechamento: W_el = Ex_Q + Σ Ex_destruída.
    O evaporador inclui a irreversibilidade da diferença de temperatura ar-refrigerante,
    e o condensador a do calor rejeitado ao ambiente."""
    e, m = ciclo.estados, b.m
    d = {
        "compressor": T0 * (m * (e["2"].s - e["1s"].s) + b.Q_carcaca / T0),
        "condensador": T0 * (m * (e["3"].s - e["2"].s) + b.Q_cond / T0),
        "trocador interno": T0 * m * ((e["1s"].s - e["1"].s) + (e["3i"].s - e["3"].s)),
        "capilar": T0 * m * (e["4"].s - e["3i"].s),
        "evaporador": T0 * (m * (e["1"].s - e["4"].s) - b.Q_evap / T_frio),
    }
    ex_q = b.Q_evap * (T0 / T_frio - 1.0)
    d = {k: (0.0 if abs(v) < 1e-6 else v) for k, v in d.items()}
    fech = (b.W_el - ex_q - sum(d.values())) / b.W_el
    return {"destruicao_W": d, "Ex_util_W": ex_q, "eta_II": ex_q / b.W_el, "erro_fechamento": fech}


def escala_dT_para_cop(fl: Fluido, cop_alvo: float, T_ar_evap: float, T_ar_cond: float,
                       p: ParametrosCiclo, eta_global: float) -> Optional[float]:
    """Fator s que multiplica (dT_evap, dT_cond) para que um compressor de η_global dado atinja
    o COP alvo. s < 1 => trocadores maiores. Retorna None se inatingível mesmo com s = 0,02.
    O limite superior de s respeita T_cond < T_c − 5 K (validade do ciclo subcrítico)."""
    q = ParametrosCiclo(**{**p.__dict__, "eta_global_sem_cop": eta_global})

    def resid(s):
        c = resolver_ciclo(fl, T_ar_evap - s * p.dT_evap, T_ar_cond + s * p.dT_cond, q, None)
        return c.COP - cop_alvo

    s_min = 0.02
    s_max = min(3.0, (fl.Tc - 5.1 - T_ar_cond) / p.dT_cond, (T_ar_evap - fl.Ttriplo - 5.1) / p.dT_evap)
    try:
        a = resid(s_min)
    except ErroProjeto:
        return None
    if a < 0.0 or s_max <= s_min:
        return None
    try:
        b = resid(s_max)
    except ErroProjeto:
        return None
    if b >= 0.0:
        return s_max
    return brentq(resid, s_min, s_max, xtol=1e-4)

In [ ]:
%%writefile predimref/compressor.py
"""Compressor alternativo hermético: deslocamento volumétrico e conversão para ponto de catálogo.

Rendimento volumétrico (modelo de espaço morto com fator de perdas):
    η_v = f_v [1 + C − C (P_desc/P_suc)^(1/n)]
C fração de espaço morto, n expoente de reexpansão (≈ cp/cv do vapor na sucção) e f_v o
fator que reúne aquecimento do gás na sucção, vazamentos e perdas nas válvulas.
Referência: Stoecker & Jones, Refrigeration and Air Conditioning, cap. 11; Gosney,
Principles of Refrigeration, cap. 2. Os valores padrão são típicos; o ideal é calibrar
η_v e η_global com o catálogo do compressor candidato (Embraco, Secop, Tecumseh...).

Condições nominais de catálogo (tabela de condições de ensaio do catálogo Embraco):
    EN 12900 LBP −35/40 °C, MBP −10/45 °C, HBP +5/50 °C; gás de retorno 20 °C; sem sub-resfriamento
    ASHRAE LBP −23,3/54,4 °C; retorno 32,2 °C; líquido 32,2 °C
    ASHRAE M/HBP +7,2/54,4 °C; retorno 35 °C; líquido 46,1 °C
    CECOMAF LBP −25/55 °C; retorno 32 °C; sem sub-resfriamento
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

from .propriedades import Estado, Fluido

CONDICOES_NOMINAIS = {
    # nome: (T_evap °C, T_cond °C, T_retorno °C, T_liquido °C ou None = líquido saturado)
    "EN12900_LBP": (-35.0, 40.0, 20.0, None),
    "EN12900_MBP": (-10.0, 45.0, 20.0, None),
    "EN12900_HBP": (5.0, 50.0, 20.0, None),
    "ASHRAE_LBP": (-23.3, 54.4, 32.2, 32.2),
    "ASHRAE_MHBP": (7.2, 54.4, 35.0, 46.1),
    "CECOMAF_LBP": (-25.0, 55.0, 32.0, None),
}


@dataclass
class ParametrosCompressor:
    rotacao_rpm: float = 3500.0        # motor de 2 polos em 60 Hz (México, Colômbia, Brasil)
    fracao_espaco_morto: float = 0.04
    expoente_reexpansao: Optional[float] = None  # None -> cp/cv do vapor na sucção
    fator_volumetrico: float = 0.88
    condicao_catalogo: str = "EN12900_MBP"


def eficiencia_volumetrica(razao_pressao: float, p: ParametrosCompressor, n: float) -> float:
    C = p.fracao_espaco_morto
    return p.fator_volumetrico * (1.0 + C - C * razao_pressao ** (1.0 / n))


@dataclass
class ResultadoCompressor:
    m: float
    vazao_volumetrica_succao: float   # m³/s
    eta_v: float
    deslocamento_cm3: float           # por rotação
    rotacao_rpm: float
    W_el: float
    T_descarga: float
    catalogo: dict


def dimensionar_compressor(fl: Fluido, m: float, suc: Estado, P_desc: float, W_el: float,
                           T_desc: float, eta_global: float, p: ParametrosCompressor) -> ResultadoCompressor:
    n = p.expoente_reexpansao or fl.kappa(suc.P, suc.T, suc.rho)
    eta_v = eficiencia_volumetrica(P_desc / suc.P, p, n)
    Vdot = m / suc.rho
    desloc = Vdot / eta_v / (p.rotacao_rpm / 60.0)
    cat = ponto_catalogo(fl, desloc, p, eta_global)
    return ResultadoCompressor(m, Vdot, eta_v, desloc * 1e6, p.rotacao_rpm, W_el, T_desc, cat)


def ponto_catalogo(fl: Fluido, deslocamento_m3: float, p: ParametrosCompressor, eta_global: float) -> dict:
    """Capacidade e potência estimadas do MESMO compressor na condição de catálogo, para
    facilitar a seleção comercial. Mantém η_global e o modelo de η_v (estimativa ±15 %)."""
    Te, Tc, Tr, Tl = CONDICOES_NOMINAIS[p.condicao_catalogo]
    Te, Tc, Tr = Te + 273.15, Tc + 273.15, Tr + 273.15
    Pe, Pc = fl.Psat(Te), fl.Psat(Tc)
    suc = fl.estado_PT(Pe, Tr)
    liq = fl.estado_PQ(Pc, 0.0) if Tl is None else fl.estado_PT(Pc, Tl + 273.15)
    n = p.expoente_reexpansao or fl.kappa(Pe, Tr, suc.rho)
    eta_v = eficiencia_volumetrica(Pc / Pe, p, n)
    m = deslocamento_m3 * p.rotacao_rpm / 60.0 * eta_v * suc.rho
    q = suc.h - liq.h
    w_s = fl.estado_PS(Pc, suc.s).h - suc.h
    Q = m * q
    W = m * w_s / eta_global
    return {"condicao": p.condicao_catalogo, "T_evap_C": Te - 273.15, "T_cond_C": Tc - 273.15,
            "T_retorno_C": Tr - 273.15, "Q_W": Q, "Q_Btuh": Q * 3.412142, "W_el_W": W, "COP": Q / W,
            "m_kg_h": m * 3600.0, "eta_v": eta_v}

In [ ]:
%%writefile predimref/trocador.py
"""Dimensionamento de condensador e evaporador tubo-aleta (aletas planas, tubos escalonados).

Método: zonas por fase (dessuperaquecimento / bifásico / sub-resfriamento no condensador;
bifásico / superaquecimento no evaporador), cada zona discretizada em segmentos ao longo do
circuito de refrigerante. Em cada segmento:

* propriedades locais do refrigerante (P e h locais, CoolProp);
* coeficiente interno local: Shah (2022) na condensação, Gungor-Winterton (1987) ou
  Liu-Winterton (1991) na ebulição (com secagem de parede acima de x_di), Gnielinski no
  monofásico;
* lado do ar: Wang et al. (2000) + eficiência de aleta de Schmidt; no evaporador, fator de
  umidade ξ = Q_total/Q_sens (analogia de Lewis) quando há remoção de umidade;
* UA por metro de tubo com resistências em série (ar, incrustação, parede, refrigerante);
* ε-NTU de escoamento cruzado (refrigerante misturado, ar não misturado); no bifásico
  C_r = 0 e ε = 1 − exp(−NTU);
* o ar que atravessa cada trecho entra na temperatura de entrada da serpentina, com vazão
  proporcional ao comprimento (distribuição paralela de ar, hipótese padrão dos modelos de
  fronteira móvel): Ċ_ar por metro = ρ V_face P_t c_p / N_fileiras. Assim o NTU global é
  consistente com o UA e a vazão de ar totais da serpentina;
* queda de pressão: atrito (Müller-Steinhagen & Heck no bifásico; Churchill no monofásico)
  + aceleração com fração de vazio de Rouhani-Axelsson; a pressão é atualizada ao longo do
  circuito e a temperatura de saturação acompanha. A diferença de temperatura refrigerante-ar
  é verificada com sinal em cada trecho: se a queda de pressão levar a saturação para o "lado
  errado" do ar (pinch), o ponto é rejeitado em vez de produzir calor no sentido contrário.

Duas serpentinas são reportadas:
* **de projeto (sem margem)** — o comprimento requerido; com a vazão de ar requerida ela entrega
  exatamente a capacidade e as temperaturas do ciclo (é a que reproduz ``alvo_validacao``);
* **construtiva (com margem)** — comprimento × (1 + margem_area), com o mesmo nº de fileiras,
  tubos por fileira inteiros e o comprimento do passe ajustado (padrão) ou com tubos inteiros
  num passe fixo. Na mesma velocidade de face ela tem a mesma ε, porém mais vazão de ar e mais
  UA: num modelo de sistema ela condensa abaixo de T_cond e evapora acima de T_evap (é a folga).
Curvas de retorno (U) não entram na perda de carga (pequenas aos fluxos mássicos destes sistemas).

Modo de eficácia (``ParametrosTrocador.eficacia``)
--------------------------------------------------
A eficácia é a do lado do ar, referida à temperatura de saturação do refrigerante (definição usual de
condensadores e evaporadores, em que o refrigerante bifásico tem C → ∞ e C_min = Ċ_ar):

    ε = (T_ar,saída − T_ar,entrada) / (T_sat − T_ar,entrada) = Q_sens / [Ċ_ar (T_sat − T_ar,entrada)]

com T_sat = T_cond (entrada do condensador) ou T_evap (saída do evaporador) e Q_sens = Q/ξ.
Dados Q (do ciclo), ε e as temperaturas, a vazão de ar fica determinada (Ċ_ar = Q_sens/(ε ΔT)).
Na distribuição paralela de ar, ε independe do comprimento: depende da velocidade de face V, do nº
de fileiras N, do coeficiente interno e, no condensador, também da temperatura de descarga (o ar
que passa pelo dessuperaquecimento sai acima de T_sat; por isso no condensador ε não segue
1 − e^(−NTU) e, a velocidades muito baixas, pode até passar de 1). Na zona bifásica vale
ε_zona ≤ 1 − e^(−NTU_zona) (desigualdade de Jensen) e os valores por zona são exportados.
Com perda de carga relevante ε(V) pode deixar de ser monotônica; por isso, para cada N, a busca de V
percorre uma grade logarítmica de velocidade_max para velocidade_min, refina por Brent a primeira
troca de sinal encontrada e adota essa raiz, a de maior V (menos tubo e menos perda de carga). Com ``fileiras_automaticas`` escolhe-se
o N (≤ ``fileiras_max``) cuja V fica mais perto da ``velocidade_face`` preferida.
Também se reporta a eficácia "estrita" Q/Q_max com o Q_max termodinâmico (calores específicos
variáveis): Q_max = min[ṁ |h_ref,entrada − h(P, T_ar,entrada)|, ξ Ċ_ar |T_ref,entrada − T_ar,entrada|],
útil se o componente do Amesim usar essa definição.
"""
from __future__ import annotations

import copy
import dataclasses
import math
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
from scipy.optimize import brentq

from .avisos import ErroProjeto
from .correlacoes import bifasico as bf
from .correlacoes.lado_ar import (DesempenhoAr, GeometriaAletada, desempenho_lado_ar,
                                  queda_pressao_ar)
from .correlacoes.monofasico import gradiente_pressao_monofasico, h_interno_monofasico
from .propriedades import Fluido, ar_seco

class ErroPerdaCarga(ErroProjeto):
    """Inviabilidade causada pela perda de carga do refrigerante (pinch ou queda excessiva):
    resolvida com mais circuitos ou tubo de maior diâmetro."""


FOLGA_PINCH_K = 0.2          # diferença mínima refrigerante-ar aceita num trecho [K]
FRACAO_DP_MAX = 0.5          # queda de pressão máxima admitida, fração da pressão de entrada
N_GRADE_VELOCIDADE = 10      # pontos da grade logarítmica de V no modo de eficácia


@dataclass
class ParametrosTrocador:
    geometria: GeometriaAletada
    velocidade_face: float = 2.0           # m/s, na entrada (preferida, no modo de eficácia)
    n_circuitos: int = 1
    comprimento_passe_m: float = 0.40      # comprimento aletado de cada tubo (largura da face)
    ajustar_comprimento_passe: bool = True  # serpentina construtiva com L = L_req (1 + margem) exato
    margem_area: float = 0.10
    n_segmentos: int = 40                  # por zona bifásica (≤ 0,15 % de diferença para 240)
    n_segmentos_monofasico: int = 8        # por zona monofásica (perfis suaves: converge com poucos)
    R_incrustacao_ar: float = 0.0          # m²K/W, sobre a área externa (pó, geada)
    R_incrustacao_ref: float = 0.0         # m²K/W, sobre a área interna (óleo)
    correlacao_bifasico: str = "shah_2022"  # condensação: shah_2022|shah_2013|shah_2009|shah_1979|chato_1962
    #                                          ebulição: gungor_winterton_1987|liu_winterton_1991
    x_secagem: float = 0.90
    eficiencia_ventilador: float = 0.20    # elétrica total de ventiladores axiais pequenos
    # modo de eficácia: ε alvo referido à saturação; None -> velocidade_face fixa (geometria livre)
    eficacia: Optional[float] = None
    fileiras_automaticas: bool = True      # com ε: escolhe N ≤ fileiras_max com V mais perto de velocidade_face
    fileiras_max: int = 6                  # limite da base de dados de Wang et al. (2000)
    velocidade_min: float = 0.3            # m/s (Re_Dc ≈ 300–400, limite inferior de Wang)
    velocidade_max: float = 4.0            # m/s
    circuitos_automaticos: bool = True     # se a perda de carga impedir o projeto, tenta n_circuitos + 1
    circuitos_max: int = 4


def condensador_padrao() -> ParametrosTrocador:
    """Condensador de ar forçado típico de visi-cooler (tubo 3/8", aleta 2,5 mm ≈ 10 FPI)."""
    return ParametrosTrocador(
        geometria=GeometriaAletada(Do=9.52e-3, esp_parede=0.35e-3, esp_aleta=0.12e-3, passo_aleta=2.5e-3,
                                   Pt=25.4e-3, Pl=22.0e-3, n_fileiras=3),
        velocidade_face=2.0, comprimento_passe_m=0.30, correlacao_bifasico="shah_2022")


def evaporador_padrao() -> ParametrosTrocador:
    """Evaporador de ar forçado para média temperatura com degelo natural (6 FPI, 4 fileiras)."""
    return ParametrosTrocador(
        geometria=GeometriaAletada(Do=9.52e-3, esp_parede=0.35e-3, esp_aleta=0.15e-3, passo_aleta=4.23e-3,
                                   Pt=25.4e-3, Pl=22.0e-3, n_fileiras=4),
        velocidade_face=1.5, comprimento_passe_m=0.35, correlacao_bifasico="gungor_winterton_1987")


@dataclass
class Segmento:
    zona: str
    L: float              # comprimento de tubo do segmento (por circuito) [m]
    Q: float              # calor do segmento (por circuito) [W]
    P_in: float
    P_out: float
    h_in: float
    h_out: float
    T_in: float
    T_out: float
    x_in: float
    x_out: float
    h_int: float          # coeficiente interno [W/m²K]
    UA_m: float           # UA por metro [W/mK] (no evaporador, base úmida: ξ no filme de ar e na aleta)
    eps: float
    T_ar_out: float       # temperatura de bulbo seco do ar que deixa o trecho
    alpha: float          # fração de vazio média (0 no líquido, 1 no vapor)
    rho_media: float      # massa específica média in situ [kg/m³]
    dP_atrito: float
    dP_acel: float
    regime: str = ""
    UA_m_seco: float = 0.0  # UA por metro com ξ = 1 (superfície seca)


@dataclass
class Serpentina:
    """Uma realização geométrica da serpentina (de projeto ou construtiva)."""
    descricao: str
    comprimento_tubo: float      # total, todos os circuitos [m]
    comprimento_passe: float     # comprimento aletado de cada tubo [m]
    tubos_por_fileira: int
    n_tubos: int
    altura_face: float
    area_face: float
    area_externa: float
    area_interna: float
    volume_interno: float
    n_aletas: int
    vazao_ar_m3h: float          # na velocidade de face de projeto
    UA: float                    # base úmida no evaporador (ver Segmento.UA_m)
    UA_seco: float


@dataclass
class ResultadoTrocador:
    tipo: str
    fluido: str
    m: float
    Q: float
    segmentos: list
    zonas: dict
    projeto: Serpentina          # sem margem: reproduz o ponto de projeto do ciclo
    construtiva: Serpentina      # com margem e tubos inteiros
    n_circuitos: int
    n_fileiras: int
    T_ar_in: float
    T_ar_out: float              # serpentina de projeto, vazão requerida
    dP_ar: float                 # depende só de V e N (igual nas duas serpentinas)
    potencia_ventilador_aero: float   # serpentina construtiva
    potencia_ventilador_est: float    # serpentina construtiva
    P_in: float
    P_out: float
    dT_sat_queda: float
    G_ref: float
    ar: DesempenhoAr
    fator_umidade: float
    geometria: GeometriaAletada
    velocidade_face: float
    T_sat_ref: float             # temperatura de saturação de referência da eficácia [K]
    eficacia: float              # obtida, referida à saturação (ver docstring do módulo)
    eficacia_alvo: Optional[float]
    eficacia_Qmax: float         # Q / Q_max termodinâmico
    C_ar: float                  # capacidade térmica sensível do ar na serpentina de projeto [W/K]
    NTU: float                   # UA_projeto / (ξ C_ar)
    avisos: list = field(default_factory=list)
    fileiras_escolhidas_automaticamente: bool = False
    ar_seco: Optional[DesempenhoAr] = None   # lado do ar com ξ = 1 (coeficiente sensível)

    # ----------------------------------------------------- atalhos (serpentina construtiva)
    @property
    def dP_ref(self) -> float:
        return self.P_in - self.P_out

    @property
    def L_requerido(self) -> float:
        return self.projeto.comprimento_tubo

    @property
    def L_adotado(self) -> float:
        return self.construtiva.comprimento_tubo

    @property
    def UA_requerido(self) -> float:
        return self.projeto.UA

    @property
    def UA_adotado(self) -> float:
        return self.construtiva.UA

    @property
    def vazao_ar_requerida_m3h(self) -> float:
        return self.projeto.vazao_ar_m3h

    @property
    def vazao_ar_m3h(self) -> float:
        return self.construtiva.vazao_ar_m3h

    @property
    def tubos_por_fileira(self) -> int:
        return self.construtiva.tubos_por_fileira

    @property
    def n_tubos(self) -> int:
        return self.construtiva.n_tubos

    @property
    def comprimento_passe(self) -> float:
        return self.construtiva.comprimento_passe

    @property
    def altura_face(self) -> float:
        return self.construtiva.altura_face

    @property
    def area_face(self) -> float:
        return self.construtiva.area_face

    @property
    def area_externa(self) -> float:
        return self.construtiva.area_externa

    @property
    def area_interna(self) -> float:
        return self.construtiva.area_interna

    @property
    def volume_interno(self) -> float:
        return self.construtiva.volume_interno

    @property
    def n_aletas(self) -> int:
        return self.construtiva.n_aletas


# ============================================================================ utilidades
class _Ctx:
    def __init__(self, fl: Fluido, m: float, p: ParametrosTrocador, T_ar_in: float, T_ar_m: float,
                 xi: float, sentido: float):
        self.fl, self.p, self.geo = fl, p, p.geometria
        g = self.geo
        self.m_c = m / p.n_circuitos
        self.A_cs = math.pi * g.Di ** 2 / 4.0
        self.G = self.m_c / self.A_cs
        self.Ao, self.Ai = g.area_externa_por_m, g.area_interna_por_m
        self.Rw = g.resistencia_parede_por_m()
        a_in, a_m = ar_seco(T_ar_in), ar_seco(T_ar_m)
        self.rho_ar_in = a_in.rho
        self.cp_ar = a_m.cp
        self.ar = desempenho_lado_ar(p.velocidade_face, a_in.rho, a_m.cp, a_m.mu, a_m.Pr, g, xi)
        self.ar_seco = (self.ar if xi == 1.0 else
                        desempenho_lado_ar(p.velocidade_face, a_in.rho, a_m.cp, a_m.mu, a_m.Pr, g, 1.0))
        self.xi = xi
        self.Ca_seco = a_in.rho * p.velocidade_face * g.Pt * a_m.cp / g.n_fileiras  # W/(K·m)
        self.Ca = xi * self.Ca_seco
        self.T_ar_in = T_ar_in
        self.sentido = sentido          # +1: refrigerante mais quente que o ar (condensador); −1: evaporador
        self.M_kmol = fl.M * 1000.0
        self.hidrocarboneto = fl.hidrocarboneto

    def _UA(self, h_i: float, h_ar_eq: float, eta_o: float) -> float:
        R = (1.0 / (h_ar_eq * eta_o * self.Ao) + self.p.R_incrustacao_ar / self.Ao + self.Rw
             + 1.0 / (h_i * self.Ai) + self.p.R_incrustacao_ref / self.Ai)
        return 1.0 / R

    def UA_m(self, h_i: float) -> float:
        """UA por metro na base úmida (ξ no filme de ar e na eficiência de aleta)."""
        return self._UA(h_i, self.xi * self.ar.h, self.ar.eta_superficie)

    def UA_m_seco(self, h_i: float) -> float:
        return self._UA(h_i, self.ar_seco.h, self.ar_seco.eta_superficie)

    def dT_motriz(self, T_ref: float, zona: str) -> float:
        """Diferença refrigerante-ar no sentido esperado; rejeita pinch (inclusive o causado pela
        queda de pressão, que leva a saturação para o lado errado do ar)."""
        dT = self.sentido * (T_ref - self.T_ar_in)
        if dT <= FOLGA_PINCH_K:
            raise ErroPerdaCarga(f"pinch no {zona}: diferença refrigerante-ar de {dT:.2f} K "
                              "(perda de carga alta demais? aumente n_circuitos ou o diâmetro do tubo)")
        return dT


def _checar_pressao(P_entrada: float, P: float, zona: str) -> None:
    if not (P > (1.0 - FRACAO_DP_MAX) * P_entrada) or not math.isfinite(P):
        raise ErroPerdaCarga(f"queda de pressão excessiva no {zona} (> {FRACAO_DP_MAX:.0%} da pressão de entrada): "
                          "aumente n_circuitos ou o diâmetro do tubo")


def _eps_cruzado(NTU: float, C_ref: float, C_ar: float) -> float:
    """ε de escoamento cruzado, refrigerante misturado e ar não misturado (Incropera Tab. 11.3)."""
    Cmin, Cmax = min(C_ref, C_ar), max(C_ref, C_ar)
    Cr = Cmin / Cmax
    if Cr < 1e-9:
        return 1.0 - math.exp(-NTU)
    if C_ref <= C_ar:   # C_min misturado (refrigerante), C_max não misturado
        return 1.0 - math.exp(-(1.0 - math.exp(-Cr * NTU)) / Cr)
    return (1.0 - math.exp(-Cr * (1.0 - math.exp(-NTU)))) / Cr   # C_max misturado


def _L_monofasico(Q: float, C_ref: float, UA_m: float, Ca_m: float, dT_in: float) -> float:
    """Comprimento que transfere Q num trecho monofásico (raiz de ε C_min ΔT_in = Q)."""
    if dT_in <= 0 or Q >= 0.999999 * C_ref * dT_in:
        raise ErroProjeto("trecho monofásico impossível: a variação de temperatura pedida ao refrigerante "
                          "excede a diferença para o ar (reduza sub-resfriamento/superaquecimento ou "
                          "aumente a diferença de temperatura do trocador).")

    def f(L):
        Ca = Ca_m * L
        Cmin = min(Ca, C_ref)
        return _eps_cruzado(UA_m * L / Cmin, C_ref, Ca) * Cmin * dT_in - Q

    lo, hi = 1e-9, max(Q / (Ca_m * dT_in), 1e-6) * 2.0
    while f(hi) < 0:
        hi *= 2.0
        if hi > 1e5:
            raise ErroProjeto("não convergiu o comprimento do trecho monofásico")
    return brentq(f, lo, hi, xtol=1e-9, rtol=1e-10)


def _segmento_monofasico(ctx: _Ctx, zona: str, P_in: float, h_in: float, h_out: float) -> Segmento:
    fl, g = ctx.fl, ctx.geo
    e_in = fl.estado_PH(P_in, h_in, transporte=False)
    dT_in = ctx.dT_motriz(e_in.T, zona)
    dP = 0.0
    for it in range(4):
        _checar_pressao(P_in, P_in - dP, zona)
        e_out = fl.estado_PH(P_in - dP, h_out, transporte=False)
        e_m = fl.estado_PH(P_in - 0.5 * dP, 0.5 * (h_in + h_out))
        if e_m.mu is None:
            raise ErroPerdaCarga(f"estado médio bifásico num trecho de {zona} (queda de pressão excessiva)")
        Q = ctx.m_c * abs(h_in - h_out)
        dT_r = abs(e_in.T - e_out.T)
        C_ref = Q / dT_r if dT_r > 1e-9 else 1e12
        h_i = h_interno_monofasico(ctx.G, g.Di, e_m.mu, e_m.k, e_m.Pr)
        UA = ctx.UA_m(h_i)
        L = _L_monofasico(Q, C_ref, UA, ctx.Ca, dT_in)
        dP_f = gradiente_pressao_monofasico(ctx.G, g.Di, e_m.rho, e_m.mu, g.rugosidade_int) * L
        dP_a = ctx.G ** 2 * (1.0 / e_out.rho - 1.0 / e_in.rho)
        dP_novo = dP_f + dP_a
        conv = abs(dP_novo - dP) <= 1e-6 * P_in + 1e-3 * abs(dP_novo)
        dP = dP_novo
        if conv and it > 0:
            break
    _checar_pressao(P_in, P_in - dP, zona)
    Cmin = min(ctx.Ca * L, C_ref)
    eps = Q / (Cmin * dT_in)
    T_ar_out = ctx.T_ar_in + ctx.sentido * Q / (ctx.Ca * L)
    alpha = 1.0 if e_in.x >= 1.0 else 0.0
    return Segmento(zona, L, Q, P_in, P_in - dP, h_in, h_out, e_in.T, e_out.T, e_in.x, e_out.x, h_i, UA, eps,
                    T_ar_out, alpha, e_m.rho, dP_f, dP_a,
                    "laminar" if ctx.G * g.Di / e_m.mu < 2300 else "turbulento", ctx.UA_m_seco(h_i))


def _h_bifasico(ctx: _Ctx, x_m: float, sat, q: float, condensando: bool) -> tuple[float, str]:
    p, D = ctx.p, ctx.geo.Di
    c = p.correlacao_bifasico
    if condensando:
        if c == "shah_2022":
            return bf.condensacao_shah_2022(ctx.G, x_m, D, sat, ctx.hidrocarboneto)
        if c == "shah_2013":
            return bf.condensacao_shah_2013(ctx.G, x_m, D, sat)
        if c == "shah_2009":
            return bf.condensacao_shah_2009(ctx.G, x_m, D, sat)
        if c == "shah_1979":
            return bf.condensacao_shah_1979(ctx.G, x_m, D, sat)
        if c == "chato_1962":
            h0 = bf.condensacao_shah_2013(ctx.G, x_m, D, sat)[0]
            h = h0
            for _ in range(30):   # ΔT_parede = q/h (forma implícita de Nusselt)
                h_novo, reg = bf.condensacao_chato_1962(D, sat, q / h)
                if abs(h_novo - h) < 1e-4 * h:
                    break
                h = h_novo
            return h_novo, reg
        raise ValueError(f"correlação de condensação desconhecida: {c}")
    h = bf.ebulicao_com_secagem(ctx.G, x_m, D, q, sat, ctx.M_kmol, c, p.x_secagem)
    return h, ("secagem" if x_m > p.x_secagem else "ebulição")


def _segmento_bifasico(ctx: _Ctx, zona: str, P_in: float, x_in: float, x_out: float,
                       condensando: bool) -> Segmento:
    fl, g = ctx.fl, ctx.geo
    sat_in = fl.sat(P_in)
    ctx.dT_motriz(sat_in.T, zona)
    h_in = sat_in.hl + x_in * sat_in.hlv
    dP, q = 0.0, 5000.0
    for _ in range(6):
        _checar_pressao(P_in, P_in - dP, zona)
        P_out = P_in - dP
        sat_m, sat_out = fl.sat(P_in - 0.5 * dP), fl.sat(P_out)
        x_m = 0.5 * (x_in + x_out)
        h_out = sat_out.hl + x_out * sat_out.hlv
        Q = ctx.m_c * abs(h_in - h_out)
        dT = ctx.dT_motriz(sat_m.T, zona)
        for _k in range(30):   # ponto fixo no fluxo de calor (correlações dependentes de q)
            h_i, regime = _h_bifasico(ctx, x_m, sat_m, q, condensando)
            UA = ctx.UA_m(h_i)
            eps = 1.0 - math.exp(-UA / ctx.Ca)
            L = Q / (eps * ctx.Ca * dT)
            q_novo = Q / (ctx.Ai * L)
            conv = abs(q_novo - q) < 1e-4 * q
            q = q_novo
            if conv:
                break
        dP_f = bf.gradiente_atrito_msh(ctx.G, x_m, g.Di, sat_m) * L
        a_in = bf.fracao_vazio_rouhani_axelsson(x_in, ctx.G, sat_in)
        a_out = bf.fracao_vazio_rouhani_axelsson(x_out, ctx.G, sat_out)
        dP_a = ctx.G ** 2 * (bf.fluxo_momento_especifico(x_out, a_out, sat_out)
                             - bf.fluxo_momento_especifico(x_in, a_in, sat_in))
        dP_novo = dP_f + dP_a
        conv = abs(dP_novo - dP) <= 1e-6 * P_in + 1e-3 * abs(dP_novo)
        dP = dP_novo
        if conv:
            break
    _checar_pressao(P_in, P_in - dP, zona)
    ctx.dT_motriz(fl.sat(P_in - dP).T, zona)   # a saturação na saída do trecho também precisa de folga
    alpha = 0.5 * (a_in + a_out)
    rho_m = 0.5 * (a_in * sat_in.rhov + (1 - a_in) * sat_in.rhol + a_out * sat_out.rhov + (1 - a_out) * sat_out.rhol)
    T_ar_out = ctx.T_ar_in + ctx.sentido * Q / (ctx.Ca * L)
    return Segmento(zona, L, Q, P_in, P_in - dP, h_in, h_out, sat_in.T, sat_out.T, x_in, x_out, h_i, UA, eps,
                    T_ar_out, alpha, rho_m, dP_f, dP_a, regime, ctx.UA_m_seco(h_i))


def _serpentina(descricao: str, L_total: float, N: int, Nc: int, p: ParametrosTrocador, UA_m_medio: float,
                UA_m_seco_medio: float, ajustar: bool) -> Serpentina:
    """Arranjo de tubos inteiros para um comprimento total de tubo."""
    g = p.geometria
    if ajustar:
        tpf = max(1, math.ceil(L_total / (N * p.comprimento_passe_m) - 1e-9))
        while (tpf * N) % Nc:
            tpf += 1
        L_passe = L_total / (N * tpf)
    else:
        tpf = max(1, math.ceil(math.ceil(L_total / p.comprimento_passe_m - 1e-9) / N))
        while (tpf * N) % Nc:
            tpf += 1
        L_passe = p.comprimento_passe_m
    n_tubos = tpf * N
    L = n_tubos * L_passe
    altura = tpf * g.Pt
    A_face = altura * L_passe
    return Serpentina(descricao, L, L_passe, tpf, n_tubos, altura, A_face, g.area_externa_por_m * L,
                      g.area_interna_por_m * L, math.pi * g.Di ** 2 / 4.0 * L, int(L_passe / g.passo_aleta),
                      p.velocidade_face * A_face * 3600.0, UA_m_medio * L, UA_m_seco_medio * L)


def _consolidar(tipo: str, fl: Fluido, m: float, segs: list, ctx: _Ctx, p: ParametrosTrocador,
                T_ar_in: float, T_sat_ref: float) -> ResultadoTrocador:
    g = p.geometria
    Nc = p.n_circuitos
    N = g.n_fileiras
    zonas: dict = {}
    for s in segs:
        z = zonas.setdefault(s.zona, {"L": 0.0, "Q": 0.0, "UA": 0.0, "UA_seco": 0.0})
        z["L"] += s.L * Nc
        z["Q"] += s.Q * Nc
        z["UA"] += s.UA_m * s.L * Nc
        z["UA_seco"] += s.UA_m_seco * s.L * Nc
    for z in zonas.values():
        C_z = ctx.Ca_seco * z["L"]
        z["eficacia"] = z["Q"] / (ctx.xi * C_z * abs(T_sat_ref - T_ar_in))
        z["NTU"] = z["UA"] / (ctx.xi * C_z)
    L_req = sum(s.L for s in segs) * Nc
    Q = sum(s.Q for s in segs) * Nc
    UA_req = sum(s.UA_m * s.L for s in segs) * Nc
    UA_seco_req = sum(s.UA_m_seco * s.L for s in segs) * Nc
    UA_m, UA_m_s = UA_req / L_req, UA_seco_req / L_req
    # serpentina de projeto (sem margem): comprimento exato, passe ajustado ao arranjo da construtiva
    construtiva = _serpentina("construtiva (com margem)", L_req * (1.0 + p.margem_area), N, Nc, p, UA_m, UA_m_s,
                              p.ajustar_comprimento_passe)
    proj = Serpentina("projeto (sem margem)", L_req, L_req / (N * construtiva.tubos_por_fileira),
                      construtiva.tubos_por_fileira, construtiva.n_tubos, construtiva.altura_face,
                      construtiva.altura_face * L_req / (N * construtiva.tubos_por_fileira),
                      g.area_externa_por_m * L_req, g.area_interna_por_m * L_req,
                      math.pi * g.Di ** 2 / 4.0 * L_req,
                      int(L_req / (N * construtiva.tubos_por_fileira) / g.passo_aleta),
                      ctx.Ca_seco * L_req / (ctx.rho_ar_in * ctx.cp_ar) * 3600.0, UA_req, UA_seco_req)
    T_out = sum(s.T_ar_out * s.L for s in segs) / sum(s.L for s in segs)
    rho_out = ar_seco(T_out).rho
    dP_ar = queda_pressao_ar(ctx.ar.G_max, ctx.ar.f, g, ctx.rho_ar_in, rho_out)
    P_in, P_out = segs[0].P_in, segs[-1].P_out
    dTsat = abs(fl.Tsat(P_in) - fl.Tsat(P_out))
    av = list(g.verificar_faixa_wang())
    if not 100 <= ctx.ar.Re_Dc <= 20000:
        av.append(f"Re_Dc do ar = {ctx.ar.Re_Dc:.0f} fora da faixa usual da correlação de Wang")
    if dTsat > 1.5:
        av.append(f"queda de pressão do refrigerante equivale a {dTsat:.2f} K de saturação: considere mais circuitos")
    C_ar = ctx.Ca_seco * L_req                       # sensível, serpentina de projeto
    eps = abs(T_out - T_ar_in) / abs(T_sat_ref - T_ar_in)
    if eps > 1.0:
        av.append(f"eficácia referida à saturação = {eps:.3f} > 1: o ar sai acima de T_sat por causa do "
                  "dessuperaquecimento (velocidade de face muito baixa)")
    # eficácia "estrita" com o Q_max termodinâmico (calores específicos variáveis, Shah & Sekulic 2003):
    # refrigerante levado à temperatura de entrada do ar, ou ar levado à temperatura de entrada do refrigerante
    T_ref_in = segs[0].T_in
    h_ref_lim = fl.estado_PT(P_out, T_ar_in).h
    Qmax_ref = m * abs(segs[0].h_in - h_ref_lim)
    Qmax_ar = ctx.xi * C_ar * abs(T_ref_in - T_ar_in)
    eps_Qmax = Q / min(Qmax_ref, Qmax_ar)
    Vdot_c = construtiva.vazao_ar_m3h / 3600.0
    return ResultadoTrocador(
        tipo, fl.nome, m, Q, segs, zonas, proj, construtiva, Nc, N, T_ar_in, T_out, dP_ar, Vdot_c * dP_ar,
        Vdot_c * dP_ar / p.eficiencia_ventilador, P_in, P_out, dTsat, ctx.G, ctx.ar, ctx.xi, g,
        p.velocidade_face, T_sat_ref, eps, p.eficacia, eps_Qmax, C_ar, UA_req / (ctx.xi * C_ar), av,
        ar_seco=ctx.ar_seco)


def _blindar(tipo: str, funcao, *args):
    """Converte falhas numéricas de propriedades (estados fora do domínio do CoolProp, divisão por
    zero) em ErroProjeto, para que o solucionador trate o ponto como inválido."""
    try:
        return funcao(*args)
    except ErroProjeto:
        raise
    except (ValueError, TypeError, ZeroDivisionError, OverflowError) as e:
        raise ErroProjeto(f"falha numérica na marcha do {tipo}: {e}") from None


# ============================================================================ condensador
def _condensador_fixo(fl: Fluido, m: float, P_in: float, h_in: float, subresfriamento: float,
                      T_ar_in: float, p: ParametrosTrocador) -> ResultadoTrocador:
    """Marcha do condensador para geometria e velocidade de face fixas."""
    return _blindar("condensador", _condensador_marcha, fl, m, P_in, h_in, subresfriamento, T_ar_in, p)


def _condensador_marcha(fl, m, P_in, h_in, subresfriamento, T_ar_in, p):
    n, n1 = p.n_segmentos, p.n_segmentos_monofasico
    T_ar_m = T_ar_in + 5.0
    for _ in range(3):   # propriedades do ar na temperatura média
        ctx = _Ctx(fl, m, p, T_ar_in, T_ar_m, 1.0, +1.0)
        segs: list = []
        P, h = P_in, h_in
        sat = fl.sat(P)
        if h > sat.hv:   # dessuperaquecimento
            for k in range(1, n1 + 1):
                h_t = h_in + (sat.hv - h_in) * k / n1
                s = _segmento_monofasico(ctx, "dessuperaquecimento", P, h, h_t)
                segs.append(s)
                P, h = s.P_out, s.h_out
        _checar_pressao(P_in, P, "condensador")
        sat = fl.sat(P)
        x0 = min((h - sat.hl) / sat.hlv, 1.0)
        for k in range(n):   # condensação
            x_a = x0 * (1.0 - k / n)
            x_b = 0.0 if k == n - 1 else x0 * (1.0 - (k + 1) / n)
            s = _segmento_bifasico(ctx, "condensação", P, x_a, x_b, True)
            segs.append(s)
            P = s.P_out
            _checar_pressao(P_in, P, "condensador")
        h = fl.sat(P).hl
        if subresfriamento > 0:
            T_sat = fl.Tsat(P)
            h_fim = fl.estado_PT(P, T_sat - subresfriamento).h
            for k in range(1, n1 + 1):
                h_t = h + (h_fim - h) * k / n1 if k < n1 else h_fim
                h_a = segs[-1].h_out if segs[-1].zona == "sub-resfriamento" else h
                s = _segmento_monofasico(ctx, "sub-resfriamento", P, h_a, h_t)
                segs.append(s)
                P = s.P_out
        T_out_mix = sum(s.T_ar_out * s.L for s in segs) / sum(s.L for s in segs)
        T_novo = 0.5 * (T_ar_in + T_out_mix)
        if abs(T_novo - T_ar_m) < 0.05:
            break
        T_ar_m = T_novo
    return _consolidar("condensador", fl, m, segs, ctx, p, T_ar_in, fl.Tsat(P_in))


# ============================================================================ evaporador
def _evaporador_fixo(fl: Fluido, m: float, h_in: float, P_out_alvo: float, superaquecimento: float,
                     T_ar_in: float, p: ParametrosTrocador, fator_umidade: float = 1.0) -> ResultadoTrocador:
    """Marcha do evaporador para geometria e velocidade de face fixas. A pressão de entrada
    (= saída do capilar) é obtida iterando a queda de pressão."""
    return _blindar("evaporador", _evaporador_marcha, fl, m, h_in, P_out_alvo, superaquecimento, T_ar_in, p,
                    fator_umidade)


def _evaporador_marcha(fl, m, h_in, P_out_alvo, superaquecimento, T_ar_in, p, fator_umidade):
    n, n1 = p.n_segmentos, p.n_segmentos_monofasico
    T_sat_out = fl.Tsat(P_out_alvo)
    h_fim = fl.estado_PT(P_out_alvo, T_sat_out + superaquecimento).h if superaquecimento > 0 else fl.sat(P_out_alvo).hv
    dP_tot = 0.0
    T_ar_m = T_ar_in - 2.0
    erro = math.inf
    for _ in range(12):
        ctx = _Ctx(fl, m, p, T_ar_in, T_ar_m, fator_umidade, -1.0)
        P = P_out_alvo + dP_tot
        if P >= fl.Pc:
            raise ErroProjeto("queda de pressão do evaporador incompatível (pressão de entrada ≥ crítica)")
        P_ent = P
        sat = fl.sat(P)
        x_in = (h_in - sat.hl) / sat.hlv
        if not 0.0 < x_in < 1.0:
            raise ErroProjeto(f"título na entrada do evaporador fora de (0,1): {x_in:.3f}")
        segs: list = []
        for k in range(n):
            x_a = x_in + (1.0 - x_in) * k / n
            x_b = 1.0 if k == n - 1 else x_in + (1.0 - x_in) * (k + 1) / n   # último ponto exato
            s = _segmento_bifasico(ctx, "evaporação", P, x_a, x_b, False)
            segs.append(s)
            P = s.P_out
            _checar_pressao(P_ent, P, "evaporador")
        if superaquecimento > 0:
            h = fl.sat(P).hv
            for k in range(1, n1 + 1):
                h_t = h + (h_fim - h) * k / n1 if k < n1 else h_fim
                h_a = segs[-1].h_out if segs[-1].zona == "superaquecimento" else h
                s = _segmento_monofasico(ctx, "superaquecimento", P, h_a, h_t)
                segs.append(s)
                P = s.P_out
        erro = P - P_out_alvo
        T_out_mix = sum(s.T_ar_out * s.L for s in segs) / sum(s.L for s in segs)
        T_ar_m = 0.5 * (T_ar_in + T_out_mix)
        dP_tot = segs[0].P_in - segs[-1].P_out
        if abs(erro) < 1.0:
            break
    if abs(erro) > max(100.0, 0.01 * P_out_alvo):
        raise ErroProjeto("a pressão de entrada do evaporador não convergiu (queda de pressão instável): "
                          "aumente n_circuitos")
    return _consolidar("evaporador", fl, m, segs, ctx, p, T_ar_in, T_sat_out)


# ============================================================================ modo de eficácia
def _com(p: ParametrosTrocador, N: int, V: float) -> ParametrosTrocador:
    pp = copy.copy(p)
    pp.geometria = dataclasses.replace(p.geometria, n_fileiras=int(N))
    pp.velocidade_face = float(V)
    return pp


class _PontoInvalido(Exception):
    pass


@dataclass
class BuscaVelocidade:
    """Resultado da busca de V para um nº de fileiras."""
    N: int
    V: Optional[float]
    motivo: str                 # "ok" | "alvo acima do maximo" | "alvo abaixo do minimo" | "sem ponto valido"
    eps_max: Optional[float] = None
    V_eps_max: Optional[float] = None
    eps_min: Optional[float] = None
    V_eps_min: Optional[float] = None
    erro: str = ""
    invalidos: list = field(default_factory=list)   # velocidades rejeitadas (pinch, ΔP, falha numérica)
    dTsat_no_max: Optional[float] = None             # queda de saturação no ponto de ε máxima [K]

    @property
    def limitado_perda_carga(self) -> bool:
        """ε máxima limitada pela perda de carga: há pontos rejeitados abaixo da V de ε máxima, o máximo
        é interior (ε cai ao reduzir V) ou a queda de saturação nesse ponto é relevante."""
        if self.V_eps_max is None:
            return bool(self.invalidos)
        interior = self.V_eps_min is not None and self.eps_min is not None and any(
            V < self.V_eps_max for V in self.invalidos)
        return interior or (self.dTsat_no_max or 0.0) > 1.5 or self._max_interior

    _max_interior: bool = False


class _Solucionador:
    """Resolve a velocidade de face (e, se pedido, o nº de fileiras) para uma eficácia alvo."""

    def __init__(self, rodar, p: ParametrosTrocador, tipo: str):
        if p.eficacia is None or not 0.0 < p.eficacia < 1.0:
            raise ErroProjeto(f"eficácia do {tipo} deve estar entre 0 e 1 (recebido {p.eficacia}).")
        if not 0.0 < p.velocidade_min < p.velocidade_max:
            raise ErroProjeto("faixa de velocidade de face inválida")
        self.rodar, self.p, self.tipo, self.alvo = rodar, p, tipo, p.eficacia
        self._cache: dict = {}

    def resultado(self, N: int, V: float) -> Optional[ResultadoTrocador]:
        k = (int(N), round(float(V), 7))
        if k not in self._cache:
            try:
                self._cache[k] = self.rodar(_com(self.p, N, V))
            except ErroProjeto as e:
                self._cache[k] = e
        r = self._cache[k]
        return None if isinstance(r, ErroProjeto) else r

    def eps(self, N: int, V: float) -> Optional[float]:
        r = self.resultado(N, V)
        return None if r is None else r.eficacia

    def _f(self, N: int, V: float) -> float:
        e = self.eps(N, V)
        if e is None:
            raise _PontoInvalido()
        return e - self.alvo

    def _brent(self, N: int, a: float, b: float) -> Optional[float]:
        try:
            return brentq(lambda V: self._f(N, V), a, b, xtol=1e-4, rtol=1e-6)
        except (_PontoInvalido, ValueError):
            return None

    def velocidade(self, N: int, V_dica: Optional[float] = None) -> BuscaVelocidade:
        p = self.p
        lo, hi = p.velocidade_min, p.velocidade_max
        if V_dica is not None:   # intervalo estreito em torno da solução anterior (iterações do ciclo)
            a, b = max(lo, 0.85 * V_dica), min(hi, 1.15 * V_dica)
            fa, fb = self.eps(N, a), self.eps(N, b)
            if a < b and fa is not None and fb is not None and fa >= self.alvo >= fb:
                V = self._brent(N, a, b)
                if V is not None:
                    return BuscaVelocidade(N, V, "ok")
        # varredura de V_max para baixo: a primeira troca de sinal encontrada é a raiz de maior V
        # (ramo de operação normal: menos tubo e menos perda de carga); para no primeiro achado.
        grade = np.geomspace(lo, hi, N_GRADE_VELOCIDADE)[::-1]
        validos: list = []
        invalidos: list = []
        for V in grade:
            e = self.eps(N, float(V))
            if e is None:
                invalidos.append(float(V))
                continue
            if e == self.alvo:
                return BuscaVelocidade(N, float(V), "ok")
            if validos and (validos[-1][1] - self.alvo) * (e - self.alvo) < 0.0:
                Vr = self._brent(N, float(V), validos[-1][0])
                if Vr is not None:
                    return BuscaVelocidade(N, Vr, "ok")
            validos.append((float(V), e))
        if not validos:
            ult = self._cache.get((int(N), round(float(grade[0]), 7)))
            return BuscaVelocidade(N, None, "sem ponto valido", erro=str(ult) if ult is not None else "",
                                   invalidos=invalidos)
        V_emax, e_max = max(validos, key=lambda t: t[1])
        V_emin, e_min = min(validos, key=lambda t: t[1])
        motivo = "alvo acima do maximo" if e_max < self.alvo else "alvo abaixo do minimo"
        r_max = self.resultado(N, V_emax)
        b = BuscaVelocidade(N, None, motivo, e_max, V_emax, e_min, V_emin, invalidos=invalidos,
                            dTsat_no_max=r_max.dT_sat_queda if r_max is not None else None)
        b._max_interior = V_emax > min(V for V, _ in validos) * (1.0 + 1e-9)
        return b

    def resolver(self, dica: Optional[tuple] = None) -> ResultadoTrocador:
        p = self.p
        if not p.fileiras_automaticas:
            N = p.geometria.n_fileiras
            b = self.velocidade(N, dica[1] if dica and dica[0] == N else None)
            if b.V is None:
                self._levantar([b], fixo=True)
            return self._final(N, b.V, False)
        if dica is not None and 1 <= dica[0] <= p.fileiras_max:
            b = self.velocidade(dica[0], dica[1])
            if b.V is not None:
                return self._final(dica[0], b.V, True)
        Vp = min(max(p.velocidade_face, p.velocidade_min), p.velocidade_max)
        Ns = list(range(1, p.fileiras_max + 1))
        e_Vp = {N: self.eps(N, Vp) for N in Ns}
        N_alto = next((N for N in Ns if e_Vp[N] is not None and e_Vp[N] >= self.alvo), None)
        if N_alto is None:
            cand = [Ns[-1]] if any(e is not None for e in e_Vp.values()) else Ns[::-1]
        else:
            cand = [N_alto] + ([N_alto - 1] if N_alto > 1 else [])
        buscas = [self.velocidade(N) for N in cand]
        sols = [b for b in buscas if b.V is not None]
        if not sols and N_alto is None and Ns[-1] in cand and len(cand) == 1:
            buscas += [self.velocidade(N) for N in Ns[-2::-1]]   # máx. de fileiras inválido: tenta as demais
            sols = [b for b in buscas if b.V is not None]
        if not sols:
            self._levantar(buscas, fixo=False)
        b = min(sols, key=lambda s: (abs(math.log(s.V / Vp)), -s.V))
        return self._final(b.N, b.V, True)

    def _final(self, N: int, V: float, auto: bool) -> ResultadoTrocador:
        r = self.resultado(N, V)
        if r is None:
            raise ErroProjeto(f"ponto final do {self.tipo} inválido ({N} fileiras, {V:.3f} m/s)")
        r = copy.copy(r)
        r.avisos = list(r.avisos)
        r.fileiras_escolhidas_automaticamente = auto
        if abs(r.eficacia - self.alvo) > 2e-3:
            r.avisos.append(f"eficácia obtida {r.eficacia:.3f} difere do alvo {self.alvo:.3f}")
        return r

    def _levantar(self, buscas: list, fixo: bool) -> None:
        msg, por_dp = self._mensagem(buscas, fixo)
        raise (ErroPerdaCarga if por_dp else ErroProjeto)(msg)

    def _mensagem(self, buscas: list, fixo: bool) -> tuple:
        p = self.p
        Nc = p.n_circuitos
        acima = [b for b in buscas if b.motivo == "alvo acima do maximo"]
        abaixo = [b for b in buscas if b.motivo == "alvo abaixo do minimo"]
        if acima:
            b = max(acima, key=lambda t: t.N)
            base = (f"eficácia {self.alvo:.2f} acima do máximo atingível no {self.tipo} com {b.N} fileira(s) e "
                    f"{Nc} circuito(s): ε máx. = {b.eps_max:.3f} a {b.V_eps_max:.2f} m/s "
                    f"(faixa {p.velocidade_min}–{p.velocidade_max} m/s). ")
            if any(t.limitado_perda_carga for t in acima):
                dT = max((t.dTsat_no_max or 0.0) for t in acima)
                return base + (f"O limite vem da perda de carga do refrigerante (queda de saturação de {dT:.1f} K no "
                               "ponto de ε máxima; abaixo dessa velocidade há pinch ou queda excessiva): aumente "
                               "n_circuitos ou o diâmetro do tubo antes de reduzir a eficácia."), True
            if fixo:
                return base + ("Aumente geometria.n_fileiras (ou ative fileiras_automaticas), diminua o passo de aleta "
                               "ou reduza a eficácia."), False
            return base + ("Reduza a eficácia, aumente fileiras_max, diminua o passo de aleta ou aumente a área por "
                           "fileira (com fluxo mássico baixo o lado do refrigerante limita o NTU)."), False
        if abaixo:
            b = min(abaixo, key=lambda t: t.N)
            base = (f"eficácia {self.alvo:.2f} abaixo do mínimo atingível no {self.tipo} com {b.N} fileira(s) na "
                    f"faixa de velocidade: ε mín. = {b.eps_min:.3f} a {b.V_eps_min:.2f} m/s. ")
            if fixo:
                return base + ("Use menos fileiras (geometria.n_fileiras) ou ative fileiras_automaticas, aumente "
                               "velocidade_max ou aumente a eficácia."), False
            return base + "Aumente velocidade_max ou a eficácia.", False
        erros = "; ".join(b.erro for b in buscas if b.erro)[:300]
        return (f"nenhum ponto de operação válido no {self.tipo} para as fileiras testadas "
                f"({', '.join(str(b.N) for b in buscas)}) com {Nc} circuito(s): "
                f"{erros or 'pinch ou perda de carga excessiva'}. Aumente n_circuitos ou o diâmetro do tubo."), True


def _com_circuitos(resolver, p: ParametrosTrocador, tipo: str):
    """Tenta resolver; se a perda de carga impedir (ErroPerdaCarga) e ``circuitos_automaticos``,
    repete com mais circuitos (até ``circuitos_max``) e registra o aviso."""
    pp, nc0 = p, p.n_circuitos
    while True:
        try:
            r = resolver(pp)
        except ErroPerdaCarga:
            if not p.circuitos_automaticos or pp.n_circuitos >= p.circuitos_max:
                raise
            pp = copy.copy(pp)
            pp.n_circuitos += 1
            continue
        if pp.n_circuitos != nc0:
            r.avisos.append(f"n_circuitos aumentado de {nc0} para {pp.n_circuitos}: com {nc0} a perda de carga do "
                            f"refrigerante impedia o projeto do {tipo}")
        return r


def dimensionar_condensador(fl: Fluido, m: float, P_in: float, h_in: float, subresfriamento: float,
                            T_ar_in: float, p: ParametrosTrocador, dica: Optional[tuple] = None) -> ResultadoTrocador:
    """Comprimento de tubo para levar o vapor de descarga (P_in, h_in) a líquido sub-resfriado.
    Com ``p.eficacia`` definido, resolve velocidade de face e fileiras para essa eficácia
    (``dica`` = (N, V) de uma solução anterior acelera as iterações do ciclo)."""
    def resolver(pc):
        def rodar(pp):
            return _condensador_fixo(fl, m, P_in, h_in, subresfriamento, T_ar_in, pp)
        if pc.eficacia is None:
            return rodar(pc)
        d = dica if (dica is None or pc.n_circuitos == p.n_circuitos) else None
        return _Solucionador(rodar, pc, "condensador").resolver(d)
    return _com_circuitos(resolver, p, "condensador")


def dimensionar_evaporador(fl: Fluido, m: float, h_in: float, P_out_alvo: float, superaquecimento: float,
                           T_ar_in: float, p: ParametrosTrocador, fator_umidade: float = 1.0,
                           dica: Optional[tuple] = None) -> ResultadoTrocador:
    """Comprimento de tubo do evaporador: entrada (h_in, pressão a determinar), saída na pressão
    de sucção ``P_out_alvo`` com o superaquecimento pedido. Com ``p.eficacia`` definido, resolve
    velocidade de face e fileiras para essa eficácia."""
    def resolver(pc):
        def rodar(pp):
            return _evaporador_fixo(fl, m, h_in, P_out_alvo, superaquecimento, T_ar_in, pp, fator_umidade)
        if pc.eficacia is None:
            return rodar(pc)
        d = dica if (dica is None or pc.n_circuitos == p.n_circuitos) else None
        return _Solucionador(rodar, pc, "evaporador").resolver(d)
    return _com_circuitos(resolver, p, "evaporador")


def alternativas_fileiras(rodar_fixo, p: ParametrosTrocador, fileiras=range(1, 7)) -> list[dict]:
    """Para a eficácia alvo de ``p``, a solução (V, comprimento, vazão de ar, ΔP do ar) de cada nº de
    fileiras — o compromisso área × ventilador que o projetista escolhe."""
    sol = _Solucionador(rodar_fixo, p, "trocador")
    linhas = []
    for N in fileiras:
        b = sol.velocidade(N)
        if b.V is None:
            linhas.append({"fileiras": N, "viavel": False, "motivo": b.motivo, "eficacia_max": b.eps_max,
                           "eficacia_min": b.eps_min, "limitado_perda_carga": b.limitado_perda_carga})
            continue
        r = sol.resultado(N, b.V)
        linhas.append({"fileiras": N, "viavel": True, "motivo": "ok", "velocidade_face_m_s": b.V,
                       "L_requerido_m": r.L_requerido, "L_adotado_m": r.L_adotado,
                       "tubos_por_fileira": r.tubos_por_fileira, "comprimento_passe_m": r.comprimento_passe,
                       "altura_face_m": r.altura_face, "area_externa_m2": r.area_externa,
                       "vazao_ar_requerida_m3_h": r.vazao_ar_requerida_m3h,
                       "vazao_ar_construtiva_m3_h": r.vazao_ar_m3h, "dP_ar_Pa": r.dP_ar,
                       "potencia_ventilador_W": r.potencia_ventilador_est, "Re_Dc": r.ar.Re_Dc})
    return linhas

In [ ]:
%%writefile predimref/capilar.py
"""Tubo capilar adiabático: modelo homogêneo de equilíbrio (HEM) com blocagem crítica.

Equações (escoamento 1-D, regime permanente, adiabático, horizontal):
  massa:      G = ṁ / A = constante
  energia:    h + G² v² / 2 = h₀ = h_in + G² v_in² / 2
  quantidade de movimento:  −dP = f G² v / (2D) dL + G² dv

Região líquida (sub-resfriada, incompressível): perda de entrada (1 + K) G² v_l / 2 e
    L_liq = 2 D ρ_l (P₁ − P_sat(T_in)) / (f_l G²)
Região bifásica: integração em pressão a partir do ponto de flash; em cada pressão o título
sai da equação da energia (raiz de uma quadrática em x) e dL da quantidade de movimento.
Blocagem (escoamento crítico): quando −dv/dP ≥ 1/G² o numerador (−dP − G² dv) se anula,
dL/dP → 0 e a entropia passa por um máximo (linha de Fanno). Se isso ocorre antes de atingir
a pressão do evaporador, o capilar está blocado e seu comprimento é o comprimento até a
seção crítica (a expansão final ocorre fora do tubo).

Viscosidade bifásica: McAdams (1/μ = x/μ_v + (1−x)/μ_l), Cicchitti (μ = xμ_v + (1−x)μ_l) ou
Dukler (μ = ρ_tp[x v_v μ_v + (1−x) v_l μ_l]); fator de atrito de Churchill.

O HEM despreza o atraso de vaporização (líquido metaestável), o que tende a dar comprimentos
um pouco menores que os reais para a mesma vazão (ver validacao/ com dados de Li et al. 1990).
Referências: Stoecker & Jones (1982) cap. 11; Bittle & Pate (1994) ASHRAE Trans. 100:52-64;
Wongwises et al. (2002) ScienceAsia 28:71-76; Hermes, Melo & Knabben (2010) Appl. Therm. Eng.
30:449-457; ASHRAE Handbook—Refrigeration, cap. "Refrigerant-Control Devices".
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field
from typing import Optional

from scipy.optimize import brentq

from .avisos import ErroProjeto
from .correlacoes.monofasico import fator_atrito_churchill
from .propriedades import Fluido

SERIE_POLEGADA_IN = (0.026, 0.028, 0.031, 0.036, 0.040, 0.042, 0.049, 0.054, 0.059, 0.064, 0.070, 0.075, 0.080)
SERIE_METRICA_MM = (0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.8, 2.0)


@dataclass
class ParametrosCapilar:
    serie: str = "polegada"                 # "polegada" (comercial na América Latina) ou "metrica"
    diametros_mm: Optional[tuple] = None    # sobrepõe a série
    rugosidade_m: float = 0.75e-6
    K_entrada: float = 0.5                  # contração abrupta
    modelo_viscosidade: str = "mcadams"
    L_min: float = 1.0
    L_max: float = 5.0
    L_alvo: float = 2.5
    n_passos: int = 300
    tolerancia_diametro_mm: float = 0.0127  # ±0,0005"

    def diametros(self) -> list:
        if self.diametros_mm is not None:
            return [d * 1e-3 for d in self.diametros_mm]
        if self.serie == "metrica":
            return [d * 1e-3 for d in SERIE_METRICA_MM]
        return [d * 25.4e-3 for d in SERIE_POLEGADA_IN]


def viscosidade_bifasica(x: float, sat, modelo: str) -> float:
    if modelo == "mcadams":
        return 1.0 / (x / sat.muv + (1.0 - x) / sat.mul)
    if modelo == "cicchitti":
        return x * sat.muv + (1.0 - x) * sat.mul
    if modelo == "dukler":
        vv, vl = sat.vv, sat.vl
        return (x * vv * sat.muv + (1.0 - x) * vl * sat.mul) / (x * vv + (1.0 - x) * vl)
    raise ValueError(f"modelo de viscosidade desconhecido: {modelo}")


@dataclass
class ResultadoCapilar:
    D: float
    m: float
    G: float
    L: float
    L_liquido: float
    L_bifasico: float
    blocado: bool
    P_saida: float          # pressão na seção de saída (crítica se blocado)
    x_saida: float
    T_saida: float
    P_flash: float
    perfil: dict = field(default_factory=dict)


def _estado_hem(fl: Fluido, P: float, h0: float, G: float, modelo: str):
    sat = fl.sat(P)
    vf, vg = sat.vl, sat.vv
    vfg = vg - vf
    a = 0.5 * G * G * vfg * vfg
    b = sat.hlv + G * G * vf * vfg
    c = sat.hl + 0.5 * G * G * vf * vf - h0
    x = (-b + math.sqrt(b * b - 4.0 * a * c)) / (2.0 * a) if a > 0 else -c / b
    x = max(x, 0.0)
    v = vf + x * vfg
    s = sat.sl + x * (sat.sv - sat.sl)
    return x, v, s, viscosidade_bifasica(x, sat, modelo), sat.T


def comprimento_capilar(fl: Fluido, m: float, D: float, P_in: float, h_in: float, P_out: float,
                        p: ParametrosCapilar, guardar_perfil: bool = False) -> ResultadoCapilar:
    A = math.pi * D * D / 4.0
    G = m / A
    eD = p.rugosidade_m / D
    e_in = fl.estado_PH(P_in, h_in)
    h0 = h_in + 0.5 * (G / e_in.rho) ** 2
    perfil = {"L": [], "P": [], "T": [], "x": [], "v": [], "s": []} if guardar_perfil else {}
    L_liq = 0.0
    if e_in.x < 0.0:   # líquido sub-resfriado
        v_l = 1.0 / e_in.rho
        P1 = P_in - (1.0 + p.K_entrada) * G * G * v_l / 2.0
        if P1 <= P_out:
            raise ErroProjeto("vazão alta demais para o capilar: a perda de entrada já excede ΔP disponível")

        # O líquido escoa de forma isentálpica (adiabático, energia cinética ~constante). O flash
        # ocorre onde h_l(P) + G² v_l(P)²/2 = h₀. Não é P_sat(T_in): para Tβ > 1 (propano perto de
        # 40 °C) o líquido esfria na expansão e o flash acontece abaixo de P_sat(T_in).
        def g_flash(P):
            sat = fl.sat(P)
            return sat.hl + 0.5 * G * G * sat.vl ** 2 - h0

        if g_flash(P1) <= 0.0:
            P_flash = P1
        elif g_flash(P_out) >= 0.0:
            P_flash = None   # o líquido chega ao evaporador sem vaporizar no tubo
        else:
            P_flash = brentq(g_flash, P_out, P1, xtol=1e-3, rtol=1e-12)
        P_lim = P_out if P_flash is None else P_flash
        sat_lim = fl.sat(P_lim)
        mu_l = 0.5 * (e_in.mu + sat_lim.mul)
        v_lm = 0.5 * (v_l + sat_lim.vl)
        f_l = fator_atrito_churchill(G * D / mu_l, eD)
        L_liq = 2.0 * D * (P1 - P_lim) / (f_l * G * G * v_lm)
        if guardar_perfil:
            T_lim = fl.estado_PH(P_lim, h_in).T
            for LL, PP, TT in ((0.0, P_in, e_in.T), (0.0, P1, e_in.T), (L_liq, P_lim, T_lim)):
                perfil["L"].append(LL); perfil["P"].append(PP); perfil["T"].append(TT)
                perfil["x"].append(0.0); perfil["v"].append(v_l); perfil["s"].append(e_in.s)
        if P_flash is None:
            return ResultadoCapilar(D, m, G, L_liq, L_liq, 0.0, False, P_out, 0.0, sat_lim.T, P_out, perfil)
    else:              # entrada já bifásica (sem sub-resfriamento)
        P_flash = P_in - (1.0 + p.K_entrada) * G * G * e_in.v / 2.0
        if P_flash <= P_out:
            raise ErroProjeto("vazão alta demais para o capilar (entrada bifásica)")

    P = P_flash
    x, v, s, mu, T = _estado_hem(fl, P, h0, G, p.modelo_viscosidade)
    L = L_liq
    dP = (P_flash - P_out) / p.n_passos
    tol = max(1.0, 1e-7 * P_flash)
    blocado = False
    if guardar_perfil:
        for k, val in zip(("L", "P", "T", "x", "v", "s"), (L, P, T, x, v, s)):
            perfil[k].append(val)
    while True:
        P2 = max(P - dP, P_out)
        x2, v2, s2, mu2, T2 = _estado_hem(fl, P2, h0, G, p.modelo_viscosidade)
        num = (P - P2) - G * G * (v2 - v)
        if num <= 0.0 or s2 < s - 1e-9:
            if dP <= tol:
                blocado = True
                break
            dP *= 0.25
            continue
        v_m = 0.5 * (v + v2)
        f = fator_atrito_churchill(G * D / (0.5 * (mu + mu2)), eD)
        L += 2.0 * D * num / (f * G * G * v_m)
        P, x, v, s, mu, T = P2, x2, v2, s2, mu2, T2
        if guardar_perfil:
            for k, val in zip(("L", "P", "T", "x", "v", "s"), (L, P, T, x, v, s)):
                perfil[k].append(val)
        if P <= P_out * (1.0 + 1e-12):
            break
    return ResultadoCapilar(D, m, G, L, L_liq, L - L_liq, blocado, P, x, T, P_flash, perfil)


def vazao_capilar(fl: Fluido, D: float, L: float, P_in: float, h_in: float, P_out: float,
                  p: ParametrosCapilar) -> float:
    """Vazão mássica [kg/s] de um capilar dado (D, L) — problema inverso, por bissecção de Brent."""
    def f(m):
        try:
            return comprimento_capilar(fl, m, D, P_in, h_in, P_out, p).L - L
        except ErroProjeto:
            return -L
    lo, hi = 1e-6, 1e-4
    while f(hi) > 0:
        lo, hi = hi, hi * 2.0
        if hi > 1.0:
            raise ErroProjeto("não foi possível delimitar a vazão do capilar")
    while f(lo) < 0:
        lo /= 2.0
    return brentq(f, lo, hi, xtol=1e-9, rtol=1e-7)


@dataclass
class SelecaoCapilar:
    escolhido: ResultadoCapilar
    candidatos: list
    sensibilidade: dict
    avisos: list


def dimensionar_capilar(fl: Fluido, m: float, P_in: float, h_in: float, P_out: float,
                        p: ParametrosCapilar) -> SelecaoCapilar:
    candidatos, av = [], []
    for D in p.diametros():
        try:
            candidatos.append(comprimento_capilar(fl, m, D, P_in, h_in, P_out, p))
        except ErroProjeto:
            continue
    if not candidatos:
        raise ErroProjeto("nenhum diâmetro da série consegue passar a vazão pedida")
    validos = [c for c in candidatos if p.L_min <= c.L <= p.L_max]
    if validos:
        esc = min(validos, key=lambda c: abs(c.L - p.L_alvo))
    else:
        esc = min(candidatos, key=lambda c: abs(math.log(c.L / p.L_alvo)))
        av.append(f"nenhum diâmetro dá comprimento entre {p.L_min} e {p.L_max} m; escolhido o mais próximo "
                  f"(D = {esc.D * 1e3:.3f} mm, L = {esc.L:.2f} m)")
    esc = comprimento_capilar(fl, m, esc.D, P_in, h_in, P_out, p, guardar_perfil=True)
    tol = p.tolerancia_diametro_mm * 1e-3
    sens = {}
    for nome, Dx in (("D_menos_tol", esc.D - tol), ("D_mais_tol", esc.D + tol)):
        try:
            sens[nome] = comprimento_capilar(fl, m, Dx, P_in, h_in, P_out, p).L
        except ErroProjeto:
            sens[nome] = float("nan")
    m_menos = vazao_capilar(fl, esc.D - tol, esc.L, P_in, h_in, P_out, p)
    m_mais = vazao_capilar(fl, esc.D + tol, esc.L, P_in, h_in, P_out, p)
    sens["vazao_rel_D_menos_tol"] = m_menos / m - 1.0
    sens["vazao_rel_D_mais_tol"] = m_mais / m - 1.0
    if esc.blocado:
        av.append(f"escoamento blocado na saída do capilar (P_crit = {esc.P_saida / 1e5:.3f} bar > "
                  f"P_evap = {P_out / 1e5:.3f} bar): vazão insensível à pressão do evaporador.")
    return SelecaoCapilar(esc, candidatos, sens, av)

In [ ]:
%%writefile predimref/carga_fluido.py
"""Estimativa da carga de refrigerante e comparação com limites de hidrocarbonetos.

Massa = Σ ρ_in-situ · V por trecho. Nos trechos bifásicos dos trocadores a massa específica
in situ usa a fração de vazio de Rouhani-Axelsson (deslizamento entre fases), mais realista
que o modelo homogêneo, que subestima a massa de líquido. No capilar usa-se o perfil HEM.
Inclui linhas, filtro secador, gás na carcaça do compressor e refrigerante dissolvido no óleo
(hidrocarbonetos são muito solúveis em óleo mineral/POE — premissa a calibrar).

Referências de limite de carga para refrigerantes inflamáveis (A2L, A2 e A3; apenas indicativas —
consultar a edição vigente da IEC 60335-2-89 e a regulamentação local, p. ex. NOM no México e NTC
na Colômbia): 150 g (IEC 60335-2-89:2010); 8×LFL e 13×LFL limitado a 1,2 kg (≈ 304 g e 494 g para
R290) nas edições recentes, sempre com teto de 1,2 kg, conforme o tipo de aparelho e requisitos adicionais.
"""
from __future__ import annotations

import math
from dataclasses import dataclass

from .propriedades import CLASSE_SEGURANCA, LFL_KG_M3, Fluido


@dataclass
class ParametrosCargaFluido:
    linha_liquido_L_m: float = 0.3
    linha_liquido_Di_m: float = 3.3e-3
    filtro_secador_cm3: float = 10.0       # volume livre ocupado por líquido
    linha_succao_L_m: float = 1.5
    linha_succao_Di_m: float = 6.0e-3
    linha_descarga_L_m: float = 0.8
    linha_descarga_Di_m: float = 4.75e-3
    volume_livre_carcaca_L: float = 1.0
    T_gas_carcaca_C: float = 50.0
    massa_oleo_kg: float = 0.20
    solubilidade_oleo: float = 0.08        # kg de refrigerante / kg de óleo


def _massa_trocador(res, construtiva: bool = True) -> float:
    """Massa no trocador. A distribuição de fases é a da marcha (serpentina de projeto); a
    construtiva é escalada pelo comprimento (aproximação: a folga de área em operação real
    redistribui as zonas)."""
    A = math.pi * res.geometria.Di ** 2 / 4.0
    fator = res.L_adotado / res.L_requerido if construtiva else 1.0
    return sum(s.rho_media * A * s.L for s in res.segmentos) * res.n_circuitos * fator


def _massa_capilar(cap) -> float:
    pf = cap.perfil
    if not pf or len(pf["L"]) < 2:
        return 0.0
    A = math.pi * cap.D ** 2 / 4.0
    m = 0.0
    for i in range(1, len(pf["L"])):
        dL = pf["L"][i] - pf["L"][i - 1]
        m += A * dL * 2.0 / (pf["v"][i] + pf["v"][i - 1])
    return m


def estimar_carga(fl: Fluido, ciclo, cond, evap, cap, p: ParametrosCargaFluido) -> dict:
    e = ciclo.estados
    vol = lambda L, Di: math.pi * Di ** 2 / 4.0 * L  # noqa: E731
    gas_carcaca = fl.estado_PT(e["1s"].P, p.T_gas_carcaca_C + 273.15).rho * p.volume_livre_carcaca_L * 1e-3
    partes = {
        "condensador": _massa_trocador(cond),
        "evaporador": _massa_trocador(evap),
        "linha de líquido + filtro": e["3"].rho * (vol(p.linha_liquido_L_m, p.linha_liquido_Di_m)
                                                   + p.filtro_secador_cm3 * 1e-6),
        "capilar": _massa_capilar(cap),
        "linha de sucção": e["1s"].rho * vol(p.linha_succao_L_m, p.linha_succao_Di_m),
        "linha de descarga": e["2"].rho * vol(p.linha_descarga_L_m, p.linha_descarga_Di_m),
        "gás na carcaça": gas_carcaca,
        "dissolvido no óleo": p.massa_oleo_kg * p.solubilidade_oleo,
    }
    total = sum(partes.values())
    # mesma conta com as serpentinas de projeto (sem margem), coerente com alvo_validacao
    total_proj = total - partes["condensador"] - partes["evaporador"] + _massa_trocador(cond, False) + \
        _massa_trocador(evap, False)
    limites = {}
    lfl = LFL_KG_M3.get(fl.nome)
    classe = CLASSE_SEGURANCA.get(fl.nome, "?")
    if lfl is not None:   # inflamáveis (A2L, A2, A3): limites de aparelhos autocontidos
        limites = {"IEC 60335-2-89:2010 (150 g)": 0.150,
                   f"8×LFL limitado a 1,2 kg ({min(8 * lfl, 1.2) * 1e3:.0f} g)": min(8 * lfl, 1.2),
                   f"13×LFL limitado a 1,2 kg ({min(13 * lfl, 1.2) * 1e3:.0f} g)": min(13 * lfl, 1.2)}
    return {"partes_kg": partes, "total_kg": total, "total_projeto_kg": total_proj, "limites_kg": limites,
            "inflamavel": lfl is not None, "A3": fl.hidrocarboneto, "classe": classe}

In [ ]:
%%writefile predimref/operacao.py
"""Operação fora do ponto de projeto: casamento compressor–trocadores e pull-down quase estático.

Por que importa para a potência máxima
--------------------------------------
Num compressor alternativo de rotação fixa a potência elétrica não é constante. Com o gabinete
quente (início do pull-down) a pressão de evaporação é alta, a massa específica na sucção também, e
a potência passa por um máximo: para um compressor ideal sem espaço morto e pressão de descarga
fixa, W ∝ P_suc [(P_desc/P_suc)^((k−1)/k) − 1], que é máxima em P_desc/P_suc = k^(k/(k−1)) (≈ 2,9 para
k = 1,13). Por isso o motor é verificado no pull-down e não só no regime (Stoecker & Jones,
Refrigeration and Air Conditioning, 2ª ed., cap. 11). Um limite de potência máxima de ensaio
precisa dizer se vale para o regime ou para o pico; aqui calculam-se os dois.

Modelo do sistema (casamento de componentes; Stoecker & Jones, cap. 14)
-----------------------------------------------------------------------
Geometria fixa (a do projeto) e incógnitas T_evap e T_cond:
    compressor   ṁ = V_desl (N/60) η_v(r) ρ_1s ,   W_el = ṁ (h_2s − h_1s) / η_global
    evaporador   ṁ (h_1 − h_4) = ξ ε_e Ċ_ar,e (T_ar,e − T_evap)
    condensador  ṁ (h_2 − h_3) = ε_c Ċ_ar,c (T_cond − T_ar,c)
η_v pelo mesmo modelo de espaço morto do dimensionamento; η_global e η_eletromecânica constantes
(as do ponto de projeto); superaquecimento, sub-resfriamento e eficácia do trocador interno mantidos;
ε e Ċ_ar das serpentinas de projeto (sem margem), com as eficácias referidas a T_evap (saída do
evaporador) e T_cond (entrada do condensador), como no dimensionamento; quedas de pressão do
refrigerante fixas nas de projeto (P_evap é a de saída e h_4 = h_3i, então elas quase não mexem em Q e W).
No ponto de projeto o modelo reproduz T_evap, T_cond, ṁ, Q e W do dimensionamento (teste de consistência).

Pull-down quase estático (porta fechada)
----------------------------------------
Produto (bebida + embalagem + ``Uso.massa_adicional_kg``) concentrado num nó à T_p, que troca calor
com o ar interno à T_a por hA; o ar tem capacidade térmica desprezível frente à do produto e fica em
equilíbrio a cada instante:
    0 = hA (T_p − T_a) + UA_gab (T_amb − T_a) + Q_base + Q_int − Q_evap(T_a)
    C_p dT_p/dt = −hA (T_p − T_a)   →   t(T_p) = ∫ C_p dT_p / [hA (T_p − T_a(T_p))]
A transmissão é a do dimensionamento (afim em T_a: UA_gab e o acréscimo da base sobre o
compartimento do compressor). Q_evap(T_a), W(T_a) etc. vêm de uma grade de pontos de operação
interpolada por PCHIP (monotônica, sem oscilação); o pico de potência é refinado com o modelo completo.

Hipóteses e limites (estimativa de pré-dimensionamento, não substitui o Amesim):
* ε constante: o lado do ar domina a resistência; ε muda pouco com o coeficiente interno.
* η_global constante: em compressores reais ela varia com a razão de pressão; calibrar com o
  polinômio de catálogo (EN 12900 / AHRI 540) ou com ensaio do compressor candidato.
* superaquecimento fixo: com capilar o evaporador fica sub-alimentado no início do pull-down; o Amesim,
  com o capilar e a carga de fluido, representa isso.
* hA ar–produto é a premissa mais incerta do tempo de pull-down (convecção mista a baixa velocidade,
  garrafas encostadas): calibrar com um ensaio de pull-down do laboratório. Massa do gabinete (liner,
  prateleiras não informadas) desprezada: o tempo real tende a ser maior.
"""
from __future__ import annotations

import copy
import math
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.optimize import brentq, minimize_scalar

from .avisos import Avisos, ErroProjeto
from .ciclo import Ciclo, ParametrosCiclo, resolver_ciclo
from .compressor import ParametrosCompressor, eficiencia_volumetrica
from .constantes import C_para_K
from .gabinete import cargas_internas, transmissao
from .propriedades import Fluido


@dataclass
class ParametrosOperacao:
    """Premissas do pull-down quase estático."""
    T_ambiente_C: Optional[float] = None         # None -> a do projeto
    T_produto_inicial_C: Optional[float] = None  # None -> Uso.T_produto_entrada_C ou a ambiente
    h_produto_W_m2K: float = 5.0                 # ar–garrafa na área lateral (convecção mista, garrafas encostadas)
    hA_produto_W_K: Optional[float] = None       # sobrepõe h × área (use o valor calibrado em ensaio)
    fator_umidade: float = 1.0                   # porta fechada: serpentina praticamente seca
    n_pontos: int = 25                           # grade de temperatura do ar interno
    n_passos: int = 400                          # passos de integração em T_produto


@dataclass
class ModeloSistema:
    """Componentes congelados no projeto (geometria, compressor, ε e Ċ_ar das serpentinas de projeto)."""
    fluido: str
    deslocamento_m3: float
    rotacao_rpm: float
    p_compressor: ParametrosCompressor
    p_ciclo: ParametrosCiclo          # cópia com η_global e η_em do ponto de projeto
    eta_global: float
    eps_evap: float
    C_ar_evap: float
    eps_cond: float
    C_ar_cond: float
    dP_cond: float
    dP_evap: float
    dT_evap_proj: float               # T_ar,e − T_evap no projeto (chute inicial)
    dT_cond_proj: float               # T_cond − T_ar,c no projeto (chute inicial)
    Q_proj: float

    @classmethod
    def do_projeto(cls, ciclo: Ciclo, comp, cond, evap, p_ciclo: ParametrosCiclo,
                   p_comp: ParametrosCompressor) -> "ModeloSistema":
        pc = copy.deepcopy(p_ciclo)
        pc.eta_global_sem_cop = ciclo.eta_global
        pc.eta_eletromecanica = ciclo.eta_em
        return cls(ciclo.fluido, comp.deslocamento_cm3 * 1e-6, comp.rotacao_rpm, copy.deepcopy(p_comp), pc,
                   ciclo.eta_global, evap.eficacia, evap.C_ar, cond.eficacia, cond.C_ar, cond.dP_ref, evap.dP_ref,
                   evap.T_ar_in - ciclo.T_evap, ciclo.T_cond - cond.T_ar_in, evap.Q)


def modelo_do_projeto(res) -> ModeloSistema:
    p = res.parametros
    return ModeloSistema.do_projeto(res.ciclo, res.compressor, res.condensador, res.evaporador, p.ciclo, p.compressor)


@dataclass
class PontoOperacao:
    T_ar_evap: float
    T_ar_cond: float
    T_evap: float
    T_cond: float
    P_evap: float
    P_cond: float
    m: float
    Q_evap: float
    Q_cond: float
    W_el: float
    eta_v: float
    T_descarga: float

    @property
    def COP(self) -> float:
        return self.Q_evap / self.W_el

    @property
    def razao_pressao(self) -> float:
        return self.P_cond / self.P_evap


def _avaliar(modelo: ModeloSistema, fl: Fluido, T_e: float, T_c: float):
    c = resolver_ciclo(fl, T_e, T_c, modelo.p_ciclo, None, modelo.dP_cond, modelo.dP_evap, Avisos())
    s = c.estados["1s"]
    n = modelo.p_compressor.expoente_reexpansao or fl.kappa(s.P, s.T, s.rho)
    eta_v = eficiencia_volumetrica(c.estados["2"].P / s.P, modelo.p_compressor, n)
    m = modelo.deslocamento_m3 * modelo.rotacao_rpm / 60.0 * eta_v * s.rho
    return c, m, eta_v


def ponto_operacao(modelo: ModeloSistema, T_ar_evap: float, T_ar_cond: float, xi: float = 1.0,
                   chute: Optional[tuple] = None, fl: Optional[Fluido] = None) -> PontoOperacao:
    """Ponto de equilíbrio compressor–evaporador–condensador para as temperaturas de entrada do ar dadas.

    Newton 2D com jacobiana por diferenças finitas a partir do chute (ou do ΔT de projeto); se não
    convergir, bissecção aninhada (T_cond para cada T_evap e depois T_evap), que é robusta porque os dois
    resíduos são monotônicos."""
    fl = fl or Fluido(modelo.fluido)
    Ce = xi * modelo.eps_evap * modelo.C_ar_evap
    Cc = modelo.eps_cond * modelo.C_ar_cond
    Q_ref = max(modelo.Q_proj, 1.0)
    T_e_min = fl.Ttriplo + 5.5
    T_c_max = fl.Tc - 5.5
    SR = modelo.p_ciclo.subresfriamento

    def residuos(T_e: float, T_c: float):
        c, m, eta_v = _avaliar(modelo, fl, T_e, T_c)
        e = c.estados
        r = np.array([(m * c.q_evap - Ce * (T_ar_evap - T_e)) / Q_ref,
                      (m * (e["2"].h - e["3"].h) - Cc * (T_c - T_ar_cond)) / Q_ref])
        return r, (c, m, eta_v)

    def valido(T_e: float, T_c: float) -> bool:
        return T_e_min < T_e < T_ar_evap and T_ar_cond < T_c < T_c_max and T_c - SR > T_e + 0.5

    x = np.array(chute if chute is not None else (T_ar_evap - modelo.dT_evap_proj, T_ar_cond + modelo.dT_cond_proj),
                 dtype=float)
    sol = None
    if valido(*x):
        try:
            sol = _newton(residuos, valido, x)
        except (ErroProjeto, ValueError, ZeroDivisionError, np.linalg.LinAlgError):
            sol = None
    if sol is None:
        sol = _bisseccao(residuos, fl, T_ar_evap, T_ar_cond, T_e_min, T_c_max, SR)
    (T_e, T_c), (c, m, eta_v) = sol
    e = c.estados
    return PontoOperacao(T_ar_evap, T_ar_cond, T_e, T_c, e["1"].P, e["2"].P, m, m * c.q_evap,
                         m * (e["2"].h - e["3"].h), m * c.w_s / modelo.eta_global, eta_v, e["2"].T)


def _newton(residuos, valido, x, tol: float = 1e-9, max_iter: int = 30):
    r, aux = residuos(*x)
    for _ in range(max_iter):
        if np.max(np.abs(r)) < tol:
            return x, aux
        J = np.empty((2, 2))
        for j in range(2):
            dx = np.zeros(2)
            dx[j] = 1e-3
            J[:, j] = (residuos(*(x + dx))[0] - r) / 1e-3
        passo = np.linalg.solve(J, -r)
        lim = np.max(np.abs(passo))
        if lim > 5.0:
            passo *= 5.0 / lim
        lam = 1.0
        while lam > 1e-3:
            xn = x + lam * passo
            if valido(*xn):
                rn, auxn = residuos(*xn)
                if np.max(np.abs(rn)) < np.max(np.abs(r)):
                    x, r, aux = xn, rn, auxn
                    break
            lam *= 0.5
        else:
            return None
    return (x, aux) if np.max(np.abs(r)) < tol else None


def _bisseccao(residuos, fl, T_ar_e, T_ar_c, T_e_min, T_c_max, SR):
    def Tc_de(T_e):
        lo = max(T_ar_c + 1e-3, T_e + SR + 0.6)
        hi = min(T_c_max, T_ar_c + 150.0)
        f = lambda T_c: residuos(T_e, T_c)[0][1]  # noqa: E731
        if f(lo) <= 0.0:
            return lo
        if f(hi) > 0.0:
            raise ErroProjeto(f"ponto de operação: o condensador não rejeita o calor nem com T_cond = "
                              f"{hi - 273.15:.0f} °C (ar a {T_ar_c - 273.15:.1f} °C).")
        return brentq(f, lo, hi, xtol=1e-7)

    def g(T_e):
        return residuos(T_e, Tc_de(T_e))[0][0]

    lo, hi = max(T_e_min, T_ar_e - 80.0), T_ar_e - 1e-3
    if g(lo) > 0.0:
        raise ErroProjeto("ponto de operação: compressor grande demais para o evaporador mesmo no limite "
                          "inferior de T_evap.")
    T_e = brentq(g, lo, hi, xtol=1e-7)
    T_c = Tc_de(T_e)
    return np.array([T_e, T_c]), residuos(T_e, T_c)[1]


# =========================================================================== pull-down
@dataclass
class ResultadoPulldown:
    T_ambiente: float
    T_produto_inicial: float
    T_alvo: float
    C_produto_J_K: float
    hA_produto_W_K: float
    UA_gabinete_W_K: float
    Q_internas_W: float
    fator_umidade: float
    concluido: bool                  # o produto chega a T_alvo
    tempo_s: float                   # até T_alvo (ou até 0,5 K da temperatura de equilíbrio, se não concluir)
    T_produto_equilibrio: Optional[float]
    t: np.ndarray
    T_produto: np.ndarray
    T_ar: np.ndarray
    T_evap: np.ndarray
    T_cond: np.ndarray
    Q_evap: np.ndarray
    W_el: np.ndarray
    T_descarga: np.ndarray
    pico: PontoOperacao              # máximo de W entre o ar no fim do pull-down e T_p0 (inclui a partida)
    t_pico_s: float
    pico_na_partida: bool            # o máximo está na partida (ar ainda acima do valor quase estático)
    W_pico_trajetoria: float         # máximo na trajetória quase estática (t > 0)
    E_compressor_J: float
    grade: dict = field(default_factory=dict)

    @property
    def tempo_h(self) -> float:
        return self.tempo_s / 3600.0

    @property
    def W_pico(self) -> float:
        return self.pico.W_el

    @property
    def W_inicial(self) -> float:
        return float(self.W_el[0])

    @property
    def W_final(self) -> float:
        return float(self.W_el[-1])

    @property
    def E_compressor_kWh(self) -> float:
        return self.E_compressor_J / 3.6e6

    @property
    def T_descarga_max(self) -> float:
        return float(np.max(self.T_descarga))

    def curva(self, n: int = 61) -> dict:
        """Curva reamostrada em tempo uniforme (para exportar e comparar com o Amesim)."""
        tt = np.linspace(0.0, self.t[-1], n)
        f = lambda y: np.interp(tt, self.t, y).tolist()  # noqa: E731
        return {"t_s": tt.tolist(), "T_produto_K": f(self.T_produto), "T_ar_K": f(self.T_ar),
                "T_evap_K": f(self.T_evap), "T_cond_K": f(self.T_cond), "Q_evap_W": f(self.Q_evap),
                "W_el_W": f(self.W_el), "T_descarga_K": f(self.T_descarga)}


def simular_pulldown(modelo: ModeloSistema, cap, gab, uso, garrafa, pc: ParametrosCiclo, T_amb_proj_C: float,
                     T_alvo_C: float, po: ParametrosOperacao) -> ResultadoPulldown:
    """Pull-down de portas fechadas da carga cheia, do produto à T_inicial até T_alvo (ver docstring do módulo)."""
    fl = Fluido(modelo.fluido)
    T_amb_C = T_amb_proj_C if po.T_ambiente_C is None else po.T_ambiente_C
    if po.T_produto_inicial_C is not None:
        T_p0_C = po.T_produto_inicial_C
    else:
        T_p0_C = T_amb_C if uso.T_produto_entrada_C is None else uso.T_produto_entrada_C
    if T_p0_C <= T_alvo_C + 0.5:
        raise ErroProjeto("pull-down: o produto inicial deve estar acima de T_alvo + 0,5 K.")
    if T_amb_C <= T_alvo_C:
        raise ErroProjeto("pull-down: a temperatura ambiente deve ser maior que a alvo.")
    if po.n_pontos < 5 or po.n_passos < 20:
        raise ErroProjeto("pull-down: use n_pontos ≥ 5 e n_passos ≥ 20.")
    T_amb, T_p0, T_f = C_para_K(T_amb_C), C_para_K(T_p0_C), C_para_K(T_alvo_C)
    E = (cap.n_garrafas * garrafa.energia_resfriamento(T_p0_C, T_alvo_C)
         + uso.massa_adicional_kg * uso.cp_adicional * (T_p0_C - T_alvo_C))
    C_p = E / (T_p0_C - T_alvo_C)
    hA = po.hA_produto_W_K
    if hA is None:
        hA = po.h_produto_W_m2K * cap.n_garrafas * math.pi * garrafa.diametro_m * garrafa.altura_m
    if not (hA > 0.0 and C_p > 0.0):
        raise ErroProjeto("pull-down: hA do produto e capacidade térmica precisam ser positivos.")
    tr = transmissao(cap, gab, T_amb, T_f)
    UA, Qt_f = tr["UA_total"], tr["Q_total"]
    Q_int = cargas_internas(uso)
    T_ar_c = T_amb + pc.dT_recirculacao_cond
    dTr = pc.dT_retorno_evap
    xi = po.fator_umidade

    def carga_sem_produto(T_a):
        return Qt_f + UA * (T_f - T_a) + Q_int

    def ponto(T_a, chute=None):
        return ponto_operacao(modelo, T_a + dTr, T_ar_c, xi, chute, fl)

    # grade de T_a: do limite inferior (ar no fim do pull-down) até T_p0 -------------------------------
    p_f = ponto(T_f)
    N0 = p_f.Q_evap - carga_sem_produto(T_f)
    T_lo = T_f - max(N0, 0.0) / hA - 1.0
    Ts = np.linspace(T_lo, T_p0, po.n_pontos)
    i0 = int(np.argmin(np.abs(Ts - T_f)))
    pts: list = [None] * len(Ts)
    pts[i0] = ponto(Ts[i0], (p_f.T_evap, p_f.T_cond))
    for ordem in (range(i0 + 1, len(Ts)), range(i0 - 1, -1, -1)):
        anterior = i0
        for i in ordem:
            q = pts[anterior]
            pts[i] = ponto(Ts[i], (q.T_evap + (Ts[i] - Ts[anterior]) * 0.6, q.T_cond))
            anterior = i
    arr = {k: np.array([getattr(q, k) for q in pts]) for k in ("Q_evap", "W_el", "T_evap", "T_cond", "T_descarga")}
    it = {k: PchipInterpolator(Ts, v) for k, v in arr.items()}

    def liquido(T_a):  # calor líquido que o evaporador tira do produto com o ar à T_a
        return float(it["Q_evap"](T_a)) - carga_sem_produto(T_a)

    # fim: T_alvo, ou perto do equilíbrio se o sistema não chegar lá -------------------------------
    concluido, T_eq = True, None
    if liquido(T_f) <= 0.0:
        concluido = False
        if liquido(T_p0) <= 0.0:
            raise ErroProjeto("pull-down: a capacidade não supera a carga do gabinete nem com o produto quente.")
        T_eq = brentq(liquido, T_f, T_p0, xtol=1e-6)
        T_fim = T_eq + 0.5
        if T_fim >= T_p0:
            raise ErroProjeto("pull-down: o equilíbrio fica a menos de 0,5 K da temperatura inicial do produto.")
    else:
        T_fim = T_f
    Tp = np.linspace(T_p0, T_fim, po.n_passos + 1)
    Ta = np.empty_like(Tp)
    for k, T_p in enumerate(Tp):
        F = lambda T_a: liquido(T_a) - hA * (T_p - T_a)  # noqa: E731
        Ta[k] = brentq(F, T_lo, T_p, xtol=1e-7) if F(T_p) > 0.0 else T_p
    Q_prod = hA * (Tp - Ta)
    if np.any(Q_prod <= 0.0):
        raise ErroProjeto("pull-down: troca ar–produto nula na integração (revise hA e a capacidade).")
    dtdT = C_p / Q_prod                                  # s/K (T_p decrescente)
    t = -cumulative_trapezoid(dtdT, Tp, initial=0.0)
    W = it["W_el"](Ta)
    E_comp = -float(np.trapezoid(W * dtdT, Tp)) if hasattr(np, "trapezoid") else -float(np.trapz(W * dtdT, Tp))

    # pico de potência --------------------------------------------------------------------------
    # Na trajetória quase estática o ar já está abaixo do produto em t = 0⁺; nos primeiros minutos
    # (gabinete e produto encharcados à T_p0) o evaporador ainda vê ar mais quente. O pico reportado é o
    # máximo de W(T_a) entre o ar no fim do pull-down e T_p0: limite superior que inclui a partida e não
    # depende de hA (despreza a inércia térmica do evaporador e o transitório de pressões da partida).
    def chute(T_a):
        q = pts[int(np.argmin(np.abs(Ts - T_a)))]
        return (q.T_evap + (T_a + dTr - q.T_ar_evap) * 0.6, q.T_cond)

    TT = np.linspace(Ta[-1], T_p0, 400)
    WW = it["W_el"](TT)
    k = int(np.argmax(WW))
    if 0 < k < len(TT) - 1:
        opt = minimize_scalar(lambda T: -ponto(T, chute(T)).W_el, bounds=(TT[k - 1], TT[k + 1]), method="bounded",
                              options={"xatol": 1e-3})
        T_pico = float(opt.x)
    else:
        T_pico = float(TT[k])
    pico = ponto(T_pico, chute(T_pico))
    na_partida = T_pico > Ta[0]
    t_pico = 0.0 if na_partida else float(np.interp(-T_pico, -Ta, t))
    return ResultadoPulldown(T_amb, T_p0, T_f, C_p, hA, UA, Q_int, xi, concluido, float(t[-1]), T_eq, t, Tp, Ta,
                             it["T_evap"](Ta), it["T_cond"](Ta), it["Q_evap"](Ta), W, it["T_descarga"](Ta),
                             pico, t_pico, na_partida, float(np.max(W)), E_comp, {"T_ar": Ts, **arr})


__all__ = ["ParametrosOperacao", "ModeloSistema", "modelo_do_projeto", "PontoOperacao", "ponto_operacao",
           "ResultadoPulldown", "simular_pulldown"]

In [ ]:
%%writefile predimref/projeto.py
"""Orquestração do pré-dimensionamento completo (entradas do cliente -> parâmetros para o Amesim)."""
from __future__ import annotations

import copy
import dataclasses
import math
from dataclasses import dataclass, field
from typing import Optional

from .avisos import Avisos, ErroProjeto
from .capilar import ParametrosCapilar, SelecaoCapilar, dimensionar_capilar
from .carga_fluido import ParametrosCargaFluido, estimar_carga
from .ciclo import (BalancoSistema, Ciclo, ParametrosCiclo, balanco, escala_dT_para_cop, exergia,
                    resolver_ciclo)
from .compressor import ParametrosCompressor, ResultadoCompressor, dimensionar_compressor
from .constantes import C_para_K
from .gabinete import (Capacidade, CargaTermica, Gabinete, Garrafa, Uso, calcular_carga,
                       resolver_capacidade)
from .operacao import ModeloSistema, ParametrosOperacao, ResultadoPulldown, simular_pulldown
from .propriedades import Fluido, ar_seco
from .trocador import (ParametrosTrocador, ResultadoTrocador, condensador_padrao,
                       dimensionar_condensador, dimensionar_evaporador, evaporador_padrao)


@dataclass
class EntradasProjeto:
    """As quatro entradas do cliente (+ umidade e fluido).

    COP: capacidade frigorífica do evaporador / potência elétrica do compressor [W/W],
         na condição de projeto. ``None`` usa ``ParametrosCiclo.eta_global_sem_cop``.
    volume_interno_L ou n_garrafas: capacidade do refrigerador (garrafas de 600 mL por padrão).
    eficacia_condensador / eficacia_evaporador: ε do lado do ar referida à saturação,
         ε = (T_ar,saída − T_ar,entrada)/(T_sat − T_ar,entrada). Define a vazão de ar e, com ela, a
         geometria (fileiras, velocidade de face, comprimento). ``None`` usa a velocidade de face fixa
         de ``ParametrosTrocador`` e a eficácia vira resultado.
    potencia_max_compressor_W: limite de potência elétrica do compressor (requisito de ensaio). O critério
         (regime ou pico de pull-down) e o modo (limitar ou só verificar) estão em ``RestricaoPotencia``.
    """
    COP: Optional[float]
    T_ambiente_C: float
    T_alvo_C: float
    volume_interno_L: Optional[float] = None
    n_garrafas: Optional[int] = None
    UR_ambiente: float = 0.60
    fluido: str = "R290"
    eficacia_condensador: Optional[float] = 0.65
    eficacia_evaporador: Optional[float] = 0.65
    potencia_max_compressor_W: Optional[float] = None


@dataclass
class RestricaoPotencia:
    """Como aplicar ``EntradasProjeto.potencia_max_compressor_W``.

    criterio: "regime"        -> potência no ponto de projeto (gabinete na T_alvo, compressor ligado);
              "pico_pulldown" -> maior potência estimada no pull-down de portas fechadas, incluindo a
                                 partida com o gabinete encharcado (``operacao.simular_pulldown``).
    modo:     "limitar"   -> se o limite for violado, reduz a capacidade de projeto (compressor menor) até
                             atender e avisa o que muda (fração de tempo ligado, tempo de pull-down);
              "verificar" -> mantém o projeto pelos requisitos de carga e só avisa.
    Com COP fixo, potência e capacidade andam juntas (W = Q/COP): reduzir a potência sem perder capacidade
    exige COP maior (trocadores maiores, ΔT menores, compressor mais eficiente)."""
    criterio: str = "regime"
    modo: str = "limitar"
    tolerancia: float = 0.01      # banda aceita abaixo do limite no critério de pico (iterativo)
    max_iter: int = 8


@dataclass
class ParametrosProjeto:
    garrafa: Garrafa = field(default_factory=Garrafa)
    gabinete: Gabinete = field(default_factory=Gabinete)
    uso: Uso = field(default_factory=Uso)
    ciclo: ParametrosCiclo = field(default_factory=ParametrosCiclo)
    compressor: ParametrosCompressor = field(default_factory=ParametrosCompressor)
    condensador: ParametrosTrocador = field(default_factory=condensador_padrao)
    evaporador: ParametrosTrocador = field(default_factory=evaporador_padrao)
    capilar: ParametrosCapilar = field(default_factory=ParametrosCapilar)
    carga_fluido: ParametrosCargaFluido = field(default_factory=ParametrosCargaFluido)
    potencia_ventilador_cond_W: Optional[float] = None  # None -> estimada pela ΔP do ar e η do ventilador
    restricao_potencia: RestricaoPotencia = field(default_factory=RestricaoPotencia)
    operacao: ParametrosOperacao = field(default_factory=ParametrosOperacao)
    simular_pulldown: bool = True
    analise_sensibilidade: bool = True
    max_iter: int = 10


@dataclass
class ResultadoProjeto:
    entradas: EntradasProjeto
    parametros: ParametrosProjeto
    capacidade: Capacidade
    carga: CargaTermica
    ciclo: Ciclo
    balanco: BalancoSistema
    compressor: ResultadoCompressor
    condensador: ResultadoTrocador
    evaporador: ResultadoTrocador
    capilar: SelecaoCapilar
    carga_refrigerante: dict
    exergia: dict
    energia: dict
    sensibilidade: dict
    avisos: Avisos
    iteracoes: int
    pulldown: Optional[ResultadoPulldown] = None
    potencia: dict = field(default_factory=dict)

    def resumo(self) -> str:
        from .relatorio import resumo_texto
        return resumo_texto(self)

    def para_dict(self) -> dict:
        from .relatorio import para_dict
        return para_dict(self)

    def exportar_json(self, caminho: str) -> str:
        from .relatorio import exportar_json
        return exportar_json(self, caminho)


def dimensionar(entradas: EntradasProjeto, parametros: Optional[ParametrosProjeto] = None) -> ResultadoProjeto:
    """Pré-dimensiona o sistema completo. Lança ``ErroProjeto`` para entradas impossíveis (e também,
    com a causa original encadeada, para falhas numéricas não previstas)."""
    try:
        return _dimensionar(entradas, parametros)
    except ErroProjeto:
        raise
    except (ValueError, TypeError, ZeroDivisionError, OverflowError) as err:
        raise ErroProjeto(f"falha numérica não prevista ({type(err).__name__}: {err}); revise as entradas "
                          "(e reporte o caso se elas forem razoáveis)") from err


def _acoplar(fl, entradas, p, carga, T_evap, T_cond, T_ar_evap, T_ar_cond, dicas):
    """Ciclo <-> trocadores: as quedas de pressão realimentam o ciclo até convergir."""
    pc = p.ciclo
    dPc = dPe = 0.0
    dica_c, dica_e = dicas
    it, conv = 0, False
    for it in range(1, p.max_iter + 1):
        av_ciclo = Avisos()
        try:
            ciclo = resolver_ciclo(fl, T_evap, T_cond, pc, entradas.COP, dPc, dPe, av_ciclo)
        except ErroProjeto as err:
            raise ErroProjeto(str(err) + _sugestao_cop(fl, entradas, T_ar_evap, T_ar_cond, pc)) from None
        bal = balanco(ciclo, carga.Q_projeto, av_ciclo)
        e = ciclo.estados
        cond = dimensionar_condensador(fl, bal.m, e["2"].P, e["2"].h, pc.subresfriamento, T_ar_cond, p.condensador,
                                       dica_c)
        evap = dimensionar_evaporador(fl, bal.m, e["3i"].h, e["1"].P, pc.superaquecimento, T_ar_evap,
                                      p.evaporador, carga.fator_umidade, dica_e)
        # o nº de circuitos escolhido (automático por perda de carga) passa a valer nas iterações seguintes
        p.condensador.n_circuitos, p.evaporador.n_circuitos = cond.n_circuitos, evap.n_circuitos
        dica_c = (cond.n_fileiras, cond.velocidade_face)
        dica_e = (evap.n_fileiras, evap.velocidade_face)
        conv = abs(cond.dP_ref - dPc) < 20.0 and abs(evap.dP_ref - dPe) < 20.0
        dPc, dPe = cond.dP_ref, evap.dP_ref
        if conv:
            break
    return ciclo, bal, cond, evap, av_ciclo, it, conv, (dica_c, dica_e)


def umidade_equilibrio(entradas: EntradasProjeto, p: ParametrosProjeto, carga: CargaTermica, bal: BalancoSistema,
                       evap: ResultadoTrocador) -> dict:
    """Umidade interna de regime: a água que entra por infiltração é removida pela serpentina.

    Serpentina úmida pelo modelo de superfície efetiva com Lewis = 1 (Braun, Klein & Mitchell 1989,
    ASHRAE Trans. 95(2); ASHRAE Handbook—Fundamentals, cap. "Psychrometrics"/"Air-cooling coils"):
        T_s,ef = T_e − (T_e − T_s)/(1 − e^(−NTU_o)),  NTU_o = h_o η_o A_o / (ṁ_ar c_p)
        ṁ_água,serp = ṁ_ar (1 − e^(−NTU_o)) (W_i − W_sat(T_s,ef))
    Em média diária, com o compressor ligado a fração f = Q_regime/Q_evap:
        A (W_e − W_i) = f ṁ_ar (1 − e^(−NTU_o)) (W_i − W_s)  →  W_i = (A W_e + B W_s)/(A + B)
    com A = aberturas/dia × m_ar,abertura / 86400 [kg_as/s] e B = f ṁ_ar (1 − e^(−NTU_o))."""
    import CoolProp.CoolProp as CP
    from .propriedades import ar_umido
    T_int = C_para_K(entradas.T_alvo_C)
    W_e = ar_umido(C_para_K(entradas.T_ambiente_C), entradas.UR_ambiente)["W"]
    cp = ar_seco(evap.T_ar_in).cp
    m_ar = evap.C_ar / cp
    NTU_o = evap.ar.h * evap.ar.eta_superficie * evap.projeto.area_externa / (m_ar * cp)
    eps_o = 1.0 - math.exp(-NTU_o)
    T_s = evap.T_ar_in - (evap.T_ar_in - evap.T_ar_out) / eps_o
    W_s = CP.HAPropsSI("W", "T", T_s, "P", 101325.0, "R", 1.0)
    f_on = min(carga.Q_regime / bal.Q_evap, 1.0)
    A = p.uso.aberturas_por_dia * carga.infiltracao.get("m_ar_abertura", 0.0) / 86400.0
    B = f_on * m_ar * eps_o
    W_i = (A * W_e + B * W_s) / (A + B) if A + B > 0 else W_s
    UR = min(CP.HAPropsSI("R", "T", T_int, "P", 101325.0, "W", W_i), 1.0)
    return {"W_interno": W_i, "UR_interna": UR, "T_superficie_efetiva_K": T_s, "W_superficie": W_s,
            "NTU_ar": NTU_o, "fracao_ligado": f_on}


@dataclass
class _Nucleo:
    carga: CargaTermica
    ciclo: Ciclo
    bal: BalancoSistema
    cond: ResultadoTrocador
    evap: ResultadoTrocador
    comp: ResultadoCompressor
    av_carga: Avisos
    av_ciclo: Avisos
    it: int
    conv: bool
    umid: dict
    umid_conv: bool
    Q_requisitos: float          # capacidade de projeto pelos requisitos de carga (antes do teto de potência)
    dicas: tuple


def _aplicar_teto(carga: CargaTermica, Q_teto: Optional[float]) -> None:
    """Limita a capacidade de projeto (restrição de potência). A fração latente fica a mesma: com um
    compressor menor ele passa mais tempo ligado e a água removida por hora ligada cai na mesma proporção
    da capacidade, então o balanço de umidade continua fechado com o mesmo ξ."""
    if Q_teto is None or carga.Q_projeto <= Q_teto:
        return
    carga.Q_latente_projeto *= Q_teto / carga.Q_projeto
    carga.Q_projeto = Q_teto
    carga.caso_governante = f"limitado pela potência máxima do compressor (requisitos: {carga.caso_governante})"


def _nucleo(fl, entradas: EntradasProjeto, p: ParametrosProjeto, cap: Capacidade, temps: tuple, dicas: tuple,
            Q_teto: Optional[float]) -> _Nucleo:
    """Carga térmica <-> ciclo <-> trocadores, fechando o balanço de umidade do gabinete, e compressor."""
    T_evap, T_cond, T_ar_evap, T_ar_cond = temps
    equilibrio = p.uso.umidade_interna == "equilibrio"
    umid, umid_conv = None, True
    for _ in range(1, 5 if equilibrio else 2):
        av_carga = Avisos()
        carga = calcular_carga(cap, p.gabinete, p.uso, p.garrafa, entradas.T_ambiente_C, entradas.UR_ambiente,
                               entradas.T_alvo_C, av_carga)
        Q_req = carga.Q_projeto
        _aplicar_teto(carga, Q_teto)
        ciclo, bal, cond, evap, av_ciclo, it, conv, dicas = _acoplar(fl, entradas, p, carga, T_evap, T_cond,
                                                                     T_ar_evap, T_ar_cond, dicas)
        umid = umidade_equilibrio(entradas, p, carga, bal, evap)
        if not equilibrio:
            break
        UR_ant = p.uso.UR_interna
        p.uso.UR_interna = umid["UR_interna"]
        if abs(umid["UR_interna"] - UR_ant) < 0.003:
            break
    else:
        umid_conv = False
    carga.infiltracao["umidade"] = umid
    e = ciclo.estados
    comp = dimensionar_compressor(fl, bal.m, e["1s"], e["2"].P, bal.W_el, e["2"].T, ciclo.eta_global, p.compressor)
    return _Nucleo(carga, ciclo, bal, cond, evap, comp, av_carga, av_ciclo, it, conv, umid, umid_conv, Q_req, dicas)


def _pulldown(nuc: _Nucleo, p: ParametrosProjeto, cap: Capacidade, entradas: EntradasProjeto) -> ResultadoPulldown:
    modelo = ModeloSistema.do_projeto(nuc.ciclo, nuc.comp, nuc.cond, nuc.evap, p.ciclo, p.compressor)
    return simular_pulldown(modelo, cap, p.gabinete, p.uso, p.garrafa, p.ciclo, entradas.T_ambiente_C,
                            entradas.T_alvo_C, p.operacao)


def _potencia_criterio(rp: RestricaoPotencia, nuc: _Nucleo, pd: Optional[ResultadoPulldown],
                       pd_erro: Optional[str]) -> float:
    if rp.criterio == "regime":
        return nuc.bal.W_el
    if pd is None:
        raise ErroProjeto(f"limite de potência no pico de pull-down: a simulação de pull-down falhou ({pd_erro}).")
    return max(pd.W_pico, nuc.bal.W_el)


def _proximo_teto(hist: list, W_alvo: float) -> float:
    """Secante em (Q_projeto, W_critério); W cresce com Q (compressor maior)."""
    Q1, W1 = hist[-1]
    if len(hist) >= 2:
        Q0, W0 = hist[-2]
        if Q1 != Q0 and (W1 - W0) / (Q1 - Q0) > 0.0:
            return Q1 + (W_alvo - W1) * (Q1 - Q0) / (W1 - W0)
    return Q1 * W_alvo / W1


def _cop_necessario(nuc: _Nucleo, W_c: float, W_max: float, Q: float) -> float:
    """COP para entregar a capacidade Q com o critério de potência em W_max (W_critério ∝ Q/COP)."""
    return nuc.ciclo.COP * (W_c / W_max) * (Q / nuc.carga.Q_projeto)


def _mensagem_potencia_inviavel(fl, entradas, p, nuc, W_c, W_max, rp, T_ar_evap, T_ar_cond) -> str:
    car, u = nuc.carga, p.uso
    nome = "potência de regime" if rp.criterio == "regime" else "pico de potência no pull-down"
    Q_teto = car.Q_projeto * W_max / W_c
    cop_min = _cop_necessario(nuc, W_c, W_max, car.Q_regime * (1.0 + u.margem_seguranca))
    cop_req = _cop_necessario(nuc, W_c, W_max, nuc.Q_requisitos)
    txt = (f"limite de {W_max:.0f} W ({nome}) inviável com COP {nuc.ciclo.COP:.2f}: o maior compressor que o respeita "
           f"entrega ≈ {Q_teto:.0f} W no ponto de projeto, abaixo da carga média de regime ({car.Q_regime:.0f} W) — o "
           f"gabinete não manteria {entradas.T_alvo_C:.1f} °C nem com o compressor 100 % ligado. Com W ≤ {W_max:.0f} W "
           f"seriam necessários COP ≥ {cop_min:.2f} para o regime com a margem de {u.margem_seguranca:.0%} e "
           f"COP ≈ {cop_req:.2f} para todos os requisitos (fator de utilização {u.fator_utilizacao_max:.0%}, "
           f"pull-down em {u.tempo_pulldown_h:g} h).")
    return txt + _sugestao_cop(fl, dataclasses.replace(entradas, COP=cop_req), T_ar_evap, T_ar_cond, p.ciclo)


def _avisos_potencia(av: Avisos, entradas, p, nuc: _Nucleo, pd, pd_erro, W_max, rp, fl, T_ar_evap,
                     T_ar_cond) -> dict:
    """Avisos e resumo da restrição de potência e do pull-down estimado."""
    car, bal, u = nuc.carga, nuc.bal, p.uso
    limitado = car.Q_projeto < nuc.Q_requisitos * (1.0 - 1e-9)
    if pd_erro:
        av.atencao("pull-down", f"simulação de pull-down não concluída: {pd_erro}")
    if pd is not None:
        quando = "na partida, com o gabinete ainda encharcado" if pd.pico_na_partida else \
            f"em t ≈ {pd.t_pico_s / 3600.0:.1f} h"
        av.info("pull-down", f"estimativa quase estática (porta fechada, produto de {pd.T_produto_inicial - 273.15:.1f} a "
                f"{pd.T_alvo - 273.15:.1f} °C, ambiente {pd.T_ambiente - 273.15:.1f} °C, hA ar–produto "
                f"{pd.hA_produto_W_K:.0f} W/K): {pd.tempo_h:.1f} h; potência do compressor de {pd.W_inicial:.0f} W "
                f"(início) a {pd.W_final:.0f} W (fim), pico {pd.W_pico:.0f} W {quando} = "
                f"{pd.W_pico / bal.W_el:.2f}× a de regime. Calibrar hA com um ensaio de pull-down.")
        if not pd.concluido:
            av.critico("pull-down", f"com a porta fechada o produto não chega a {entradas.T_alvo_C:.1f} °C: equilíbrio "
                       f"em ≈ {pd.T_produto_equilibrio - 273.15:.1f} °C (capacidade seca menor que a carga do gabinete).")
        elif pd.tempo_h > u.tempo_pulldown_h:
            av.atencao("pull-down", f"tempo estimado {pd.tempo_h:.1f} h > requisito de {u.tempo_pulldown_h:g} h: a "
                       f"troca ar–produto (hA = {pd.hA_produto_W_K:.0f} W/K) e a capacidade disponível não bastam; a carga "
                       "de pull-down do dimensionamento supõe o produto à temperatura do ar.")
        if pd.T_descarga_max - 273.15 > p.ciclo.T_descarga_max_C:
            av.atencao("pull-down", f"temperatura de descarga chega a {pd.T_descarga_max - 273.15:.0f} °C no pull-down "
                       f"(> {p.ciclo.T_descarga_max_C:.0f} °C).")
    if W_max is None:
        return {} if pd is None else {"limite_W": None, "W_regime_W": bal.W_el, "W_pico_pulldown_W": pd.W_pico,
                                      "razao_pico_regime": pd.W_pico / bal.W_el}
    nome = "potência de regime" if rp.criterio == "regime" else "pico de potência no pull-down"
    W_c = bal.W_el if rp.criterio == "regime" else (None if pd is None else max(pd.W_pico, bal.W_el))
    atende = None if W_c is None else W_c <= W_max * (1.0 + 1e-6)
    f_on = car.Q_regime / car.Q_projeto
    if W_c is None:
        av.atencao("potência", f"limite de {W_max:.0f} W no pico de pull-down não verificado (simulação falhou).")
    elif not atende:
        cop_req = _cop_necessario(nuc, W_c, W_max, nuc.Q_requisitos)
        av.critico("potência", f"{nome} {W_c:.0f} W acima do limite de {W_max:.0f} W. Para atender todos os requisitos "
                   f"de carga com esse limite seria preciso COP ≈ {cop_req:.2f} (hoje {nuc.ciclo.COP:.2f})"
                   + (" ou aceitar um compressor menor (RestricaoPotencia.modo = 'limitar')." if rp.modo == "verificar"
                      else ".") + _sugestao_cop(fl, dataclasses.replace(entradas, COP=cop_req), T_ar_evap, T_ar_cond,
                                                p.ciclo))
    elif limitado:
        msg = (f"limite de {W_max:.0f} W ({nome}) ativo: capacidade de projeto reduzida de {nuc.Q_requisitos:.0f} para "
               f"{car.Q_projeto:.0f} W (compressor de {nuc.comp.deslocamento_cm3:.2f} cm³); em regime o compressor fica "
               f"ligado ≈ {f_on:.0%} do tempo (premissa: ≤ {u.fator_utilizacao_max:.0%})")
        if pd is not None:
            msg += f"; pull-down estimado em {pd.tempo_h:.1f} h (requisito {u.tempo_pulldown_h:g} h)"
        if f_on > 1.0 / (1.0 + u.margem_seguranca):
            av.critico("potência", msg + f" — sem a margem de segurança de {u.margem_seguranca:.0%}: reveja o COP ou o "
                       "limite de potência.")
        else:
            av.atencao("potência", msg + ".")
    else:
        av.info("potência", f"{nome} {W_c:.0f} W dentro do limite de {W_max:.0f} W.")
    if rp.criterio == "regime" and pd is not None and pd.W_pico > W_max * (1.0 + 1e-6):
        av.atencao("potência", f"o pico estimado no pull-down ({pd.W_pico:.0f} W"
                   f"{', na partida' if pd.pico_na_partida else ''}) passa do limite de {W_max:.0f} W, que foi aplicado "
                   "à potência de regime; se o requisito vale para o ensaio inteiro, use "
                   "RestricaoPotencia(criterio='pico_pulldown').")
    return {"limite_W": W_max, "criterio": rp.criterio, "modo": rp.modo, "W_criterio_W": W_c, "atende": atende,
            "limitado": limitado, "W_regime_W": bal.W_el, "W_pico_pulldown_W": None if pd is None else pd.W_pico,
            "razao_pico_regime": None if pd is None else pd.W_pico / bal.W_el,
            "Q_requisitos_W": nuc.Q_requisitos, "Q_projeto_W": car.Q_projeto, "fracao_ligado_regime": f_on}


def _dimensionar(entradas: EntradasProjeto, parametros: Optional[ParametrosProjeto]) -> ResultadoProjeto:
    p = copy.deepcopy(parametros) if parametros is not None else ParametrosProjeto()
    av = Avisos()
    fl = Fluido(entradas.fluido)
    if entradas.COP is not None and entradas.COP <= 0:
        raise ErroProjeto("COP deve ser positivo.")

    # 1) capacidade --------------------------------------------------------------------------
    cap = resolver_capacidade(entradas.volume_interno_L, entradas.n_garrafas, p.garrafa, p.gabinete, av)

    # 2) temperaturas de projeto -------------------------------------------------------------
    pc = p.ciclo
    T_ar_evap = C_para_K(entradas.T_alvo_C) + pc.dT_retorno_evap
    T_ar_cond = C_para_K(entradas.T_ambiente_C) + pc.dT_recirculacao_cond
    T_evap, T_cond = T_ar_evap - pc.dT_evap, T_ar_cond + pc.dT_cond
    if pc.superaquecimento >= pc.dT_evap:
        raise ErroProjeto("superaquecimento ≥ ΔT do evaporador: o ar não consegue superaquecer o vapor.")
    if pc.subresfriamento >= pc.dT_cond:
        raise ErroProjeto("sub-resfriamento ≥ ΔT do condensador: o ar não consegue sub-resfriar o líquido.")
    p.condensador.eficacia = entradas.eficacia_condensador
    p.evaporador.eficacia = entradas.eficacia_evaporador

    # 3) carga térmica <-> ciclo <-> trocadores <-> compressor, com a restrição de potência --------------
    W_max = entradas.potencia_max_compressor_W
    rp = p.restricao_potencia
    if W_max is not None:
        if not (W_max > 0.0 and math.isfinite(W_max)):
            raise ErroProjeto("potencia_max_compressor_W deve ser um número positivo.")
        if rp.criterio not in ("regime", "pico_pulldown"):
            raise ErroProjeto("RestricaoPotencia.criterio deve ser 'regime' ou 'pico_pulldown'.")
        if rp.modo not in ("limitar", "verificar"):
            raise ErroProjeto("RestricaoPotencia.modo deve ser 'limitar' ou 'verificar'.")
    limitar = W_max is not None and rp.modo == "limitar"
    no_pico = W_max is not None and rp.criterio == "pico_pulldown"
    temps = (T_evap, T_cond, T_ar_evap, T_ar_cond)
    nc0 = {"condensador": p.condensador.n_circuitos, "evaporador": p.evaporador.n_circuitos}
    dicas = (None, None)
    # com COP informado W = Q/COP é exato: no critério de regime o teto de capacidade sai direto
    Q_teto = W_max * entradas.COP if (limitar and not no_pico and entradas.COP is not None) else None
    W_alvo = None if W_max is None else W_max * (1.0 - (0.5 * rp.tolerancia if no_pico else 1e-6))
    hist: list = []
    convergiu_pot = True
    for it_w in range(1, (max(rp.max_iter, 1) if limitar else 1) + 1):
        nuc = _nucleo(fl, entradas, p, cap, temps, dicas, Q_teto)
        dicas = nuc.dicas
        pd, pd_erro = None, None
        if p.simular_pulldown or no_pico:
            try:
                pd = _pulldown(nuc, p, cap, entradas)
            except (ErroProjeto, ValueError, ArithmeticError) as err:   # análise secundária
                pd_erro = str(err)
        if not limitar:
            break
        W_c = _potencia_criterio(rp, nuc, pd, pd_erro)
        limitado = nuc.carga.Q_projeto < nuc.Q_requisitos * (1.0 - 1e-9)
        if limitado and nuc.carga.Q_projeto < nuc.carga.Q_regime:
            raise ErroProjeto(_mensagem_potencia_inviavel(fl, entradas, p, nuc, W_c, W_max, rp, T_ar_evap, T_ar_cond))
        if W_c <= W_max * (1.0 + 1e-9) and (not limitado or W_c >= W_max * (1.0 - rp.tolerancia)):
            break
        hist.append((nuc.carga.Q_projeto, W_c))
        Q_teto = min(_proximo_teto(hist, W_alvo), nuc.Q_requisitos)
        if Q_teto < nuc.carga.Q_regime:
            raise ErroProjeto(_mensagem_potencia_inviavel(fl, entradas, p, nuc, W_c, W_max, rp, T_ar_evap, T_ar_cond))
    else:
        convergiu_pot = False
    carga, ciclo, bal, cond, evap, comp = nuc.carga, nuc.ciclo, nuc.bal, nuc.cond, nuc.evap, nuc.comp
    it, conv, umid = nuc.it, nuc.conv, nuc.umid
    equilibrio = p.uso.umidade_interna == "equilibrio"
    if not nuc.umid_conv:
        av.atencao("umidade", "o balanço de umidade do gabinete não convergiu em 4 passos")
    if not convergiu_pot:
        av.atencao("potência", f"a busca da capacidade que respeita {W_max:.0f} W não convergiu em {rp.max_iter} passos.")
    av.itens.extend(nuc.av_carga.itens)
    for r in (cond, evap):
        r.avisos = [a for a in r.avisos if not a.startswith("n_circuitos aumentado")]
        if r.n_circuitos != nc0[r.tipo]:
            r.avisos.append(f"n_circuitos aumentado de {nc0[r.tipo]} para {r.n_circuitos}: com {nc0[r.tipo]} a perda de "
                            f"carga do refrigerante impedia o projeto do {r.tipo} (pinch ou ε inatingível)")
    if not equilibrio and abs(umid["UR_interna"] - p.uso.UR_interna) > 0.05:
        av.atencao("umidade", f"UR interna fixa de {p.uso.UR_interna:.0%} incompatível com a serpentina: o equilíbrio "
                   f"entre infiltração e remoção daria {umid['UR_interna']:.0%} (o ξ do evaporador e a umidade "
                   "exportada ao Amesim ficam incoerentes).")
    if not conv:
        av.atencao("iteração", "acoplamento ciclo-trocadores não convergiu dentro de max_iter")
    av.itens.extend(nuc.av_ciclo.itens)
    if entradas.COP is not None and ciclo.eta_global > pc.eta_global_faixa[1]:
        av.info("compressor", _sugestao_cop(fl, entradas, T_ar_evap, T_ar_cond, pc).strip())
    if 0.85 < ciclo.eta_is < 1.0:
        av.atencao("compressor", f"η_is implícita = {ciclo.eta_is:.2f} (> 0,85): reveja η_eletromecânica "
                   f"({pc.eta_eletromecanica:.2f}) ou o COP; a temperatura de descarga fica subestimada.")

    # 4) compressor (dimensionado em _nucleo) ---------------------------------------------------------
    e = ciclo.estados

    # 5) capilar (entrada após o IHX, saída na pressão de entrada do evaporador) ----------------
    cap_sel = dimensionar_capilar(fl, bal.m, e["3i"].P, e["3i"].h, e["4"].P, p.capilar)
    av.extend("capilar", cap_sel.avisos, "INFO")
    if pc.eficacia_ihx > 0:
        av.info("capilar", "com trocador interno o capilar real é não adiabático (soldado à sucção); o "
                "comprimento adiabático calculado é uma estimativa a ser ajustada em protótipo.")

    # 6) carga de refrigerante, exergia, energia -------------------------------------------------
    carga_ref = estimar_carga(fl, ciclo, cond, evap, cap_sel.escolhido, p.carga_fluido)
    if carga_ref["limites_kg"]:
        lim = carga_ref["limites_kg"]
        if carga_ref["total_kg"] > min(lim.values()):
            av.atencao("carga de fluido", f"carga estimada {carga_ref['total_kg'] * 1e3:.0f} g de {fl.nome} "
                       f"(classe {carga_ref['classe']}) acima de {min(lim.values()) * 1e3:.0f} g; verificar a norma "
                       "aplicável ao produto (IEC 60335-2-89 vigente e regulamentação local).")
    T0, Tf = C_para_K(entradas.T_ambiente_C), C_para_K(entradas.T_alvo_C)
    ex = exergia(ciclo, bal, T0, Tf)
    if p.potencia_ventilador_cond_W is None:
        p.potencia_ventilador_cond_W = max(cond.potencia_ventilador_est, 3.0)
        av.info("condensador", f"potência do ventilador estimada em {p.potencia_ventilador_cond_W:.1f} W "
                f"({cond.vazao_ar_m3h:.0f} m³/h, {cond.dP_ar:.0f} Pa, η = {p.condensador.eficiencia_ventilador:.2f}).")
        p_cond_informada = False
    else:
        p_cond_informada = True
    energia = _energia_diaria(carga, bal, p, cond, evap)

    # 7) verificações finais ------------------------------------------------------------------
    for r in (cond, evap):
        av.extend(r.tipo, r.avisos)
        if r.tipo == "condensador" and not p_cond_informada:
            continue
        P_vent = p.potencia_ventilador_cond_W if r.tipo == "condensador" else p.uso.potencia_ventilador_evap_W
        if r.potencia_ventilador_est > P_vent:
            av.atencao(r.tipo, f"potência estimada do ventilador ({r.potencia_ventilador_est:.1f} W para "
                       f"{r.vazao_ar_m3h:.0f} m³/h a {r.dP_ar:.0f} Pa) maior que a informada ({P_vent:.1f} W).")
    if evap.T_ar_out - 273.15 < -1.5:
        av.info("evaporador", f"ar insuflado a {evap.T_ar_out - 273.15:.1f} °C com o compressor ligado (comum em "
                "visi-coolers); verificar na distribuição de ar o risco de congelar bebidas junto à descarga "
                "(ponto de congelamento ≈ −1 a −2 °C).")
    for r in (cond, evap):
        if r.eficacia_alvo is not None and r.fileiras_escolhidas_automaticamente:
            av.info(r.tipo, f"ε = {r.eficacia_alvo:.2f} atendida com {r.n_fileiras} fileira(s) a "
                    f"{r.velocidade_face:.2f} m/s (escolha automática; ver alternativas_fileiras).")
    v_suc = bal.m / (e["1"].rho * evap.geometria.Di ** 2 * math.pi / 4.0 * evap.n_circuitos)
    if v_suc < 3.8:
        av.info("evaporador", f"velocidade do vapor na saída do evaporador {v_suc:.1f} m/s (< 3,8 m/s, regra "
                "prática ASHRAE para arraste de óleo em linhas horizontais); com HC + óleo miscível em "
                "sistemas pequenos isso costuma ser aceitável, mas avalie no protótipo.")

    pot = _avisos_potencia(av, entradas, p, nuc, pd, pd_erro, W_max, rp, fl, T_ar_evap, T_ar_cond)

    sens = (_sensibilidade(fl, bal, ciclo, carga, p, T_ar_cond, T_ar_evap, cond, evap, cap_sel.escolhido.D)
            if p.analise_sensibilidade else {})
    return ResultadoProjeto(entradas, p, cap, carga, ciclo, bal, comp, cond, evap, cap_sel, carga_ref, ex,
                            energia, sens, av, it, pd, pot)


def _sugestao_cop(fl, entradas, T_ar_evap, T_ar_cond, pc: ParametrosCiclo) -> str:
    if entradas.COP is None:
        return ""
    eta_ref = 0.55
    s = escala_dT_para_cop(fl, entradas.COP, T_ar_evap, T_ar_cond, pc, eta_ref)
    if s is None:
        return (f" Mesmo com trocadores muito grandes, um compressor com η_global = {eta_ref:.2f} não chega a "
                f"COP {entradas.COP:.2f} nestas temperaturas.")
    return (f" Para COP {entradas.COP:.2f} com um compressor típico (η_global = {eta_ref:.2f}) seriam "
            f"necessários ΔT_evap ≈ {s * pc.dT_evap:.1f} K e ΔT_cond ≈ {s * pc.dT_cond:.1f} K "
            f"(hoje {pc.dT_evap:.1f} K e {pc.dT_cond:.1f} K).")


def _energia_diaria(carga: CargaTermica, bal: BalancoSistema, p: ParametrosProjeto, cond, evap) -> dict:
    """Consumo diário estimado no regime de uso (compressor cicla na capacidade de projeto)."""
    fr = min(carga.Q_regime / bal.Q_evap, 1.0)
    u = p.uso
    kwh = {"compressor": bal.W_el * fr * 24e-3,
           "ventilador do condensador": p.potencia_ventilador_cond_W * fr * 24e-3,
           "ventilador do evaporador": u.potencia_ventilador_evap_W * 24e-3,
           "iluminação": u.potencia_iluminacao_W * 24e-3,
           "anti-embaçante": u.potencia_antiembacante_W * 24e-3}
    cop_sist = bal.Q_evap / (bal.W_el + p.potencia_ventilador_cond_W + u.potencia_ventilador_evap_W)
    return {"fracao_ligado": fr, "kWh_dia": kwh, "total_kWh_dia": sum(kwh.values()),
            "COP_sistema": cop_sist}


def _sensibilidade(fl, bal, ciclo, carga, p: ParametrosProjeto, T_ar_cond, T_ar_evap, cond, evap,
                   D_cap: float) -> dict:
    """Incerteza de modelo: mesmo problema com correlações alternativas."""
    e = ciclo.estados
    out = {}
    try:
        pe = copy.deepcopy(p.evaporador)
        pe.correlacao_bifasico = ("liu_winterton_1991" if pe.correlacao_bifasico == "gungor_winterton_1987"
                                  else "gungor_winterton_1987")
        pe.fileiras_automaticas = False
        pe.geometria.n_fileiras = evap.n_fileiras
        r = dimensionar_evaporador(fl, bal.m, e["3i"].h, e["1"].P, p.ciclo.superaquecimento, T_ar_evap, pe,
                                   carga.fator_umidade, (evap.n_fileiras, evap.velocidade_face))
        out["evaporador"] = {"correlacao_base": p.evaporador.correlacao_bifasico, "L_base": evap.L_requerido,
                             "correlacao_alt": pe.correlacao_bifasico, "L_alt": r.L_requerido,
                             "variacao": r.L_requerido / evap.L_requerido - 1.0,
                             "V_base": evap.velocidade_face, "V_alt": r.velocidade_face}
    except (ErroProjeto, ValueError, TypeError, ZeroDivisionError) as err:   # análise secundária
        out["evaporador"] = {"erro": str(err)}
    try:
        pq = copy.deepcopy(p.condensador)
        Re_GT = cond.G_ref * cond.geometria.Di / fl.sat(e["2"].P).muv
        pq.correlacao_bifasico = "chato_1962" if Re_GT < 35000 else "shah_1979"
        pq.fileiras_automaticas = False
        pq.geometria.n_fileiras = cond.n_fileiras
        r = dimensionar_condensador(fl, bal.m, e["2"].P, e["2"].h, p.ciclo.subresfriamento, T_ar_cond, pq,
                                    (cond.n_fileiras, cond.velocidade_face))
        out["condensador"] = {"correlacao_base": p.condensador.correlacao_bifasico, "L_base": cond.L_requerido,
                              "correlacao_alt": pq.correlacao_bifasico, "L_alt": r.L_requerido,
                              "variacao": r.L_requerido / cond.L_requerido - 1.0,
                              "V_base": cond.velocidade_face, "V_alt": r.velocidade_face}
    except (ErroProjeto, ValueError, TypeError, ZeroDivisionError) as err:   # análise secundária
        out["condensador"] = {"erro": str(err)}
    try:
        from .capilar import comprimento_capilar
        Ls = {}
        for mod in ("mcadams", "cicchitti", "dukler"):
            pcap = copy.deepcopy(p.capilar)
            pcap.modelo_viscosidade = mod
            Ls[mod] = comprimento_capilar(fl, bal.m, D_cap, e["3i"].P, e["3i"].h, e["4"].P, pcap).L
        out["capilar_viscosidade"] = Ls
    except (ErroProjeto, ValueError, TypeError, ZeroDivisionError) as err:   # análise secundária
        out["capilar_viscosidade"] = {"erro": str(err)}
    return out



def alternativas_fileiras(res: ResultadoProjeto, tipo: str, fileiras=range(1, 9)) -> list[dict]:
    """Tabela de alternativas do condensador ou do evaporador para a eficácia pedida: para cada nº de
    fileiras, a velocidade de face que atende ε, o comprimento de tubo, a vazão de ar e a ΔP do ar.
    Acima de 6 fileiras a correlação de Wang (2000) é extrapolada."""
    from .trocador import _condensador_fixo, _evaporador_fixo, alternativas_fileiras as _alt
    fl = Fluido(res.entradas.fluido)
    p, e, m = res.parametros, res.ciclo.estados, res.balanco.m
    pc = p.ciclo
    if tipo == "condensador":
        pt = copy.deepcopy(p.condensador)
        T_ar = C_para_K(res.entradas.T_ambiente_C) + pc.dT_recirculacao_cond
        rodar = (lambda pp: _condensador_fixo(fl, m, e["2"].P, e["2"].h, pc.subresfriamento, T_ar, pp))
    elif tipo == "evaporador":
        pt = copy.deepcopy(p.evaporador)
        T_ar = C_para_K(res.entradas.T_alvo_C) + pc.dT_retorno_evap
        rodar = (lambda pp: _evaporador_fixo(fl, m, e["3i"].h, e["1"].P, pc.superaquecimento, T_ar, pp,
                                             res.carga.fator_umidade))
    else:
        raise ValueError("tipo deve ser 'condensador' ou 'evaporador'")
    if pt.eficacia is None:
        raise ValueError("as alternativas por fileiras exigem a eficácia definida")
    return _alt(rodar, pt, fileiras)

In [ ]:
%%writefile predimref/relatorio.py
"""Saídas: resumo legível, dicionário/JSON para o acoplamento com o Simcenter Amesim e tabelas."""
from __future__ import annotations

import json
import math
from dataclasses import asdict, is_dataclass

from .constantes import W_PARA_BTUH
from .propriedades import ar_umido


def _c(T: float) -> float:
    return T - 273.15


def tabela_estados(res) -> list[dict]:
    nomes = {"1": "saída do evaporador", "1s": "sucção do compressor", "2s": "descarga isentrópica",
             "2": "descarga real", "3": "saída do condensador", "3i": "entrada do capilar",
             "4": "entrada do evaporador"}
    linhas = []
    for k, e in res.ciclo.estados.items():
        linhas.append({"ponto": k, "descricao": nomes[k], "P_bar": e.P / 1e5, "T_C": _c(e.T),
                       "h_kJ_kg": e.h / 1e3, "s_kJ_kgK": e.s / 1e3, "rho_kg_m3": e.rho,
                       "titulo": e.x if 0.0 <= e.x <= 1.0 else None})
    return linhas


def perfil_trocador(r) -> list[dict]:
    """Perfil ao longo do circuito (posição acumulada por circuito)."""
    out, L = [], 0.0
    for s in r.segmentos:
        L += s.L
        out.append({"zona": s.zona, "L_acum_m": L, "P_bar": s.P_out / 1e5, "T_ref_C": _c(s.T_out),
                    "x": s.x_out, "h_int": s.h_int, "UA_m": s.UA_m, "T_ar_saida_C": _c(s.T_ar_out),
                    "alpha": s.alpha, "regime": s.regime})
    return out


def _serp_dict(sp, r) -> dict:
    return {"descricao": sp.descricao, "comprimento_total_tubo_m": sp.comprimento_tubo,
            "comprimento_por_circuito_m": sp.comprimento_tubo / r.n_circuitos,
            "comprimento_passe_m": sp.comprimento_passe, "tubos_por_fileira": sp.tubos_por_fileira,
            "n_tubos": sp.n_tubos, "face_altura_m": sp.altura_face, "face_largura_m": sp.comprimento_passe,
            "face_area_m2": sp.area_face, "area_externa_m2": sp.area_externa, "area_interna_m2": sp.area_interna,
            "volume_interno_L": sp.volume_interno * 1e3, "n_aletas": sp.n_aletas,
            "vazao_ar_m3_h": sp.vazao_ar_m3h, "UA_W_K": sp.UA, "UA_seco_W_K": sp.UA_seco}


def _trocador_dict(r) -> dict:
    g = r.geometria
    return {
        "tipo": r.tipo,
        "capacidade_W": r.Q,
        "geometria": {
            "tubo_diametro_externo_mm": g.Do * 1e3, "tubo_diametro_interno_mm": g.Di * 1e3,
            "tubo_espessura_mm": g.esp_parede * 1e3, "aleta_espessura_mm": g.esp_aleta * 1e3,
            "passo_aleta_mm": g.passo_aleta * 1e3, "aletas_por_polegada": 25.4e-3 / g.passo_aleta,
            "passo_transversal_mm": g.Pt * 1e3, "passo_longitudinal_mm": g.Pl * 1e3,
            "n_fileiras": r.n_fileiras, "n_circuitos": r.n_circuitos,
            "arranjo": "escalonado", "tipo_aleta": "plana contínua", "material_tubo": "cobre",
            "material_aleta": "alumínio", "k_aleta_W_mK": g.k_aleta, "rugosidade_interna_um": g.rugosidade_int * 1e6,
        },
        "serpentina_projeto": _serp_dict(r.projeto, r),
        "serpentina_construtiva": _serp_dict(r.construtiva, r),
        "zonas": {z: {"comprimento_m": v["L"], "calor_W": v["Q"], "UA_W_K": v["UA"], "UA_seco_W_K": v["UA_seco"],
                      "eficacia_saturacao": v["eficacia"], "NTU": v["NTU"]} for z, v in r.zonas.items()},
        "ar": {"velocidade_face_m_s": r.velocidade_face, "vazao_requerida_m3_h": r.vazao_ar_requerida_m3h,
               "vazao_serpentina_construtiva_m3_h": r.vazao_ar_m3h, "T_entrada_C": _c(r.T_ar_in),
               "T_saida_ponto_projeto_C": _c(r.T_ar_out), "queda_pressao_Pa": r.dP_ar,
               "h_W_m2K": r.ar.h, "eficiencia_superficie": r.ar.eta_superficie, "Re_Dc": r.ar.Re_Dc,
               "j": r.ar.j, "f": r.ar.f, "fator_umidade": r.fator_umidade,
               "potencia_ventilador_aerodinamica_W": r.potencia_ventilador_aero,
               "nota": "T_saida e vazao_requerida valem para a serpentina de projeto; a construtiva, na mesma "
                       "velocidade de face, passa a vazão indicada e tem a mesma eficácia, com mais capacidade."},
        "refrigerante": {"fluxo_massico_kg_m2s": r.G_ref, "P_entrada_bar": r.P_in / 1e5,
                         "P_saida_bar": r.P_out / 1e5, "queda_pressao_kPa": r.dP_ref / 1e3,
                         "queda_T_saturacao_K": r.dT_sat_queda},
        "eficacia": {"alvo": r.eficacia_alvo, "obtida": r.eficacia, "estrita_Q_Qmax": r.eficacia_Qmax,
                     "NTU": r.NTU, "C_ar_sensivel_W_K": r.C_ar, "T_sat_referencia_C": _c(r.T_sat_ref),
                     "definicao": "ε = (T_ar,saída − T_ar,entrada)/(T_sat − T_ar,entrada), bulbo seco; "
                                  "NTU = UA/(ξ·Ċ_ar,sensível)",
                     "fileiras_automaticas": r.fileiras_escolhidas_automaticamente},
    }


def _pulldown_dict(pd) -> dict | None:
    """Resumo e curva do pull-down estimado (°C, h)."""
    if pd is None:
        return None
    pk = pd.pico
    cv = pd.curva()
    return {
        "premissas": {"porta": "fechada", "T_ambiente_C": _c(pd.T_ambiente), "T_produto_inicial_C": _c(pd.T_produto_inicial),
                      "T_alvo_C": _c(pd.T_alvo), "capacidade_termica_produto_kJ_K": pd.C_produto_J_K / 1e3,
                      "hA_ar_produto_W_K": pd.hA_produto_W_K, "UA_gabinete_W_K": pd.UA_gabinete_W_K,
                      "cargas_internas_W": pd.Q_internas_W, "fator_umidade": pd.fator_umidade,
                      "modelo": "casamento compressor–trocadores quase estático (geometria de projeto, ε, η_global e "
                                "superaquecimento constantes); produto concentrado trocando calor com o ar por hA"},
        "concluido": pd.concluido, "tempo_h": pd.tempo_h,
        "T_produto_equilibrio_C": None if pd.T_produto_equilibrio is None else _c(pd.T_produto_equilibrio),
        "potencia_pico_W": pd.W_pico, "pico_na_partida": pd.pico_na_partida, "t_pico_h": pd.t_pico_s / 3600.0,
        "potencia_pico_trajetoria_W": pd.W_pico_trajetoria, "potencia_inicial_W": pd.W_inicial,
        "potencia_final_W": pd.W_final, "energia_compressor_kWh": pd.E_compressor_kWh,
        "T_descarga_max_C": _c(pd.T_descarga_max),
        "ponto_pico": {"T_ar_evaporador_C": _c(pk.T_ar_evap), "T_evap_C": _c(pk.T_evap), "T_cond_C": _c(pk.T_cond),
                       "razao_pressao": pk.razao_pressao, "Q_evap_W": pk.Q_evap, "vazao_massica_kg_h": pk.m * 3600.0,
                       "COP": pk.COP},
        "curva": {"t_h": [t / 3600.0 for t in cv["t_s"]],
                  **{k.replace("_K", "_C"): [_c(T) for T in v] for k, v in cv.items() if k.endswith("_K")},
                  "Q_evap_W": cv["Q_evap_W"], "W_el_W": cv["W_el_W"]},
        "nota": "pico = máximo de W(T_ar) entre o ar no fim do pull-down e o produto inicial: inclui a partida com o "
                "gabinete encharcado (limite superior dos primeiros minutos) e não depende de hA; o tempo depende de hA, "
                "a calibrar com ensaio de pull-down.",
    }


def para_dict(res) -> dict:
    """Dicionário JSON-serializável. O bloco 'amesim' reúne os parâmetros de entrada típicos de
    um modelo de sistema (compressor volumétrico, trocadores tubo-aleta, capilar)."""
    c, b, comp = res.ciclo, res.balanco, res.compressor
    cap = res.capilar.escolhido
    car = res.carga
    ent = res.entradas
    d = {
        "entradas": {"COP": ent.COP, "T_ambiente_C": ent.T_ambiente_C, "T_alvo_C": ent.T_alvo_C,
                     "UR_ambiente": ent.UR_ambiente, "volume_interno_L": res.capacidade.volume_interno_L,
                     "n_garrafas": res.capacidade.n_garrafas, "fluido": ent.fluido,
                     "eficacia_condensador": ent.eficacia_condensador,
                     "eficacia_evaporador": ent.eficacia_evaporador,
                     "potencia_max_compressor_W": ent.potencia_max_compressor_W},
        "gabinete": {"largura_int_m": res.capacidade.largura_m, "profundidade_int_m": res.capacidade.profundidade_m,
                     "altura_int_m": res.capacidade.altura_m, "volume_ar_livre_L": res.capacidade.volume_ar_livre_L,
                     "massa_produto_kg": res.capacidade.massa_produto_kg,
                     "UA_gabinete_W_K": car.transmissao["UA_total"]},
        "carga_termica_W": {**car.componentes, "regime": car.Q_regime,
                            "requerida_regime": car.Q_requerida_regime, "pull_down": car.Q_pulldown,
                            "caso_governante": car.caso_governante, "projeto": car.Q_projeto,
                            "latente_projeto": car.Q_latente_projeto, "fator_umidade": car.fator_umidade,
                            "agua_removida_kg_dia": car.infiltracao.get("agua_kg_dia", 0.0),
                            "umidade_interna": car.infiltracao.get("umidade")},
        "ciclo": {"T_evap_C": _c(c.T_evap), "T_cond_C": _c(c.T_cond), "P_evap_bar": c.estados["1"].P / 1e5,
                  "P_cond_bar": c.estados["2"].P / 1e5, "razao_pressao": c.razao_pressao,
                  "COP": c.COP, "COP_ciclo_ideal": c.COP_ciclo_ideal, "COP_carnot_ciclo": c.COP_carnot_ciclo,
                  "eta_global": c.eta_global, "eta_isentropica": c.eta_is, "eta_eletromecanica": c.eta_em,
                  "estados": tabela_estados(res)},
        "balanco_W": {"vazao_massica_kg_h": b.m * 3600, "Q_evaporador": b.Q_evap, "W_eletrica_compressor": b.W_el,
                      "Q_condensador": b.Q_cond, "Q_carcaca": b.Q_carcaca, "Q_trocador_interno": b.Q_ihx,
                      "erro_relativo": b.erro_balanco},
        "compressor": {"potencia_eletrica_W": b.W_el, "capacidade_W": b.Q_evap,
                       "capacidade_Btuh": b.Q_evap * W_PARA_BTUH, "deslocamento_cm3": comp.deslocamento_cm3,
                       "rotacao_rpm": comp.rotacao_rpm, "eficiencia_volumetrica": comp.eta_v,
                       "vazao_volumetrica_succao_m3_h": comp.vazao_volumetrica_succao * 3600,
                       "T_descarga_C": _c(comp.T_descarga), "ponto_catalogo": comp.catalogo,
                       "potencia_pico_pulldown_W": None if res.pulldown is None else res.pulldown.W_pico},
        "potencia_compressor": res.potencia,
        "pulldown": _pulldown_dict(res.pulldown),
        "condensador": _trocador_dict(res.condensador),
        "evaporador": _trocador_dict(res.evaporador),
        "capilar": {"diametro_interno_mm": cap.D * 1e3, "diametro_interno_pol": cap.D / 25.4e-3,
                    "comprimento_m": cap.L, "comprimento_liquido_m": cap.L_liquido,
                    "rugosidade_um": res.parametros.capilar.rugosidade_m * 1e6, "blocado": cap.blocado,
                    "P_entrada_bar": res.ciclo.estados["3i"].P / 1e5, "P_saida_bar": cap.P_saida / 1e5,
                    "titulo_saida": cap.x_saida, "modelo": f"HEM, viscosidade {res.parametros.capilar.modelo_viscosidade}",
                    "sensibilidade_tolerancia_D": res.capilar.sensibilidade,
                    "alternativas": [{"D_mm": a.D * 1e3, "L_m": a.L, "blocado": a.blocado}
                                     for a in res.capilar.candidatos]},
        "carga_refrigerante": {"total_g": res.carga_refrigerante["total_kg"] * 1e3,
                               "total_serpentinas_projeto_g": res.carga_refrigerante["total_projeto_kg"] * 1e3,
                               "classe_seguranca": res.carga_refrigerante["classe"],
                               "partes_g": {k: v * 1e3 for k, v in res.carga_refrigerante["partes_kg"].items()},
                               "limites_g": {k: v * 1e3 for k, v in res.carga_refrigerante["limites_kg"].items()}},
        "exergia_W": {**res.exergia["destruicao_W"], "util": res.exergia["Ex_util_W"],
                      "eficiencia_2a_lei": res.exergia["eta_II"]},
        "energia": res.energia,
        "sensibilidade_modelos": res.sensibilidade,
        "avisos": [str(a) for a in res.avisos],
    }
    d["amesim"] = _bloco_amesim(res)
    return _limpar(d)


def _ar_entrada(res, tipo: str) -> dict:
    """Estado do ar na entrada do trocador (para submodelos de ar úmido)."""
    import CoolProp.CoolProp as CP
    ent, pc = res.entradas, res.parametros.ciclo
    if tipo == "condensador":
        W = ar_umido(ent.T_ambiente_C + 273.15, ent.UR_ambiente)["W"]
        T = ent.T_ambiente_C + 273.15 + pc.dT_recirculacao_cond
    else:
        T = ent.T_alvo_C + 273.15 + pc.dT_retorno_evap
        W = ar_umido(T, res.parametros.uso.UR_interna)["W"]
    UR = CP.HAPropsSI("R", "T", T, "P", 101325.0, "W", W)
    return {"T_K": T, "umidade_relativa": UR, "umidade_absoluta_kg_kg": W, "P_Pa": 101325.0}


def _bloco_amesim(res) -> dict:
    """Parâmetros em SI, agrupados por componente, prontos para mapear nos submodelos.
    O mapeamento para os nomes de parâmetro de cada submodelo do Amesim fica com a equipe de
    integração; aqui os nomes são descritivos e as unidades explícitas.

    Cada trocador traz duas serpentinas com a mesma seção (fileiras, tubos por fileira, aletas):
    ``projeto_sem_margem`` reproduz ``alvo_validacao`` com a vazão de ar requerida; a
    ``construtiva_com_margem`` é a que se fabrica (mais área e, na mesma velocidade de face, mais
    vazão): num modelo de sistema ela leva a T_cond menor e T_evap maior que o alvo — é a folga."""
    c, cap = res.ciclo, res.capilar.escolhido
    pcf = res.parametros.carga_fluido

    def serp(sp, Vdot_m3h):
        return {"comprimento_passe_m": sp.comprimento_passe, "comprimento_total_tubo_m": sp.comprimento_tubo,
                "area_face_m2": sp.area_face, "area_externa_m2": sp.area_externa, "area_interna_m2": sp.area_interna,
                "volume_interno_m3": sp.volume_interno, "vazao_ar_m3_s": Vdot_m3h / 3600.0,
                "UA_W_K": sp.UA, "UA_seco_W_K": sp.UA_seco}

    def hx(r):
        g = r.geometria
        return {"geometria": {"Do_m": g.Do, "Di_m": g.Di, "esp_parede_m": g.esp_parede, "esp_aleta_m": g.esp_aleta,
                              "passo_aleta_m": g.passo_aleta, "Pt_m": g.Pt, "Pl_m": g.Pl,
                              "n_fileiras": r.n_fileiras, "tubos_por_fileira": r.tubos_por_fileira,
                              "n_circuitos": r.n_circuitos, "arranjo": "escalonado", "tipo_aleta": "plana contínua",
                              "k_aleta_W_mK": g.k_aleta, "k_tubo_W_mK": g.k_tubo, "rugosidade_m": g.rugosidade_int},
                "projeto_sem_margem": serp(r.projeto, r.projeto.vazao_ar_m3h),
                "construtiva_com_margem": serp(r.construtiva, r.construtiva.vazao_ar_m3h),
                "margem_area": res.parametros.condensador.margem_area if r.tipo == "condensador"
                else res.parametros.evaporador.margem_area,
                "velocidade_face_m_s": r.velocidade_face, "ar_entrada": _ar_entrada(res, r.tipo),
                "eficacia_saturacao": r.eficacia, "eficacia_Q_Qmax": r.eficacia_Qmax, "NTU": r.NTU,
                "fator_umidade": r.fator_umidade,
                "notas": "eficacia_saturacao é de bulbo seco (sensível); UA_W_K inclui ξ (analogia de Lewis no "
                         "filme de ar e na aleta) e UA_seco_W_K usa ξ = 1 — use o seco se o submodelo calcular a "
                         "condensação de umidade por conta própria; NTU = UA/(ξ·Ċ_ar,sensível)."}

    partes = res.carga_refrigerante["partes_kg"]
    fora = partes.get("gás na carcaça", 0.0) + partes.get("dissolvido no óleo", 0.0)
    circuito = res.carga_refrigerante["total_kg"] - fora
    circuito_proj = res.carga_refrigerante["total_projeto_kg"] - fora
    umid = res.carga.infiltracao.get("umidade") or {}
    return {
        "fluido": c.fluido,
        "compressor": {"deslocamento_m3": res.compressor.deslocamento_cm3 * 1e-6,
                       "rotacao_rpm": res.compressor.rotacao_rpm, "eta_volumetrica": res.compressor.eta_v,
                       "eta_isentropica": c.eta_is, "eta_eletromecanica": c.eta_em,
                       "eta_global": c.eta_global, "potencia_eletrica_W": res.balanco.W_el,
                       "volume_livre_carcaca_m3": pcf.volume_livre_carcaca_L * 1e-3,
                       "massa_oleo_kg": pcf.massa_oleo_kg, "solubilidade_oleo_kg_kg": pcf.solubilidade_oleo},
        "condensador": hx(res.condensador),
        "evaporador": hx(res.evaporador),
        "capilar": {"diametro_interno_m": cap.D, "comprimento_m": cap.L,
                    "rugosidade_m": res.parametros.capilar.rugosidade_m},
        "linhas": {"liquido": {"comprimento_m": pcf.linha_liquido_L_m, "Di_m": pcf.linha_liquido_Di_m},
                   "filtro_secador_volume_m3": pcf.filtro_secador_cm3 * 1e-6,
                   "succao": {"comprimento_m": pcf.linha_succao_L_m, "Di_m": pcf.linha_succao_Di_m},
                   "descarga": {"comprimento_m": pcf.linha_descarga_L_m, "Di_m": pcf.linha_descarga_Di_m}},
        "condicoes_iniciais": {"P_alta_Pa": c.estados["2"].P, "P_baixa_Pa": c.estados["1"].P,
                               "carga_refrigerante_total_kg": res.carga_refrigerante["total_kg"],
                               "carga_circuito_kg": circuito,
                               "carga_refrigerante_total_projeto_kg": res.carga_refrigerante["total_projeto_kg"],
                               "carga_circuito_projeto_kg": circuito_proj,
                               "carga_gas_carcaca_kg": partes.get("gás na carcaça", 0.0),
                               "carga_dissolvida_oleo_kg": partes.get("dissolvido no óleo", 0.0),
                               "superaquecimento_K": res.parametros.ciclo.superaquecimento,
                               "subresfriamento_K": res.parametros.ciclo.subresfriamento,
                               "nota": "carga do circuito = trocadores + capilar + linhas + filtro; '_projeto' usa as "
                                       "serpentinas sem margem (para conferir alvo_validacao num modelo de carga fixa), "
                                       "a outra as construtivas; somar carcaça e óleo se o modelo não os representar."},
        "ar_interno": {"T_K": res.entradas.T_alvo_C + 273.15, "umidade_relativa": res.parametros.uso.UR_interna,
                       "umidade_absoluta_kg_kg": umid.get("W_interno"),
                       "modo": res.parametros.uso.umidade_interna,
                       "T_superficie_efetiva_evaporador_K": umid.get("T_superficie_efetiva_K"),
                       "nota": "no modo 'equilibrio' a umidade interna fecha o balanço água da infiltração = água "
                               "removida pela serpentina (superfície efetiva, Lewis = 1); ξ do evaporador e "
                               "ar_entrada são coerentes com ela. No caso governante de pull-down (portas fechadas) "
                               "o evaporador é dimensionado seco (ξ = 1)."},
        "restricoes": {"potencia_max_compressor_W": res.entradas.potencia_max_compressor_W,
                       "criterio": (res.potencia or {}).get("criterio"),
                       "atendida": (res.potencia or {}).get("atende"),
                       "nota": "criterio 'regime' = potência no ponto de projeto; 'pico_pulldown' = maior potência no "
                               "pull-down de portas fechadas, incluindo a partida."},
        "pulldown_estimado": None if res.pulldown is None else {
            "T_ambiente_K": res.pulldown.T_ambiente, "T_produto_inicial_K": res.pulldown.T_produto_inicial,
            "T_alvo_K": res.pulldown.T_alvo, "capacidade_termica_produto_J_K": res.pulldown.C_produto_J_K,
            "hA_ar_produto_W_K": res.pulldown.hA_produto_W_K, "cargas_internas_W": res.pulldown.Q_internas_W,
            "tempo_ate_T_alvo_s": res.pulldown.tempo_s if res.pulldown.concluido else None,
            "potencia_pico_W": res.pulldown.W_pico, "potencia_inicial_W": res.pulldown.W_inicial,
            "potencia_final_W": res.pulldown.W_final, "energia_compressor_J": res.pulldown.E_compressor_J,
            "nota": "estimativa quase estática para comparar com o ensaio de pull-down no Amesim (não é alvo exato): "
                    "massa do gabinete desprezada, superaquecimento fixo e hA ar–produto premissa."},
        "alvo_validacao": {"Q_evap_W": res.balanco.Q_evap, "W_el_W": res.balanco.W_el,
                           "vazao_massica_kg_s": res.balanco.m, "T_evap_K": c.T_evap, "T_cond_K": c.T_cond,
                           "nota": "reproduzido com as serpentinas 'projeto_sem_margem', a vazão de ar requerida e "
                                   "a carga 'carga_*_projeto_kg'; com as 'construtiva_com_margem', T_cond fica menor "
                                   "e T_evap maior (folga)."},
    }


def _limpar(o):
    if is_dataclass(o):
        o = asdict(o)
    if isinstance(o, dict):
        return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_limpar(v) for v in o]
    if isinstance(o, float):
        return None if not math.isfinite(o) else o
    if hasattr(o, "item"):
        return o.item()
    return o


def exportar_json(res, caminho: str) -> str:
    with open(caminho, "w", encoding="utf-8") as fh:
        json.dump(para_dict(res), fh, ensure_ascii=False, indent=2)
    return caminho


def resumo_texto(res) -> str:
    c, b, comp, car = res.ciclo, res.balanco, res.compressor, res.carga
    cap = res.capilar.escolhido
    ent = res.entradas
    L = []
    w = L.append
    w("=" * 78)
    w("PRÉ-DIMENSIONAMENTO DO CICLO DE REFRIGERAÇÃO")
    w("=" * 78)
    cop_txt = "não informado" if ent.COP is None else f"{ent.COP:.2f}"
    def _e(v):
        return "livre" if v is None else f"{v:.2f}"
    w(f"Entradas: COP {cop_txt} | T_amb {ent.T_ambiente_C:.1f} °C, UR {ent.UR_ambiente:.0%} | "
      f"T_alvo {ent.T_alvo_C:.1f} °C | {ent.fluido} | ε cond {_e(ent.eficacia_condensador)}, "
      f"ε evap {_e(ent.eficacia_evaporador)}")
    k = res.capacidade
    w(f"Gabinete: {k.volume_interno_L:.0f} L internos ({k.largura_m * 1e3:.0f} × {k.profundidade_m * 1e3:.0f} × "
      f"{k.altura_m * 1e3:.0f} mm), {k.n_garrafas} garrafas, {k.massa_produto_kg:.1f} kg de produto")
    w("")
    w("CARGA TÉRMICA [W]")
    for nome, v in car.componentes.items():
        w(f"  {nome:34s} {v:8.1f}")
    w(f"  regime / fator de utilização       {car.Q_requerida_regime:8.1f}")
    w(f"  pull-down (100 % ligado)           {car.Q_pulldown:8.1f}")
    w(f"  >> projeto ({car.caso_governante}, +{res.parametros.uso.margem_seguranca:.0%}): {car.Q_projeto:.1f} W "
      f"({car.Q_projeto * W_PARA_BTUH:.0f} Btu/h), ξ = {car.fator_umidade:.3f}")
    um = car.infiltracao.get("umidade")
    if um:
        w(f"  umidade interna ({res.parametros.uso.umidade_interna}): UR {res.parametros.uso.UR_interna:.0%} a "
          f"{ent.T_alvo_C:.1f} °C (W = {um['W_interno'] * 1e3:.2f} g/kg); superfície efetiva do evaporador "
          f"{_c(um['T_superficie_efetiva_K']):.1f} °C; água removida {car.infiltracao.get('agua_kg_dia', 0.0) * 1e3:.0f} g/dia")
    w("")
    w("CICLO")
    w(f"  T_evap {_c(c.T_evap):6.1f} °C ({c.estados['1'].P / 1e5:.3f} bar) | T_cond {_c(c.T_cond):5.1f} °C "
      f"({c.estados['2'].P / 1e5:.2f} bar) | razão de pressão {c.razao_pressao:.2f}")
    w(f"  COP {c.COP:.3f} | ciclo ideal {c.COP_ciclo_ideal:.3f} | Carnot {c.COP_carnot_ciclo:.3f} | "
      f"η_global {c.eta_global:.3f}, η_is {c.eta_is:.3f}, η_em {c.eta_em:.3f}")
    w(f"  {'pt':3s} {'P [bar]':>8s} {'T [°C]':>8s} {'h [kJ/kg]':>10s} {'s [kJ/kgK]':>11s} {'x':>6s}")
    for row in tabela_estados(res):
        x = "" if row["titulo"] is None else f"{row['titulo']:.3f}"
        w(f"  {row['ponto']:3s} {row['P_bar']:8.3f} {row['T_C']:8.2f} {row['h_kJ_kg']:10.2f} "
          f"{row['s_kJ_kgK']:11.4f} {x:>6s}")
    w(f"  Balanço: ṁ = {b.m * 3600:.3f} kg/h | Q_evap {b.Q_evap:.1f} + W_el {b.W_el:.1f} = Q_cond {b.Q_cond:.1f} "
      f"+ Q_carcaça {b.Q_carcaca:.1f} W (erro {b.erro_balanco:.1e})")
    w("")
    w("COMPRESSOR")
    w(f"  potência elétrica {b.W_el:.1f} W | deslocamento {comp.deslocamento_cm3:.2f} cm³ @ {comp.rotacao_rpm:.0f} rpm "
      f"| η_v {comp.eta_v:.3f} | T_descarga {_c(comp.T_descarga):.1f} °C")
    cat = comp.catalogo
    w(f"  no ponto {cat['condicao']} ({cat['T_evap_C']:.1f}/{cat['T_cond_C']:.1f} °C): ≈ {cat['Q_W']:.0f} W "
      f"({cat['Q_Btuh']:.0f} Btu/h), {cat['W_el_W']:.0f} W elétricos, COP {cat['COP']:.2f}")
    pot = res.potencia or {}
    if pot.get("limite_W"):
        estado = ("ATIVO: capacidade de projeto " f"{pot['Q_requisitos_W']:.0f} → {pot['Q_projeto_W']:.0f} W"
                  if pot["limitado"] else "inativo")
        atende = {True: "atende", False: "NÃO ATENDE", None: "não verificado"}[pot["atende"]]
        pico = "" if pot.get("W_pico_pulldown_W") is None else f", pico no pull-down {pot['W_pico_pulldown_W']:.0f} W"
        w(f"  limite de potência {pot['limite_W']:.0f} W ({pot['criterio']}, {pot['modo']}): {atende}, {estado} | "
          f"regime {pot['W_regime_W']:.1f} W{pico}")
    pd = res.pulldown
    if pd is not None:
        w("")
        w(f"PULL-DOWN ESTIMADO (porta fechada; produto {_c(pd.T_produto_inicial):.1f} → {_c(pd.T_alvo):.1f} °C; ambiente "
          f"{_c(pd.T_ambiente):.1f} °C; C {pd.C_produto_J_K / 1e3:.0f} kJ/K; hA ar–produto {pd.hA_produto_W_K:.0f} W/K)")
        fim = (f"{pd.tempo_h:.1f} h até {_c(pd.T_alvo):.1f} °C" if pd.concluido else
               f"NÃO atinge {_c(pd.T_alvo):.1f} °C (equilíbrio ≈ {_c(pd.T_produto_equilibrio):.1f} °C)")
        quando = "na partida" if pd.pico_na_partida else f"em t ≈ {pd.t_pico_s / 3600.0:.1f} h"
        w(f"  {fim} | compressor {pd.W_inicial:.0f} W (início) → {pd.W_final:.0f} W (fim); pico {pd.W_pico:.0f} W {quando} "
          f"({pd.W_pico / res.balanco.W_el:.2f}× o de regime; T_evap {_c(pd.pico.T_evap):.1f} °C, T_cond "
          f"{_c(pd.pico.T_cond):.1f} °C) | {pd.E_compressor_kWh:.2f} kWh | T_descarga máx {_c(pd.T_descarga_max):.0f} °C")
    for r in (res.condensador, res.evaporador):
        g = r.geometria
        w("")
        w(r.tipo.upper())
        w(f"  {r.Q:.1f} W | tubo Ø{g.Do * 1e3:.2f} × {g.esp_parede * 1e3:.2f} mm (Di {g.Di * 1e3:.2f} mm), "
          f"aleta {g.esp_aleta * 1e3:.2f} mm @ {g.passo_aleta * 1e3:.2f} mm ({25.4e-3 / g.passo_aleta:.1f} FPI)")
        zonas = ", ".join(f"{z} {v['L']:.2f} m" for z, v in r.zonas.items())
        pj, ct = r.projeto, r.construtiva
        w(f"  serpentina de projeto (sem margem): {pj.comprimento_tubo:.2f} m de tubo ({zonas})")
        w(f"  serpentina construtiva (+{(ct.comprimento_tubo / pj.comprimento_tubo - 1):.0%}): {r.n_fileiras} fileiras × "
          f"{ct.tubos_por_fileira} tubos × {ct.comprimento_passe:.3f} m = {ct.comprimento_tubo:.2f} m, "
          f"{r.n_circuitos} circuito(s)")
        w(f"  face {ct.comprimento_passe * 1e3:.0f} × {ct.altura_face * 1e3:.0f} mm | A_ext {ct.area_externa:.3f} m² | "
          f"A_int {ct.area_interna:.4f} m² | V_int {ct.volume_interno * 1e6:.0f} cm³")
        alvo = "" if r.eficacia_alvo is None else f" (alvo {r.eficacia_alvo:.2f})"
        auto = ", fileiras escolhidas automaticamente" if r.fileiras_escolhidas_automaticamente else ""
        seco = f", seco {pj.UA_seco:.1f}" if r.fator_umidade > 1.0 else ""
        w(f"  eficácia ε = {r.eficacia:.3f}{alvo}, ε estrita Q/Q_max = {r.eficacia_Qmax:.3f}, NTU = {r.NTU:.3f}, "
          f"UA projeto {pj.UA:.1f} W/K{seco} (construtiva {ct.UA:.1f}){auto}")
        w(f"  ar no ponto de projeto: {pj.vazao_ar_m3h:.0f} m³/h, {_c(r.T_ar_in):.1f} → {_c(r.T_ar_out):.1f} °C | "
          f"face a {r.velocidade_face:.2f} m/s (construtiva passa {ct.vazao_ar_m3h:.0f} m³/h), ΔP {r.dP_ar:.1f} Pa, "
          f"h_ar {r.ar.h:.1f} W/m²K, η_sup {r.ar.eta_superficie:.3f}")
        w(f"  refrigerante: G {r.G_ref:.1f} kg/m²s, ΔP {r.dP_ref / 1e3:.2f} kPa ({r.dT_sat_queda:.2f} K de saturação)")
    w("")
    w("CAPILAR")
    w(f"  D_int {cap.D * 1e3:.3f} mm ({cap.D / 25.4e-3:.3f}\") × L {cap.L:.2f} m (líquido {cap.L_liquido:.2f} m) | "
      f"{'blocado' if cap.blocado else 'não blocado'}, P_saída {cap.P_saida / 1e5:.3f} bar, x_saída {cap.x_saida:.3f}")
    s = res.capilar.sensibilidade
    w(f"  tolerância ±{res.parametros.capilar.tolerancia_diametro_mm * 1e3:.1f} µm no diâmetro: vazão "
      f"{s['vazao_rel_D_menos_tol']:+.1%} / {s['vazao_rel_D_mais_tol']:+.1%}")
    alts = ", ".join(f"{a.D * 1e3:.3f} mm→{a.L:.2f} m" for a in res.capilar.candidatos if 0.3 <= a.L <= 8)
    w(f"  alternativas: {alts}")
    w("")
    cr = res.carga_refrigerante
    w(f"CARGA DE REFRIGERANTE ≈ {cr['total_kg'] * 1e3:.0f} g ({cr['classe']}; com as serpentinas de projeto "
      f"{cr['total_projeto_kg'] * 1e3:.0f} g)  (" +
      ", ".join(f"{k} {v * 1e3:.0f} g" for k, v in cr["partes_kg"].items()) + ")")
    ex = res.exergia
    w(f"EXERGIA: η_II = {ex['eta_II']:.3f}; destruição [W]: " +
      ", ".join(f"{k} {v:.1f}" for k, v in ex["destruicao_W"].items()))
    en = res.energia
    w(f"ENERGIA: compressor ligado {en['fracao_ligado']:.0%} do tempo em regime; ≈ {en['total_kWh_dia']:.2f} kWh/dia; "
      f"COP do sistema (com ventiladores) {en['COP_sistema']:.2f}")
    if res.sensibilidade:
        w("")
        w("INCERTEZA DE MODELO (correlações alternativas)")
        for nome in ("evaporador", "condensador"):
            if nome in res.sensibilidade and "erro" in res.sensibilidade[nome]:
                w(f"  {nome}: correlação alternativa sem solução ({res.sensibilidade[nome]['erro'][:120]})")
            elif nome in res.sensibilidade:
                d = res.sensibilidade[nome]
                w(f"  {nome}: {d['correlacao_base']} {d['L_base']:.2f} m @ {d['V_base']:.2f} m/s vs "
                  f"{d['correlacao_alt']} {d['L_alt']:.2f} m @ {d['V_alt']:.2f} m/s ({d['variacao']:+.1%} no comprimento)")
        if "capilar_viscosidade" in res.sensibilidade and "erro" not in res.sensibilidade["capilar_viscosidade"]:
            w("  capilar (viscosidade): " + ", ".join(f"{k} {v:.2f} m" for k, v in res.sensibilidade["capilar_viscosidade"].items()))
    if len(res.avisos):
        w("")
        w("AVISOS")
        for a in sorted(res.avisos, key=lambda a: {"CRITICO": 0, "ATENCAO": 1, "INFO": 2}[a.nivel]):
            w(f"  {a}")
    return "\n".join(L)

In [ ]:
%%writefile predimref/graficos.py
"""Gráficos do pré-dimensionamento (matplotlib). Importação preguiçosa: o núcleo de cálculo não
depende de matplotlib, o que facilita rodar a rotina dentro do ambiente Python do Amesim."""
from __future__ import annotations

import numpy as np

# paleta categórica de referência (ordem fixa) e tokens de texto/superfície
AZUL, LARANJA, VERDE_AGUA, AMARELO = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
TEXTO, TEXTO_2, GRADE, SUPERFICIE, DOMO = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb", "#a3a29c"


def _estilo():
    import matplotlib.pyplot as plt
    plt.rcParams.update({
        "figure.facecolor": SUPERFICIE, "axes.facecolor": SUPERFICIE, "axes.edgecolor": GRADE,
        "axes.labelcolor": TEXTO_2, "axes.titlecolor": TEXTO, "axes.titlesize": 11, "axes.labelsize": 9,
        "xtick.color": TEXTO_2, "ytick.color": TEXTO_2, "xtick.labelsize": 8, "ytick.labelsize": 8,
        "axes.grid": True, "grid.color": GRADE, "grid.linewidth": 0.6, "grid.linestyle": "-",
        "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2.0,
        "legend.frameon": False, "legend.fontsize": 8, "font.size": 9,
    })
    return plt


def _domo(fl, n=120, Tmin=None):
    Tmin = max(fl.Ttriplo + 5, 200.0) if Tmin is None else Tmin
    Ts = np.linspace(Tmin, fl.Tc - 0.5, n)
    Ps = [fl.Psat(T) for T in Ts]
    sats = [fl.sat(P) for P in Ps]
    return Ts, np.array(Ps), sats


def _caminho_ciclo(res, fl):
    """Pontos do ciclo com as trajetórias intermediárias (condensação, evaporação) para P-h/T-s."""
    e = res.ciclo.estados
    pts = [e["1s"], e["2"]]
    for s in res.condensador.segmentos:
        pts.append(fl.estado_PH(s.P_out, s.h_out))
    pts += [e["3"], e["3i"], e["4"]]
    for s in res.evaporador.segmentos:
        pts.append(fl.estado_PH(s.P_out, s.h_out))
    pts += [e["1"], e["1s"]]
    return pts


def diagrama_ph(res, ax=None):
    from .propriedades import Fluido
    plt = _estilo()
    fl = Fluido(res.ciclo.fluido)
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4.5))
    _, Ps, sats = _domo(fl, Tmin=res.ciclo.T_evap - 30.0)
    ax.plot([s.hl / 1e3 for s in sats], Ps / 1e5, color=DOMO, lw=1.2)
    ax.plot([s.hv / 1e3 for s in sats], Ps / 1e5, color=DOMO, lw=1.2, label="saturação")
    pts = _caminho_ciclo(res, fl)
    ax.plot([p.h / 1e3 for p in pts], [p.P / 1e5 for p in pts], color=AZUL, label=f"ciclo {fl.nome}")
    e = res.ciclo.estados
    for k in ("1", "2", "3", "4"):
        ax.plot(e[k].h / 1e3, e[k].P / 1e5, "o", ms=8, color=AZUL, mec=SUPERFICIE, mew=2)
        ax.annotate(k, (e[k].h / 1e3, e[k].P / 1e5), textcoords="offset points", xytext=(6, 6),
                    color=TEXTO, fontsize=9)
    ax.set_yscale("log")
    ax.set_xlabel("entalpia h [kJ/kg]")
    ax.set_ylabel("pressão P [bar]")
    ax.set_title("Diagrama P-h")
    ax.legend(loc="lower right")
    return ax


def diagrama_ts(res, ax=None):
    from .propriedades import Fluido
    plt = _estilo()
    fl = Fluido(res.ciclo.fluido)
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4.5))
    Ts, _, sats = _domo(fl, Tmin=res.ciclo.T_evap - 30.0)
    ax.plot([s.sl / 1e3 for s in sats], Ts - 273.15, color=DOMO, lw=1.2)
    ax.plot([s.sv / 1e3 for s in sats], Ts - 273.15, color=DOMO, lw=1.2, label="saturação")
    pts = _caminho_ciclo(res, fl)
    ax.plot([p.s / 1e3 for p in pts], [p.T - 273.15 for p in pts], color=AZUL, label=f"ciclo {fl.nome}")
    e = res.ciclo.estados
    for k in ("1", "2", "3", "4"):
        ax.plot(e[k].s / 1e3, e[k].T - 273.15, "o", ms=8, color=AZUL, mec=SUPERFICIE, mew=2)
        ax.annotate(k, (e[k].s / 1e3, e[k].T - 273.15), textcoords="offset points", xytext=(6, 6), color=TEXTO)
    ax.set_xlabel("entropia s [kJ/kg·K]")
    ax.set_ylabel("temperatura [°C]")
    ax.set_title("Diagrama T-s")
    ax.legend(loc="upper left")
    return ax


def perfil_trocador(r):
    """Dois painéis: temperaturas (refrigerante e ar de saída do trecho) e coeficiente interno."""
    plt = _estilo()
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(7, 5.5), sharex=True)
    L = np.cumsum([s.L for s in r.segmentos])
    a1.plot(L, [s.T_out - 273.15 for s in r.segmentos], color=AZUL, label="refrigerante")
    a1.plot(L, [s.T_ar_out - 273.15 for s in r.segmentos], color=LARANJA, label="ar na saída do trecho")
    a1.axhline(r.T_ar_in - 273.15, color=TEXTO_2, lw=0.8)
    a1.annotate("ar na entrada", (L[-1], r.T_ar_in - 273.15), textcoords="offset points", xytext=(-4, 4),
                ha="right", color=TEXTO_2, fontsize=8)
    a1.set_ylabel("temperatura [°C]")
    a1.set_title(f"{r.tipo.capitalize()}: perfil ao longo do circuito")
    a1.legend(loc="best")
    a2.plot(L, [s.h_int for s in r.segmentos], color=AZUL)
    a2.set_ylabel("h interno [W/m²K]")
    a2.set_xlabel("comprimento de tubo por circuito [m]")
    z0 = 0.0
    for z, v in r.zonas.items():
        z1 = z0 + v["L"] / r.n_circuitos
        a2.axvline(z1, color=GRADE, lw=1.0)
        a2.annotate(z, ((z0 + z1) / 2, a2.get_ylim()[1]), ha="center", va="top", color=TEXTO_2, fontsize=8)
        z0 = z1
    fig.tight_layout()
    return fig


def perfil_capilar(res):
    plt = _estilo()
    c = res.capilar.escolhido
    pf = c.perfil
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
    a1.plot(pf["L"], np.array(pf["P"]) / 1e5, color=AZUL)
    a1.axhline(res.ciclo.estados["4"].P / 1e5, color=TEXTO_2, lw=0.8)
    a1.annotate("P entrada do evaporador", (pf["L"][-1], res.ciclo.estados["4"].P / 1e5),
                textcoords="offset points", xytext=(-4, 4), ha="right", color=TEXTO_2, fontsize=8)
    a1.set_ylabel("pressão [bar]")
    a1.set_title(f"Capilar Ø{c.D * 1e3:.3f} mm × {c.L:.2f} m ({'blocado' if c.blocado else 'não blocado'})")
    a2.plot(pf["L"], pf["x"], color=AZUL)
    a2.set_ylabel("título x")
    a2.set_xlabel("posição [m]")
    fig.tight_layout()
    return fig


def barras(dados: dict, titulo: str, unidade: str):
    """Barras horizontais de uma única série (cor única), ordenadas, com rótulo de valor."""
    plt = _estilo()
    itens = sorted(dados.items(), key=lambda kv: kv[1])
    fig, ax = plt.subplots(figsize=(7, 0.45 * len(itens) + 1.2))
    y = np.arange(len(itens))
    vals = [v for _, v in itens]
    ax.barh(y, vals, height=0.6, color=AZUL)
    ax.set_yticks(y, [k for k, _ in itens])
    for yi, v in zip(y, vals):
        ax.annotate(f"{v:.1f}", (v, yi), textcoords="offset points", xytext=(4, 0), va="center",
                    color=TEXTO, fontsize=8)
    ax.set_xlabel(unidade)
    ax.set_title(titulo)
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    return fig


def mapa_cop(fluido: str, T_alvo_C: float, T_amb_C: np.ndarray, p_ciclo=None, etas=(0.70, 0.55),
             cop_pedido=None):
    """COP alcançável em função da temperatura ambiente, com as diferenças de temperatura de
    projeto dos trocadores: ciclo ideal (compressão isentrópica) e compressores com η_global dado."""
    from .ciclo import ParametrosCiclo, resolver_ciclo
    from .propriedades import Fluido
    plt = _estilo()
    fl = Fluido(fluido)
    pc = p_ciclo or ParametrosCiclo()
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ideal, por_eta = [], {e: [] for e in etas}
    for Ta in T_amb_C:
        Te = T_alvo_C + 273.15 + pc.dT_retorno_evap - pc.dT_evap
        Tc = Ta + 273.15 + pc.dT_recirculacao_cond + pc.dT_cond
        c = resolver_ciclo(fl, Te, Tc, pc, None)
        ideal.append(c.COP_ciclo_ideal)
        for e in etas:
            por_eta[e].append(e * c.COP_ciclo_ideal)
    ax.plot(T_amb_C, ideal, color=AZUL, label="ciclo ideal (compressão isentrópica)")
    for cor, e in zip((LARANJA, VERDE_AGUA), etas):
        ax.plot(T_amb_C, por_eta[e], color=cor, label=f"compressor com η_global = {e:.2f}")
    if cop_pedido is not None:
        ax.axhline(cop_pedido, color=TEXTO_2, lw=0.8)
        ax.annotate(f"COP pedido = {cop_pedido:.2f}", (T_amb_C[0], cop_pedido), textcoords="offset points",
                    xytext=(2, 4), color=TEXTO_2, fontsize=8)
    ax.set_xlabel("temperatura ambiente [°C]")
    ax.set_ylabel("COP [W/W]")
    ax.set_title(f"COP alcançável — {fluido}, T_alvo {T_alvo_C:.1f} °C, ΔT_evap {pc.dT_evap:g} K, "
                 f"ΔT_cond {pc.dT_cond:g} K", fontsize=10)
    ax.legend(loc="upper right")
    fig.tight_layout()
    return fig


def pulldown(res):
    """Pull-down estimado (porta fechada): temperaturas e potência do compressor ao longo do tempo."""
    pd = res.pulldown
    if pd is None:
        return None
    plt = _estilo()
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.3))
    th = pd.t / 3600.0
    T_alvo = pd.T_alvo - 273.15
    curvas = ((pd.T_produto, AZUL, "produto"), (pd.T_ar, LARANJA, "ar interno"),
              (pd.T_evap, VERDE_AGUA, "evaporação"))
    for y, cor, nome in curvas:
        a1.plot(th, y - 273.15, color=cor, label=nome)
        a1.annotate(nome, (th[-1], y[-1] - 273.15), textcoords="offset points", xytext=(4, 0), va="center",
                    color=TEXTO_2, fontsize=8)
    a1.axhline(T_alvo, color=TEXTO_2, lw=0.8, ls="--")
    a1.annotate(f"alvo {T_alvo:.1f} °C", (0.55 * th[-1], T_alvo), textcoords="offset points", xytext=(0, 4),
                ha="center", color=TEXTO_2, fontsize=8)
    a1.set_xlabel("tempo [h]")
    a1.set_ylabel("temperatura [°C]")
    fim = f"{pd.tempo_h:.1f} h até {T_alvo:.1f} °C" if pd.concluido else "não atinge o alvo"
    a1.set_title(f"Pull-down estimado (porta fechada): {fim}")
    a1.legend(loc="upper right")
    a1.set_xlim(0.0, th[-1] * 1.12)

    a2.plot(th, pd.Q_evap, color=AZUL, label="capacidade frigorífica")
    a2.plot(th, pd.W_el, color=LARANJA, label="potência elétrica do compressor")
    for y, nome in ((pd.Q_evap, "capacidade"), (pd.W_el, "potência")):
        a2.annotate(nome, (th[-1], y[-1]), textcoords="offset points", xytext=(4, 0), va="center", color=TEXTO_2,
                    fontsize=8)
    a2.plot(pd.t_pico_s / 3600.0, pd.W_pico, "o", ms=8, color=LARANJA, mec=SUPERFICIE, mew=2, clip_on=False,
            zorder=5)
    a2.annotate(f"pico {pd.W_pico:.0f} W" + (" (partida)" if pd.pico_na_partida else ""),
                (pd.t_pico_s / 3600.0, pd.W_pico), textcoords="offset points", xytext=(8, 6), color=TEXTO, fontsize=8)
    lim = (res.potencia or {}).get("limite_W")
    if lim:
        a2.axhline(lim, color=TEXTO_2, lw=1.0, ls="--")
        a2.annotate(f"limite {lim:.0f} W", (th[-1], lim), textcoords="offset points", xytext=(0, 4), ha="right",
                    color=TEXTO_2, fontsize=8)
    a2.set_xlabel("tempo [h]")
    a2.set_ylabel("potência [W]")
    a2.set_ylim(0.0, max(float(np.max(pd.Q_evap)), lim or 0.0) * 1.12)
    a2.set_xlim(0.0, th[-1] * 1.15)
    a2.set_title(f"Compressor no pull-down: regime de projeto {res.balanco.W_el:.0f} W, pico {pd.W_pico:.0f} W")
    a2.legend(loc="upper right")
    fig.tight_layout()
    return fig


def painel(res):
    """Gera todas as figuras principais e devolve um dicionário nome -> Figure."""
    plt = _estilo()
    figs = {}
    fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4.5))
    diagrama_ph(res, a)
    diagrama_ts(res, b)
    fig.tight_layout()
    figs["ciclo"] = fig
    figs["carga"] = barras(res.carga.componentes, "Componentes da carga térmica", "W")
    figs["condensador"] = perfil_trocador(res.condensador)
    figs["evaporador"] = perfil_trocador(res.evaporador)
    figs["capilar"] = perfil_capilar(res)
    figs["exergia"] = barras(res.exergia["destruicao_W"], "Destruição de exergia por componente", "W")
    if getattr(res, "pulldown", None) is not None:
        figs["pulldown"] = pulldown(res)
    return figs

In [ ]:
%%writefile predimref/__init__.py
"""predimref — pré-dimensionamento de ciclos de refrigeração por compressão de vapor
para refrigeradores comerciais de bebidas (visi-coolers).

Fluxo do cálculo (ver ``projeto.dimensionar``):

1. capacidade do gabinete (litros <-> garrafas) e carga térmica (``gabinete``);
2. ciclo termodinâmico com propriedades de equação de estado (CoolProp / HEOS),
   consistência do COP pedido com Carnot e com o compressor real (``ciclo``);
3. compressor: vazão mássica, deslocamento volumétrico, ponto de catálogo (``compressor``);
4. condensador e evaporador aletados, dimensionados segmento a segmento com ε-NTU,
   correlações locais e queda de pressão (``trocador``);
5. tubo capilar adiabático pelo modelo homogêneo com detecção de blocagem (``capilar``);
6. carga de refrigerante, balanço de energia e de exergia, avisos de projeto;
7. operação fora do projeto: pull-down quase estático, pico de potência e limite de potência
   máxima do compressor (``operacao`` e ``RestricaoPotencia``).

Unidades internas: SI (Pa, K, J/kg, m, kg/s, W). A interface de alto nível aceita °C e litros.
"""

__version__ = "0.1.0"

from .avisos import ErroProjeto
from .projeto import EntradasProjeto, ParametrosProjeto, RestricaoPotencia, ResultadoProjeto, dimensionar

__all__ = ["ErroProjeto", "EntradasProjeto", "ParametrosProjeto", "RestricaoPotencia", "ResultadoProjeto",
           "dimensionar"]

In [ ]:
import importlib, predimref
print('predimref', predimref.__version__, 'pronto')

## 2. Fundamentação teórica

### 2.1 Capacidade: litros ↔ garrafas
A conversão "200 L ↔ 333 garrafas" (200 / 0,6) dá o **volume de líquido**, não o volume de gabinete. Cada garrafa
ocupa o envelope $D^2H$ (≈ 1,01 L para 65 × 239 mm) e só uma fração $\phi$ do volume interno é aproveitável
(prateleiras, folgas de circulação de ar, caixa do evaporador). Com $\phi = 0{,}55$:
$$n_{garrafas} = \left\lfloor \frac{\phi\,V_{int}}{D^2 H}\right\rfloor \qquad (200\ \text{L} \Rightarrow 108\ \text{garrafas};\ 333\ \text{garrafas} \Rightarrow 611\ \text{L})$$
A rotina avisa quando as garrafas informadas não cabem no volume ($\phi > 0{,}70$).

### 2.2 Carga térmica
**Transmissão** por face isolada $i$ (resistências em série, fatores de forma de condução de Incropera, Tab. 4.1):
$$Q_i = \frac{f_{pt}\,\Delta T_i}{\dfrac{1}{h_{int}A_{int,i}} + \dfrac{1}{k\,S_i} + \dfrac{1}{h_{ext}A_{ext,i}}},\qquad
S_i = \frac{A_{int,i}}{\delta_i} + \sum_{arestas}\tfrac12\,0{,}54\,L_a + \sum_{cantos}\tfrac13\,0{,}15\,\delta$$
com fator de pontes térmicas $f_{pt}$ e $\Delta T$ maior na base (compartimento do compressor). **Porta de vidro**:
$Q = U_p A_p \Delta T$. **Gaxeta**: $Q = \psi\,P_{porta}\,\Delta T$ (literatura: ψ ≈ 0,05–0,2 W/m·K).

**Infiltração** por abertura de porta — Gosney & Olama (1975), adotada pela ASHRAE:
$$\dot V = 0{,}221\,A\sqrt{gH}\sqrt{1-\frac{\rho_e}{\rho_i}}\,F_m,\qquad F_m=\left[\frac{2}{1+(\rho_i/\rho_e)^{1/3}}\right]^{1{,}5}$$
O fluxo estabelecido de Gosney-Olama trocaria ~6× o ar de um visi-cooler numa abertura de 8 s, o que é
fisicamente impossível. Por isso a massa trocada por abertura é
$m = \min\left(\rho_i \dot V D_f t_{ab},\; n_{ren}\,V_{livre}/v_e\right)$, e a carga é $m\,(h_e-h_i)$ com entalpias
de ar úmido (sensível + latente).

**Umidade interna (equilíbrio).** A UR interna não é arbitrada: em regime, a água que entra pelas aberturas é a que a
serpentina remove. Com o modelo de superfície efetiva e Lewis = 1 (Braun, Klein & Mitchell 1989):
$T_{s,ef} = T_e - (T_e-T_s)/(1-e^{-NTU_o})$, $NTU_o = h_o\eta_o A_o/(\dot m_{ar}c_p)$ e, em média diária, com o compressor
ligado a fração $f$:
$$A\,(W_e - W_i) = f\,\dot m_{ar}\,(1-e^{-NTU_o})\,(W_i - W_{sat}(T_{s,ef}))\;\Rightarrow\;W_i = \frac{A W_e + B W_s}{A+B}$$
com $A$ = aberturas/dia × massa de ar por abertura / 86 400. Isso é resolvido junto com o ciclo e o evaporador (a
carga latente, o fator ξ e a umidade exportada ao Amesim saem da mesma $W_i$).

**Produto:** $c_p$ da bebida por Choi & Okos (1986) (água + carboidrato), integrado entre a temperatura de entrada
e a alvo. **Casos de projeto:**
$$Q_{proj} = (1+m_s)\,\max\!\left(\frac{Q_{regime}}{f_{util}},\;Q_{transm}+Q_{int}+\frac{E_{carga\ cheia}}{t_{pull\text{-}down}}\right)$$

### 2.3 Ciclo termodinâmico
Temperaturas: $T_{evap} = T_{ar,evap} - \Delta T_{evap}$, $T_{cond} = T_{ar,cond} + \Delta T_{cond}$, superaquecimento
na saída do evaporador, sub-resfriamento na saída do condensador e trocador interno opcional. Todas as
propriedades vêm de equações de estado de referência (Helmholtz) via CoolProp.

**Verificação do COP pedido** (o ponto que protege a CAExperts de vender o impossível):
$$COP < COP_{Carnot} = \frac{T_{evap}}{T_{cond}-T_{evap}},\qquad COP < COP_{ideal} = \frac{h_1-h_4}{h_{2s}-h_{1s}}$$
$$\eta_{global} = \frac{COP}{COP_{ideal}} = \frac{\dot m(h_{2s}-h_{1s})}{\dot W_{el}},\quad \eta_{is}=\frac{\eta_{global}}{\eta_{em}},\quad
h_2 = h_{1s}+\frac{h_{2s}-h_{1s}}{\eta_{is}},\quad \dot Q_{carc} = \dot W_{el} - \dot m(h_2-h_{1s})\ge 0$$
Compressores herméticos alternativos pequenos têm $\eta_{global}$ ≈ 0,35–0,70. Fora disso a rotina avisa; acima do ciclo
ideal ela recusa e informa quais $\Delta T$ dos trocadores seriam necessários. Fechamento:
$\dot Q_{evap}+\dot W_{el} = \dot Q_{cond}+\dot Q_{carc}$.

**Exergia** (estado morto = ambiente): $\dot W_{el} = \dot Q_{evap}\left(\frac{T_0}{T_{frio}}-1\right) + \sum T_0\dot S_{ger}$,
com $\dot S_{ger}$ por componente; $\eta_{II} = COP/COP_{Carnot}(T_{frio},T_0)$.

### 2.4 Compressor
$$\eta_v = f_v\left[1 + C - C\,(P_{desc}/P_{suc})^{1/n}\right],\qquad V_{desl} = \frac{\dot m}{\rho_{suc}\,\eta_v\,N}$$
(motor de 2 polos em 60 Hz, N ≈ 3500 rpm). O mesmo compressor é reavaliado na condição de catálogo
(EN 12900 MBP −10/45 °C, retorno 20 °C) para facilitar a seleção comercial.

### 2.5 Trocadores tubo-aleta (condensador e evaporador)
O circuito é dividido em zonas (dessuperaquecimento / condensação / sub-resfriamento; evaporação /
superaquecimento) e cada zona em segmentos. Em cada segmento:
$$\frac{1}{UA'} = \frac{1}{\xi\,h_o\,\eta_o A_o'} + \frac{R_{f,o}}{A_o'} + \frac{\ln(D_o/D_i)}{2\pi k_t} + \frac{1}{h_i A_i'} + \frac{R_{f,i}}{A_i'},\qquad
\dot C'_{ar} = \frac{\xi\,\rho V_{face} P_t c_p}{N_{fileiras}}$$
* **Bifásico** ($C_r=0$): $\varepsilon = 1-e^{-UA'/\dot C'_{ar}}$ e $L = \dot Q/(\varepsilon\,\dot C'_{ar}\,\Delta T)$.
* **Monofásico**: ε de escoamento cruzado (refrigerante misturado, ar não misturado), $L$ obtido por raiz.
* **Lado do ar**: Wang, Chi & Chang (2000) para $j$ e $f$; $h_o = j\,G_{max}c_p\,Pr^{-2/3}$; aleta de Schmidt (1949),
  a mesma usada na redução dos dados de Wang. ΔP do ar por Kays & London.
* **Condensação**: Shah (2022), que para hidrocarbonetos usa a forma de Shah (2013) com os regimes I, II e III
  (estratificado, $h=h_{Nu}$). Os fluxos mássicos destes sistemas são muito baixos (G ≈ 10–40 kg/m²s).
* **Ebulição**: Gungor & Winterton (1987) com correção de estratificação (Fr < 0,05) e secagem acima de $x_{di}$.
* **Queda de pressão**: Müller-Steinhagen & Heck (atrito) + aceleração com fração de vazio de Rouhani-Axelsson;
  a pressão é atualizada ao longo do tubo e realimenta o ciclo até convergir.
* **Desumidificação** (evaporador): fator de umidade $\xi = Q_{total}/Q_{sens}$ (analogia de Lewis, Threlkeld).

**Eficácia como entrada (padrão 0,65).** A eficácia é a do lado do ar referida à saturação — a definição usual para
condensadores e evaporadores, em que o refrigerante bifásico tem $C\to\infty$ e $C_{min}=\dot C_{ar}$:
$$\varepsilon = \frac{T_{ar,s}-T_{ar,e}}{T_{sat}-T_{ar,e}} = \frac{Q_{sens}}{\dot C_{ar}\,(T_{sat}-T_{ar,e})}
\quad\Rightarrow\quad \dot C_{ar} = \frac{Q/\xi}{\varepsilon\,\Delta T_{aprox}}$$
Com $Q$ e $\Delta T_{aprox}$ fixados pelo ciclo, ε define a **vazão de ar**; a geometria precisa então entregar o NTU
correspondente. Na zona bifásica $\varepsilon_{zona} \le 1-e^{-NTU_{zona}}$ (igualdade com coeficiente uniforme); no
**condensador** a ε global referida a $T_{sat}$ inclui o ar aquecido no dessuperaquecimento (acima de $T_{sat}$) e
**não** segue $1-e^{-NTU}$ — por isso os valores por zona (UA, NTU, ε) também são exportados.
Na distribuição paralela de ar, ε independe do comprimento; depende da velocidade de face $V$, do nº de fileiras $N$,
do coeficiente interno e, no condensador, da temperatura de descarga. Para cada $N$ a busca de $V$ percorre uma grade
de $V_{max}$ para $V_{min}$ e refina por Brent a primeira troca de sinal (a raiz de maior $V$: menos tubo e menos perda de
carga — com perda de carga relevante ε(V) pode não ser monotônica); escolhe-se o $N\le 6$ (faixa de Wang) cuja $V$ fica
mais perto da velocidade preferida. O comprimento sai da marcha segmentada nessa $(V,N)$. Pontos com pinch (a perda de
carga levando a saturação para o lado errado do ar) ou queda de pressão excessiva são rejeitados.

**Malha e circuitos.** 40 segmentos por zona bifásica (≤ 0,15 % do resultado com 240) e 8 por zona monofásica. Se a
perda de carga impedir o projeto (pinch ou ε inatingível por ela), o nº de circuitos é aumentado automaticamente
(até 4), com aviso.

**Duas serpentinas.** A *de projeto* (sem margem) com a vazão de ar requerida entrega exatamente $Q$, $T_{cond}$ e
$T_{evap}$ do ciclo — é a que reproduz `amesim.alvo_validacao`. A *construtiva* tem a margem de área (padrão 10 %), o
mesmo nº de fileiras e de tubos por fileira e o comprimento do passe ajustado; na mesma velocidade de face tem a mesma ε,
mas 10 % mais vazão de ar e UA — num modelo de sistema ela condensa abaixo de $T_{cond}$ e evapora acima de $T_{evap}$
(é a folga, não um erro).
Também se reporta a eficácia estrita $Q/Q_{max}$ com o $Q_{max}$ termodinâmico, caso o componente do Amesim use essa
definição. ε **não** altera o ciclo (as temperaturas de evaporação e condensação vêm de $\Delta T_{evap}$ e $\Delta T_{cond}$):
ela troca vazão de ar (ventilador) por área (trocador).

### 2.6 Tubo capilar — modelo homogêneo de equilíbrio (HEM)
$$G=\dot m/A,\qquad h+\tfrac12 G^2v^2 = h_0,\qquad -dP = \frac{f\,G^2 v}{2D}\,dL + G^2\,dv$$
O líquido escoa isentalpicamente até o ponto de flash $h_l(P_{flash})+\tfrac12G^2v_l^2=h_0$ (para o propano a ~40 °C,
$T\beta>1$ e o flash ocorre **abaixo** de $P_{sat}(T_{in})$). Na região bifásica, o título sai da equação da energia
(quadrática em $x$) e $dL$ da quantidade de movimento. Há **blocagem** quando $-dv/dP \ge 1/G^2$ (máximo de
entropia na linha de Fanno) antes de atingir a pressão do evaporador. Viscosidade bifásica de McAdams
(Cicchitti e Dukler na análise de incerteza); atrito de Churchill. Validado com os dados de Li et al. (1990):
erro de vazão de −1 % a −3 %.

### 2.7 Carga de refrigerante
$m = \sum \rho_{in\,situ} V$, com $\rho_{in\,situ} = \alpha\rho_v + (1-\alpha)\rho_l$ e α de Rouhani-Axelsson nos trechos bifásicos,
mais linhas, filtro, gás na carcaça e refrigerante dissolvido no óleo. Comparada com 150 g, 8×LFL e 13×LFL
(IEC 60335-2-89; confirmar a edição e a regulamentação local).

### 2.8 Operação fora do projeto: pull-down e potência máxima
Com a geometria congelada, o sistema é resolvido pelo **casamento de componentes** (Stoecker & Jones, cap. 14),
com incógnitas $T_{evap}$ e $T_{cond}$:
$$\dot m = V_{desl}\tfrac{N}{60}\,\eta_v(r)\,\rho_{1s},\quad \dot m\,(h_1-h_4)=\xi\,\varepsilon_e\,\dot C_{ar,e}\,(T_{ar,e}-T_{evap}),
\quad \dot m\,(h_2-h_3)=\varepsilon_c\,\dot C_{ar,c}\,(T_{cond}-T_{ar,c})$$
e $W_{el}=\dot m\,(h_{2s}-h_{1s})/\eta_{global}$, com $\eta_v$, $\eta_{global}$, ε e $\dot C_{ar}$ do projeto. No ponto de projeto o
modelo reproduz o dimensionamento (erro ~10⁻⁶). No **pull-down** de porta fechada o produto (capacidade $C_p$) troca calor
com o ar interno por $hA$, e o ar fica em equilíbrio quase estático:
$$0=hA\,(T_p-T_a)+UA_{gab}(T_\infty-T_a)+Q_{base}+Q_{int}-Q_{evap}(T_a),\qquad t=\int\frac{C_p\,dT_p}{hA\,(T_p-T_a)}$$
A potência de um compressor de rotação fixa **não é constante**: com o gabinete quente a pressão de sucção e a massa
específica são altas, e a potência passa por um máximo (para o compressor ideal, em $r=k^{k/(k-1)}\approx 2{,}9$).
O pico estimado (tipicamente 1,2–1,7× o de regime) inclui a partida com o gabinete encharcado e não depende de $hA$; o
tempo de pull-down depende, e $hA$ deve ser calibrado com um ensaio.

O **limite de potência** (`potencia_max_compressor_W`) vale para o regime ou para o pico (`RestricaoPotencia.criterio`).
Com o COP fixo, $W=Q/COP$: respeitar o limite significa um compressor menor (mais tempo ligado, pull-down mais longo) ou
um COP maior. No modo `limitar` a rotina reduz a capacidade até atender e avisa o que muda; se nem com o compressor
100 % ligado o regime fecha, ela informa o COP necessário.

### 2.9 Referências principais
* ASHRAE Handbook — Refrigeration (caps. *Refrigeration Load*, *Thermal Properties of Foods*, *Refrigerant-Control Devices*).
* Bell, I.H. et al. (2014). Pure and pseudo-pure fluid thermophysical property evaluation… CoolProp. *Ind. Eng. Chem. Res.* 53:2498.
* Choi, Y., Okos, M.R. (1986). Effects of temperature and composition on the thermal properties of foods.
* Chato, J.C. (1962). Laminar condensation inside horizontal and inclined tubes. *ASHRAE J.* 4:52.
* Churchill, S.W. (1977). Friction-factor equation spans all fluid-flow regimes. *Chem. Eng.* 84(24):91.
* Gnielinski, V. (1976). *Int. Chem. Eng.* 16:359; VDI Heat Atlas (2010), G1.
* Gosney, W.B., Olama, H.A.L. (1975). Heat and enthalpy gains through cold room doorways. *Proc. Inst. Refrig.* 72:31.
* Gungor, K.E., Winterton, R.H.S. (1987). *Chem. Eng. Res. Des.* 65:148.
* Incropera, F.P. et al. *Fundamentals of Heat and Mass Transfer* (fatores de forma, ε-NTU, aletas).
* Kays, W.M., London, A.L. (1984). *Compact Heat Exchangers*, 3ª ed.
* Li, R.Y., Lin, S., Chen, Z.H. (1990). *Int. J. Refrig.* 13:181; *ASHRAE Trans.* 96:542.
* Liu, Z., Winterton, R.H.S. (1991). *Int. J. Heat Mass Transfer* 34:2759.
* Müller-Steinhagen, H., Heck, K. (1986). *Chem. Eng. Process.* 20:297.
* Rouhani, Z., Axelsson, E. (1970). *Int. J. Heat Mass Transfer* 13:383; Steiner (1993), VDI Heat Atlas.
* Schmidt, T.E. (1949). Heat transfer calculations for extended surfaces. *Refrig. Eng.* 57:351.
* Shah, M.M. (2009) *HVAC&R Res.* 15:889; (2013) *ASHRAE Trans.* 119(2); (2022) *Int. J. Heat Mass Transfer* 194:123069.
* Stoecker, W.F., Jones, J.W. (1982). *Refrigeration and Air Conditioning*, 2ª ed.
* Wang, C.C., Chi, K.Y., Chang, C.J. (2000). *Int. J. Heat Mass Transfer* 43:2693.
* Wongwises, S. et al. (2002). Pressure distribution of refrigerant flow in an adiabatic capillary tube. *ScienceAsia* 28:71.

## 3. Entradas

In [ ]:
#@title 3. Entradas do cliente { run: "auto" }
COP = 2.0  #@param {type:"number"}
informar_COP = True  #@param {type:"boolean"}
T_ambiente_C = 32.0  #@param {type:"number"}
T_alvo_C = 2.5  #@param {type:"number"}
tipo_capacidade = "litros"  #@param ["litros", "garrafas de 600 mL"]
capacidade = 200  #@param {type:"number"}
UR_ambiente = 0.60  #@param {type:"slider", min:0.2, max:0.95, step:0.05}
fluido = "R290"  #@param ["R290", "R600a", "R1270", "R134a", "R1234yf", "R152a", "R404A", "R507A"]
#@markdown **Eficácia dos trocadores** — ε = (T_ar,saída − T_ar,entrada)/(T_sat − T_ar,entrada); desmarque para usar velocidade de face fixa
usar_eficacia = True  #@param {type:"boolean"}
eficacia_condensador = 0.65  #@param {type:"slider", min:0.30, max:0.90, step:0.01}
eficacia_evaporador = 0.65  #@param {type:"slider", min:0.30, max:0.90, step:0.01}
#@markdown **Potência máxima do compressor** (requisito do laboratório; 0 = sem limite). Critério: potência de regime ou pico no pull-down
potencia_max_compressor_W = 0  #@param {type:"number"}
criterio_potencia = "regime"  #@param ["regime", "pico_pulldown"]
modo_potencia = "limitar"  #@param ["limitar", "verificar"]

from predimref import EntradasProjeto
entradas = EntradasProjeto(
    COP=COP if informar_COP else None,
    T_ambiente_C=T_ambiente_C, T_alvo_C=T_alvo_C, UR_ambiente=UR_ambiente, fluido=fluido,
    volume_interno_L=capacidade if tipo_capacidade == "litros" else None,
    n_garrafas=int(capacidade) if tipo_capacidade != "litros" else None,
    eficacia_condensador=eficacia_condensador if usar_eficacia else None,
    eficacia_evaporador=eficacia_evaporador if usar_eficacia else None,
    potencia_max_compressor_W=potencia_max_compressor_W if potencia_max_compressor_W > 0 else None,
)
entradas

In [ ]:
#@title Parâmetros avançados (premissas de projeto) — edite se necessário
from predimref import ParametrosProjeto
from predimref.projeto import RestricaoPotencia
from predimref.operacao import ParametrosOperacao
from predimref.gabinete import Garrafa, Gabinete, Uso
from predimref.ciclo import ParametrosCiclo
from predimref.compressor import ParametrosCompressor
from predimref.capilar import ParametrosCapilar
from predimref.trocador import condensador_padrao, evaporador_padrao

params = ParametrosProjeto(
    garrafa=Garrafa(diametro_m=0.065, altura_m=0.239, acucar_g_por_L=106.0),
    gabinete=Gabinete(fator_ocupacao=0.55, k_isolamento=0.022, esp_lateral_m=0.050, U_porta=2.0,
                      psi_gaxeta=0.10, fator_pontes_termicas=1.10),
    uso=Uso(aberturas_por_dia=100, tempo_abertura_s=8, UR_interna=0.75, tempo_pulldown_h=24,
            fracao_reposicao_diaria=0.5, potencia_iluminacao_W=10, potencia_ventilador_evap_W=8,
            fator_utilizacao_max=0.70, margem_seguranca=0.10),
    ciclo=ParametrosCiclo(dT_evap=10.0, dT_cond=13.0, superaquecimento=5.0, subresfriamento=3.0,
                          eficacia_ihx=0.0, eta_eletromecanica=0.72),
    compressor=ParametrosCompressor(rotacao_rpm=3500, condicao_catalogo="EN12900_MBP"),
    condensador=condensador_padrao(),   # tubo 3/8", aleta 2,5 mm (10 FPI), V preferida 2 m/s
    evaporador=evaporador_padrao(),     # tubo 3/8", aleta 4,23 mm (6 FPI), V preferida 1,5 m/s
    capilar=ParametrosCapilar(serie="polegada", L_min=1.0, L_max=5.0, L_alvo=2.5),
    restricao_potencia=RestricaoPotencia(criterio=criterio_potencia, modo=modo_potencia),
    operacao=ParametrosOperacao(h_produto_W_m2K=5.0,      # ar–garrafa; calibrar com ensaio de pull-down
                                T_ambiente_C=None),       # None = a do projeto (p. ex. 43 para classe climática)
)
# Exemplos de ajuste:
# params.condensador.velocidade_face = 1.5          # velocidade preferida (com ε, guia a escolha de fileiras)
# params.condensador.fileiras_automaticas = False   # com ε, fixa o nº de fileiras e resolve só a velocidade
# params.condensador.geometria.n_fileiras = 4
# params.condensador.comprimento_passe_m = 0.25
# params.evaporador.geometria.passo_aleta = 5.0e-3
# params.evaporador.n_circuitos = 2
print("parâmetros prontos")

## 4. Dimensionamento

In [ ]:
#@title 4. Dimensionamento
from predimref import dimensionar, ErroProjeto
try:
    res = dimensionar(entradas, params)
    print(res.resumo())
except ErroProjeto as erro:
    res = None
    print("ENTRADA INVIÁVEL:\n", erro)

## 5. Gráficos e alternativas de geometria

In [ ]:
#@title 5. Gráficos do ciclo, trocadores, capilar, carga, exergia e pull-down
import matplotlib.pyplot as plt
from IPython.display import display
from predimref.graficos import painel
if res is not None:
    for nome, fig in painel(res).items():
        display(fig)
        plt.close(fig)

In [ ]:
#@title Pull-down: sensibilidade ao hA ar–produto e à temperatura ambiente (tempo e pico de potência)
import copy
import pandas as pd
from predimref import ErroProjeto
from predimref.operacao import modelo_do_projeto, simular_pulldown
if res is not None and res.pulldown is not None:
    modelo, pp = modelo_do_projeto(res), res.parametros
    linhas = []
    for T_amb in sorted({res.entradas.T_ambiente_C, res.entradas.T_ambiente_C + 5.0, 43.0}):
        for h in (3.0, 5.0, 10.0, 1e4):
            po = copy.deepcopy(pp.operacao)
            po.T_ambiente_C, po.h_produto_W_m2K, po.hA_produto_W_K = T_amb, h, None
            try:
                r_pd = simular_pulldown(modelo, res.capacidade, pp.gabinete, pp.uso, pp.garrafa, pp.ciclo,
                                        res.entradas.T_ambiente_C, res.entradas.T_alvo_C, po)
                linhas.append({"T_amb [°C]": T_amb, "h ar–produto [W/m²K]": "∞ (produto = ar)" if h > 1e3 else h,
                               "hA [W/K]": r_pd.hA_produto_W_K, "tempo [h]": r_pd.tempo_h if r_pd.concluido else None,
                               "W pico [W]": r_pd.W_pico, "W fim [W]": r_pd.W_final,
                               "pico/regime": r_pd.W_pico / res.balanco.W_el, "kWh": r_pd.E_compressor_kWh})
            except ErroProjeto as erro:
                linhas.append({"T_amb [°C]": T_amb, "h ar–produto [W/m²K]": h, "erro": str(erro)})
    print("Compressor e trocadores do projeto; o pico não depende de hA (inclui a partida), o tempo depende.")
    display(pd.DataFrame(linhas).round(2))

In [ ]:
#@title Alternativas de geometria para a eficácia escolhida (nº de fileiras × velocidade de face)
import pandas as pd
from predimref.projeto import alternativas_fileiras
if res is not None and entradas.eficacia_condensador is not None:
    for tipo in ("condensador", "evaporador"):
        tab = pd.DataFrame(alternativas_fileiras(res, tipo, range(1, 9)))
        print(f"\n{tipo.upper()} — ε = {getattr(res, tipo).eficacia_alvo:.2f}; escolhido: {getattr(res, tipo).n_fileiras} fileiras"
              " (acima de 6 fileiras a correlação de Wang é extrapolada)")
        display(tab.round(3))

In [ ]:
#@title COP alcançável em função da temperatura ambiente
import numpy as np
from predimref.graficos import mapa_cop
fig = mapa_cop(entradas.fluido, entradas.T_alvo_C, np.arange(20.0, 44.0, 1.0), params.ciclo,
               cop_pedido=entradas.COP)
display(fig)
plt.close(fig)

## 6. Saídas para o Amesim

`res.para_dict()` devolve todos os resultados em um dicionário JSON-serializável. O bloco **`amesim`** reúne,
em SI e por componente, os parâmetros de entrada típicos de um modelo de sistema (compressor volumétrico,
trocadores tubo-aleta, capilar, condições iniciais) e os valores-alvo para conferir o modelo do Amesim em regime
permanente. O mapeamento para os nomes dos parâmetros de cada submodelo do Amesim fica com a equipe de
integração (Diego e Pedro).

Na integração em Python (fora do notebook):
```python
from predimref import EntradasProjeto, dimensionar
res = dimensionar(EntradasProjeto(COP=2.0, T_ambiente_C=32, T_alvo_C=2.5, volume_interno_L=200))
parametros_amesim = res.para_dict()["amesim"]
```
O núcleo de cálculo depende só de `numpy`, `scipy` e `CoolProp` (matplotlib apenas para gráficos).

In [ ]:
#@title Exportar JSON / CSV / Excel e baixar
import csv, json
from predimref.relatorio import tabela_estados, perfil_trocador
if res is not None:
    d = res.para_dict()
    print(json.dumps(d["amesim"], indent=2, ensure_ascii=False))
    res.exportar_json("predim_resultado.json")
    with open("predim_estados.csv", "w", newline="", encoding="utf-8") as fh:
        linhas = tabela_estados(res)
        w = csv.DictWriter(fh, fieldnames=list(linhas[0]))
        w.writeheader(); w.writerows(linhas)
    arquivos = ["predim_resultado.json", "predim_estados.csv"]
    try:
        import pandas as pd
        with pd.ExcelWriter("predim_resultado.xlsx") as xl:
            pd.DataFrame(tabela_estados(res)).to_excel(xl, sheet_name="estados", index=False)
            pd.DataFrame(perfil_trocador(res.condensador)).to_excel(xl, sheet_name="condensador", index=False)
            pd.DataFrame(perfil_trocador(res.evaporador)).to_excel(xl, sheet_name="evaporador", index=False)
            pd.DataFrame([{"componente": k, "W": v} for k, v in res.carga.componentes.items()]).to_excel(
                xl, sheet_name="carga_termica", index=False)
            if d.get("pulldown"):
                pd.DataFrame(d["pulldown"]["curva"]).to_excel(xl, sheet_name="pulldown", index=False)
        arquivos.append("predim_resultado.xlsx")
    except Exception as e:
        print("Excel não gerado:", e)
    try:
        from google.colab import files
        for a in arquivos:
            files.download(a)
    except ImportError:
        print("Arquivos gravados:", arquivos)

## 7. Sensibilidade e incerteza

Duas perguntas distintas:
1. **Sensibilidade às entradas/premissas** — como o projeto muda com a temperatura ambiente, o COP pedido e o
   perfil de uso (a carga térmica é, de longe, a maior fonte de incerteza);
2. **Incerteza de modelo** — o mesmo problema resolvido com correlações alternativas (o resumo da seção 4 já mostra
   evaporador GW1987 × LW1991, condensador Shah 2022 × Chato 1962 e capilar com três viscosidades).

In [ ]:
#@title Varredura de temperatura ambiente e de COP
import copy
import pandas as pd
from predimref import ParametrosProjeto
linhas = []
p_rapido = copy.deepcopy(params); p_rapido.analise_sensibilidade = False
for Ta in (25.0, 30.0, 32.0, 35.0, 38.0, 43.0):
    for cop in ((entradas.COP,) if entradas.COP else (None,)):
        e = copy.deepcopy(entradas); e.T_ambiente_C = Ta; e.COP = cop
        try:
            r = dimensionar(e, p_rapido)
            linhas.append({"T_amb [°C]": Ta, "Q_proj [W]": r.carga.Q_projeto, "W_el [W]": r.balanco.W_el,
                           "η_global": r.ciclo.eta_global, "desloc. [cm³]": r.compressor.deslocamento_cm3,
                           "L_cond [m]": r.condensador.L_requerido, "L_evap [m]": r.evaporador.L_requerido,
                           "capilar": f'{r.capilar.escolhido.D*1e3:.3f} mm × {r.capilar.escolhido.L:.2f} m',
                           "carga [g]": r.carga_refrigerante["total_kg"] * 1e3, "kWh/dia": r.energia["total_kWh_dia"]})
        except ErroProjeto as err:
            linhas.append({"T_amb [°C]": Ta, "Q_proj [W]": None, "capilar": "inviável: " + str(err)[:60]})
pd.set_option("display.precision", 2)
pd.DataFrame(linhas)

In [ ]:
#@title Efeito da eficácia escolhida (mesma eficácia nos dois trocadores)
linhas = []
for eps in (0.45, 0.55, 0.65, 0.75):
    e = copy.deepcopy(entradas); e.eficacia_condensador = e.eficacia_evaporador = eps
    try:
        r = dimensionar(e, p_rapido)
        linhas.append({"ε": eps,
                       "cond: fileiras × V [m/s]": f"{r.condensador.n_fileiras} × {r.condensador.velocidade_face:.2f}",
                       "cond: L [m]": r.condensador.L_requerido, "cond: ar [m³/h]": r.condensador.vazao_ar_requerida_m3h,
                       "evap: fileiras × V [m/s]": f"{r.evaporador.n_fileiras} × {r.evaporador.velocidade_face:.2f}",
                       "evap: L [m]": r.evaporador.L_requerido, "evap: ar [m³/h]": r.evaporador.vazao_ar_requerida_m3h,
                       "T ar insuflado [°C]": r.evaporador.T_ar_out - 273.15,
                       "carga R [g]": r.carga_refrigerante["total_kg"] * 1e3})
    except ErroProjeto as err:
        linhas.append({"ε": eps, "cond: fileiras × V [m/s]": "inviável: " + str(err)[:70]})
pd.DataFrame(linhas)

In [ ]:
#@title Efeito das premissas de uso (aberturas de porta e reposição de produto)
linhas = []
for ab in (25, 100, 300):
    for rep in (0.2, 0.5, 1.0):
        p2 = copy.deepcopy(p_rapido); p2.uso.aberturas_por_dia = ab; p2.uso.fracao_reposicao_diaria = rep
        try:
            r = dimensionar(entradas, p2)
            linhas.append({"aberturas/dia": ab, "reposição/dia": rep, "Q_proj [W]": r.carga.Q_projeto,
                           "caso": r.carga.caso_governante, "desloc. [cm³]": r.compressor.deslocamento_cm3,
                           "kWh/dia": r.energia["total_kWh_dia"]})
        except ErroProjeto as err:
            linhas.append({"aberturas/dia": ab, "reposição/dia": rep, "caso": "inviável"})
pd.DataFrame(linhas)

## 8. Verificação e validação

**Verificação** (o código resolve certo as equações): fechamento dos balanços de energia e de exergia, 2ª lei por
componente, independência de malha dos trocadores e do capilar, consistência do problema inverso do capilar,
eficiência de aleta de Schmidt contra a solução exata (Bessel), eficácia alvo atingida, limite ε ≤ 1 − e^(−NTU) no
evaporador só bifásico, coerência do bloco do Amesim (ε·Ċ_ar·ΔT = Q na serpentina de projeto) e casos-limite (título final
exato, pinch por perda de carga, alvo de ε acima/abaixo do atingível). O repositório tem testes automatizados
(`pytest`) com essas e outras verificações.

**Validação** (as equações certas): capilar contra os dados experimentais de Li et al. (1990). O próximo passo
recomendado é validar o sistema completo contra ensaios de um visi-cooler IMBERA existente (ver seção 9).

In [ ]:
#@title Verificações do caso atual e validação do capilar (Li et al., 1990 — R12, D = 0,66 mm, L = 1,5 m)
from predimref.propriedades import Fluido
from predimref.capilar import ParametrosCapilar, vazao_capilar
if res is not None:
    print(f"balanço de energia: erro relativo = {res.balanco.erro_balanco:.1e}")
    print(f"balanço de exergia: erro relativo = {res.exergia['erro_fechamento']:.1e}")
    print(f"2ª lei por componente: min(Ex_destruída) = {min(res.exergia['destruicao_W'].values()):.3f} W (≥ 0)")
    print(f"COP {res.ciclo.COP:.3f} < ciclo ideal {res.ciclo.COP_ciclo_ideal:.3f} < Carnot {res.ciclo.COP_carnot_ciclo:.3f}")
fl12 = Fluido("R12")
print("\nCapilar — vazão prevista × medida (Li et al. 1990):")
for Ti, Pi, m_exp in ((31.40, 9.67e5, 1.13e-3), (23.40, 7.17e5, 0.844e-3)):
    h = fl12.estado_PT(Pi, Ti + 273.15).h
    for visc in ("mcadams", "cicchitti", "dukler"):
        m = vazao_capilar(fl12, 0.66e-3, 1.5, Pi, h, 1.5e5, ParametrosCapilar(rugosidade_m=0.003 * 0.66e-3,
                                                                              modelo_viscosidade=visc))
        print(f"  T_in {Ti:5.1f} °C  {visc:9s}  medida {m_exp*1e3:.3f} g/s  HEM {m*1e3:.3f} g/s  ({m/m_exp-1:+.1%})")

## 9. Hipóteses, limitações e próximos passos

**Hipóteses principais (todas editáveis em *Parâmetros avançados*):**
* regime permanente na condição de projeto; o compressor cicla em regime e opera 100 % no pull-down;
* trocadores tubo-aleta de aletas planas, tubos escalonados, ar com distribuição paralela por trecho;
  curvas de retorno fora da perda de carga; coeficientes de correlações para tubo liso e refrigerante sem óleo;
* capilar adiabático e sem metaestabilidade (com trocador interno, o capilar real é não adiabático);
* compressor: modelo de espaço morto para η_v e η_eletromecânica constante — substituir por mapa de catálogo
  (polinômio AHRI 540 / EN 12900) assim que o compressor candidato for escolhido; o pico de potência no pull-down
  usa o mesmo modelo (η_global constante), então também melhora com o mapa;
* pull-down quase estático: superaquecimento fixo (o capilar real deixa o evaporador sub-alimentado no início),
  massa do gabinete desprezada e hA ar–produto como premissa (calibrar com ensaio);
* geada no evaporador tratada só por resistência de incrustação (padrão zero) e pelo fator de umidade.

**Relação com o Amesim (a rotina é o *step-down* do modelo de sistema):**
1. Esta rotina gera, em regime permanente, parâmetros coerentes entre si (balanços fechados, 2ª lei, COP viável)
   para montar o modelo do Amesim: compressor, trocadores com a eficácia escolhida, capilar e carga de fluido.
2. As curvas de **pull-down, consumo, fuga térmica e água drenada** vêm do Amesim (o bloco `amesim.pulldown_estimado`
   traz só uma estimativa quase estática para comparar a ordem de grandeza do tempo e do pico de potência). Primeiro passo de conferência:
   montado com as serpentinas `projeto_sem_margem` e a vazão de ar requerida, o modelo do Amesim deve reproduzir
   `amesim.alvo_validacao` (Q_evap, W_el, ṁ, T_evap, T_cond). Com as `construtiva_com_margem` (a geometria que se
   fabrica), T_cond fica abaixo e T_evap acima do alvo — é a folga de área, não um erro de mapeamento. Fora isso,
   diferenças grandes indicam parâmetro mapeado de forma diferente: definição de eficácia (aqui referida à saturação,
   de bulbo seco; a estrita Q/Q_max também é exportada), UA úmido × seco no evaporador (ambos exportados) e a carga
   (o bloco separa circuito, gás na carcaça e refrigerante dissolvido no óleo).
3. Quando o compressor for escolhido, troque o modelo genérico (espaço morto + η constante) pelo mapa de catálogo
   nos dois lados (aqui e no Amesim).
4. Se mais tarde houver ensaio de um produto IMBERA, ele calibra os dois modelos ao mesmo tempo (UA do gabinete,
   ΔT dos trocadores, rendimentos do compressor).